# Početna analiza Olist Brazilian E-Commerce skupa podataka


Analiza obuhvaća:

- učitavanje izvornih CSV datoteka,
- pregled strukture pojedinih skupova podataka,
- utvrđivanje broja zapisa i atributa,
- identifikaciju jedinstvenih poslovnih entiteta,
- analizu vremenskog raspona podataka,
- analizu nedostajućih vrijednosti,
- analizu potpunih duplikata,
- detaljniju analizu uočenih problema kvalitete podataka,
- pretvorbu vremenskih atributa u odgovarajući `datetime` format.


## 1. Uvoz potrebnih biblioteka

Za učitavanje, obradu i analizu podataka koristi se Python biblioteka `pandas`. Podaci iz CSV datoteka učitavaju se u strukture tipa `DataFrame`.

In [493]:
import pandas as pd

print("Pandas verzija:", pd.__version__)


Pandas verzija: 3.0.3


### Provjera radnog direktorija

Prije učitavanja podataka provjerava se trenutni radni direktorij Jupyter bilježnice kako bi se ispravno definirale relativne putanje do izvornih CSV datoteka.

In [494]:
import os

print(os.getcwd())

c:\Users\Ana\Desktop\diplomskiPrakticni\notebooks


## 2. Učitavanje izvornih podataka

Izvorni skup podataka sastoji se od devet međusobno povezanih CSV datoteka. Svaka datoteka predstavlja određeni dio poslovnog procesa, kao što su kupci, narudžbe, stavke narudžbi, proizvodi, prodavači, plaćanja, recenzije i geografski podaci.

CSV datoteke učitavaju se u zasebne Pandas `DataFrame` objekte. Izvorni podaci u ovoj fazi ostaju nepromijenjeni kako bi se prije ETL transformacija mogla analizirati njihova struktura i kvaliteta.

In [495]:
customers = pd.read_csv("../data/raw/olist_customers_dataset.csv")
geolocation = pd.read_csv("../data/raw/olist_geolocation_dataset.csv")
order_items = pd.read_csv("../data/raw/olist_order_items_dataset.csv")
payments = pd.read_csv("../data/raw/olist_order_payments_dataset.csv")
reviews = pd.read_csv("../data/raw/olist_order_reviews_dataset.csv")
orders = pd.read_csv("../data/raw/olist_orders_dataset.csv")
products = pd.read_csv("../data/raw/olist_products_dataset.csv")
sellers = pd.read_csv("../data/raw/olist_sellers_dataset.csv")
category_translation = pd.read_csv("../data/raw/product_category_name_translation.csv")

### Primjer strukture podataka


In [496]:
customers.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


## 3. Pregled strukture skupova podataka

Svi učitani DataFrameovi organizirani su u zajednički rječnik kako bi se nad njima mogle jednostavno provoditi iste analize.


In [497]:
dataframes = {
    "customers": customers,
    "geolocation": geolocation,
    "order_items": order_items,
    "payments": payments,
    "reviews": reviews,
    "orders": orders,
    "products": products,
    "sellers": sellers,
    "category_translation": category_translation
}


In [498]:
pregled_datasetova = pd.DataFrame({
    "dataset": dataframes.keys(),
    "broj_redaka": [df.shape[0] for df in dataframes.values()],
    "broj_stupaca": [df.shape[1] for df in dataframes.values()]
})

pregled_datasetova

,dataset,broj_redaka,broj_stupaca
0,customers,99441,5
1,geolocation,1000163,5
2,order_items,112650,7
3,payments,103886,5
4,reviews,99224,7
5,orders,99441,8
6,products,32951,9
7,sellers,3095,4
8,category_translation,71,2


Iz početnog pregleda vidljivo je da skup podataka sadrži 99.441 zapis o narudžbama, dok skup `order_items` sadrži 112.650 zapisa. Očekivana je mogućnost da bude veći broj stavki od broja narudžbi jer jedna narudžba može sadržavati više proizvoda.

Skup `payments` također sadrži više zapisa od skupa `orders`, što ukazuje na mogućnost postojanja više zapisa o plaćanju za pojedinu narudžbu.

Najveći skup je `geolocation`, koji prije čišćenja sadrži 1.000.163 zapisa. 

### Pregled atributa


In [499]:
for name, df in dataframes.items():
    print(f"\n{name.upper()}")
    print(list(df.columns))


CUSTOMERS
['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']

GEOLOCATION
['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng', 'geolocation_city', 'geolocation_state']

ORDER_ITEMS
['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']

PAYMENTS
['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']

REVIEWS
['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp']

ORDERS
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

PRODUCTS
['product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'prod

## 4. Analiza osnovnih poslovnih entiteta

Kako bi se utvrdila veličina skupa podataka na razini glavnih poslovnih entiteta, određuje se broj jedinstvenih narudžbi, kupaca, proizvoda i prodavača.

In [500]:
print("Broj jedinstvenih narudžbi:",
      orders["order_id"].nunique())

print("Broj jedinstvenih kupaca:",
      customers["customer_unique_id"].nunique())

print("Broj jedinstvenih proizvoda:",
      products["product_id"].nunique())

print("Broj jedinstvenih prodavača:",
      sellers["seller_id"].nunique())

Broj jedinstvenih narudžbi: 99441
Broj jedinstvenih kupaca: 96096
Broj jedinstvenih proizvoda: 32951
Broj jedinstvenih prodavača: 3095


In [501]:
print("Broj redaka u customers:",
      len(customers))

print("Broj jedinstvenih customer_id:",
      customers["customer_id"].nunique())

print("Broj jedinstvenih customer_unique_id:",
      customers["customer_unique_id"].nunique())

Broj redaka u customers: 99441
Broj jedinstvenih customer_id: 99441
Broj jedinstvenih customer_unique_id: 96096


### Razlika između `customer_id` i `customer_unique_id`

U skupu `customers` postoji 99.441 jedinstvena vrijednost atributa `customer_id` i 96.096 jedinstvenih vrijednosti atributa `customer_unique_id`.

Atribut `customer_id` predstavlja identifikator kupca povezan s konkretnom narudžbom, dok `customer_unique_id` omogućuje identifikaciju istog stvarnog kupca kroz više različitih narudžbi.

## 5. Obrada vremenskih atributa

Prije provođenja vremenskih analiza svi atributi koji predstavljaju datum ili vrijeme pretvaraju se u Pandas `datetime` format.

Nedostajuće vremenske vrijednosti nakon pretvorbe predstavljene su vrijednošću `NaT` (*Not a Time*).

In [502]:
# ORDERS - vremenski atributi
orders["order_purchase_timestamp"] = pd.to_datetime(
    orders["order_purchase_timestamp"]
)

orders["order_approved_at"] = pd.to_datetime(
    orders["order_approved_at"]
)

orders["order_delivered_carrier_date"] = pd.to_datetime(
    orders["order_delivered_carrier_date"]
)

orders["order_delivered_customer_date"] = pd.to_datetime(
    orders["order_delivered_customer_date"]
)

orders["order_estimated_delivery_date"] = pd.to_datetime(
    orders["order_estimated_delivery_date"]
)


# ORDER ITEMS - vremenski atributi
order_items["shipping_limit_date"] = pd.to_datetime(
    order_items["shipping_limit_date"]
)


# REVIEWS - vremenski atributi
reviews["review_creation_date"] = pd.to_datetime(
    reviews["review_creation_date"]
)

reviews["review_answer_timestamp"] = pd.to_datetime(
    reviews["review_answer_timestamp"]
)

In [503]:
print("ORDERS:")
print(orders.dtypes)

print("\nORDER ITEMS:")
print(order_items.dtypes)

print("\nREVIEWS:")
print(reviews.dtypes)

ORDERS:
order_id                                    str
customer_id                                 str
order_status                                str
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

ORDER ITEMS:
order_id                          str
order_item_id                   int64
product_id                        str
seller_id                         str
shipping_limit_date    datetime64[us]
price                         float64
freight_value                 float64
dtype: object

REVIEWS:
review_id                             str
order_id                              str
review_score                        int64
review_comment_title                  str
review_comment_message                str
review_creation_date       datetime64[us]
review_answer_timestamp    datetime64[

In [504]:
print(
    "Najranija narudžba:",
    orders["order_purchase_timestamp"].min()
)

print(
    "Najkasnija narudžba:",
    orders["order_purchase_timestamp"].max()
)

Najranija narudžba: 2016-09-04 21:15:19
Najkasnija narudžba: 2018-10-17 17:30:18


In [505]:
print(
    "Najranija dostava:",
    orders["order_delivered_customer_date"].min()
)

print(
    "Najkasnija dostava:",
    orders["order_delivered_customer_date"].max()
)

Najranija dostava: 2016-10-11 13:46:32
Najkasnija dostava: 2018-10-17 13:22:46


Broj narudžbi po godini:

In [506]:
orders["order_purchase_timestamp"].dt.year.value_counts().sort_index()

order_purchase_timestamp
2016      329
2017    45101
2018    54011
Name: count, dtype: int64

Analizirani podaci o narudžbama obuhvaćaju razdoblje od **4. rujna 2016. do 17. listopada 2018. godine**. Najveći dio narudžbi odnosi se na 2017. i 2018. godinu, dok je za 2016. godinu dostupan znatno manji broj zapisa.

## 6. Analiza nedostajućih vrijednosti


In [507]:
for name, df in dataframes.items():
    print(f"\n{name.upper()}")
    print(df.isna().sum())


CUSTOMERS
customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

GEOLOCATION
geolocation_zip_code_prefix    0
geolocation_lat                0
geolocation_lng                0
geolocation_city               0
geolocation_state              0
dtype: int64

ORDER_ITEMS
order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64

PAYMENTS
order_id                0
payment_sequential      0
payment_type            0
payment_installments    0
payment_value           0
dtype: int64

REVIEWS
review_id                      0
order_id                       0
review_score                   0
review_comment_title       87656
review_comment_message     58247
review_creation_date           0
review_answer_timestamp        0
dtype: int64

ORDERS
order_id            

Nedostajuće vrijednosti pronađene su prvenstveno u skupovima `reviews`, `orders` i `products`.

U skupu `reviews` velik broj nedostajućih vrijednosti postoji u atributima `review_comment_title` i `review_comment_message`. To se opravdava time da kupac može dati numeričku ocjenu bez pisanja tekstualnog komentara, pa te vrijednosti ne smatram nužno problemom kvalitete podataka.

U skupu `orders` nedostaju pojedini datumi povezani s odobravanjem, slanjem i dostavom narudžbi. Njihovo značenje potrebno je analizirati zajedno sa statusom narudžbe.

U skupu `products` za 610 proizvoda nedostaju podaci o kategoriji i opisnim karakteristikama proizvoda, dok za dva proizvoda nedostaju pojedini podaci o fizičkim dimenzijama.

### 6.1. Analiza nedostajućih datuma dostave

Posebno se analizira atribut `order_delivered_customer_date`, koji predstavlja stvarni datum dostave narudžbe kupcu.

Kako bi se utvrdilo predstavljaju li nedostajuće vrijednosti pogrešku ili očekivano stanje, narudžbe bez evidentiranog datuma dostave grupiraju se prema statusu.

In [508]:
orders[
    orders["order_delivered_customer_date"].isna()
]["order_status"].value_counts()


order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64

In [509]:
orders[
    (orders["order_status"] == "delivered") &
    (orders["order_delivered_customer_date"].isna())
]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,ec05a6d8558c6455f0cbbd8a420ad34f,delivered,2017-11-28 17:44:07,2017-11-28 17:56:40,2017-11-30 18:12:23,NaT,2017-12-18
20618,f5dd62b788049ad9fc0526e3ad11a097,5e89028e024b381dc84a13a3570decb4,delivered,2018-06-20 06:58:43,2018-06-20 07:19:05,2018-06-25 08:05:00,NaT,2018-07-16
43834,2ebdfc4f15f23b91474edf87475f108e,29f0540231702fda0cfdee0a310f11aa,delivered,2018-07-01 17:05:11,2018-07-01 17:15:12,2018-07-03 13:57:00,NaT,2018-07-30
79263,e69f75a717d64fc5ecdfae42b2e8e086,cfda40ca8dd0a5d486a9635b611b398a,delivered,2018-07-01 22:05:55,2018-07-01 22:15:14,2018-07-03 13:57:00,NaT,2018-07-30
82868,0d3268bad9b086af767785e3f0fc0133,4f1d63d35fb7c8999853b2699f5c7649,delivered,2018-07-01 21:14:02,2018-07-01 21:29:54,2018-07-03 09:28:00,NaT,2018-07-24
92643,2d858f451373b04fb5c984a1cc2defaf,e08caf668d499a6d643dafd7c5cc498a,delivered,2017-05-25 23:22:43,2017-05-25 23:30:16,NaT,NaT,2017-06-23
97647,ab7c89dc1bf4a1ead9d6ec1ec8968a84,dd1b84a7286eb4524d52af4256c0ba24,delivered,2018-06-08 12:09:39,2018-06-08 12:36:39,2018-06-12 14:10:00,NaT,2018-06-26
98038,20edc82cf5400ce95e1afacc25798b31,28c37425f1127d887d7337f284080a0f,delivered,2018-06-27 16:09:12,2018-06-27 16:29:30,2018-07-03 19:26:00,NaT,2018-07-19


Analizom nedostajućih vrijednosti atributa `order_delivered_customer_date` pronađeno je 2.965 zapisa bez evidentiranog datuma dostave. Većina nedostajućih vrijednosti povezana je sa statusima narudžbi kod kojih dostava nije završena ili se nije dogodila, poput shipped, canceled, unavailable, invoiced i processing. Stoga se takve vrijednosti ne smatraju nužno pogreškama u podacima. 

#### Potencijalno nekonzistentni zapisi

Uočeno je osam narudžbi koje imaju status `delivered`, ali nemaju evidentiran `order_delivered_customer_date`. Budući da status označava da je narudžba dostavljena, a stvarni datum dostave nedostaje, ovi zapisi predstavljaju potencijalnu nekonzistentnost izvornog skupa podataka.

Takvi zapisi neće moći sudjelovati u analizama koje zahtijevaju poznat stvarni datum dostave.

### 6.2. Analiza nedostajućih vrijednosti atributa `order_approved_at`

Atribut `order_approved_at` predstavlja vrijeme odobrenja narudžbe. U skupu podataka pronađeno je 160 narudžbi kojima nedostaje vrijednost ovog atributa.

Kako bih utvrdila jesu li nedostajuće vrijednosti očekivane s obzirom na stanje narudžbe ili predstavljaju potencijalnu nekonzistentnost podataka, analiziram  takve narudžbe prema njihovim statusima.

In [510]:
orders[
    orders["order_approved_at"].isna()
]["order_status"].value_counts()

order_status
canceled     141
delivered     14
created        5
Name: count, dtype: int64

141 zapis odnosi na otkazane (`canceled`), 5 na kreirane (`created`), a 14 na dostavljene (`delivered`) narudžbe.

Nedostatak vremena odobrenja kod otkazanih i tek kreiranih narudžbi može biti očekivan s obzirom na to da takve narudžbe nisu nužno prošle cjelokupan proces obrade. 

14 dostavljenih narudžbi bez evidentiranog vremena odobrenja predstavlja potencijalnu nekonzistentnost ili nedostatak u izvornim podacima. Takve narudžbe se ne mogu koristiti u analizama koje zahtijevaju poznato vrijeme odobrenja narudžbe.

In [511]:
orders[
    (orders["order_status"] == "delivered") &
    (orders["order_approved_at"].isna())
]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
5323,e04abd8149ef81b95221e88f6ed9ab6a,2127dc6603ac33544953ef05ec155771,delivered,2017-02-18 14:40:00,NaT,2017-02-23 12:04:47,2017-03-01 13:25:33,2017-03-17
16567,8a9adc69528e1001fc68dd0aaebbb54a,4c1ccc74e00993733742a3c786dc3c1f,delivered,2017-02-18 12:45:31,NaT,2017-02-23 09:01:52,2017-03-02 10:05:06,2017-03-21
19031,7013bcfc1c97fe719a7b5e05e61c12db,2941af76d38100e0f8740a374f1a5dc3,delivered,2017-02-18 13:29:47,NaT,2017-02-22 16:25:25,2017-03-01 08:07:38,2017-03-17
22663,5cf925b116421afa85ee25e99b4c34fb,29c35fc91fc13fb5073c8f30505d860d,delivered,2017-02-18 16:48:35,NaT,2017-02-22 11:23:10,2017-03-09 07:28:47,2017-03-31
23156,12a95a3c06dbaec84bcfb0e2da5d228a,1e101e0daffaddce8159d25a8e53f2b2,delivered,2017-02-17 13:05:55,NaT,2017-02-22 11:23:11,2017-03-02 11:09:19,2017-03-20
26800,c1d4211b3dae76144deccd6c74144a88,684cb238dc5b5d6366244e0e0776b450,delivered,2017-01-19 12:48:08,NaT,2017-01-25 14:56:50,2017-01-30 18:16:01,2017-03-01
38290,d69e5d356402adc8cf17e08b5033acfb,68d081753ad4fe22fc4d410a9eb1ca01,delivered,2017-02-19 01:28:47,NaT,2017-02-23 03:11:48,2017-03-02 03:41:58,2017-03-27
39334,d77031d6a3c8a52f019764e68f211c69,0bf35cac6cc7327065da879e2d90fae8,delivered,2017-02-18 11:04:19,NaT,2017-02-23 07:23:36,2017-03-02 16:15:23,2017-03-22
48401,7002a78c79c519ac54022d4f8a65e6e8,d5de688c321096d15508faae67a27051,delivered,2017-01-19 22:26:59,NaT,2017-01-27 11:08:05,2017-02-06 14:22:19,2017-03-16
61743,2eecb0d85f281280f79fa00f9cec1a95,a3d3c38e58b9d2dfb9207cab690b6310,delivered,2017-02-17 17:21:55,NaT,2017-02-22 11:42:51,2017-03-03 12:16:03,2017-03-20


Za svih 14 narudžbi sa statusom `delivered` kojima nedostaje vrijednost atributa `order_approved_at` evidentirani su kasniji koraci procesa, uključujući datum predaje narudžbe dostavnoj službi (`order_delivered_carrier_date`) i stvarni datum dostave kupcu (`order_delivered_customer_date`).

Kako su narudžbe uspješno prošle kroz proces dostave, nedostatak vrijednosti `order_approved_at` vjerojatno predstavlja nedostatak u evidenciji izvornog skupa podataka, a ne činjenicu da narudžbe nisu bile odobrene.

Uočeno je i da su sve navedene narudžbe nastale u siječnju i veljači 2017. godine, pri čemu je većina narudžbi kreirana u razdoblju od 17. do 19. veljače 2017. godine. Grupiranje nedostajućih vrijednosti u relativno kratkom vremenskom razdoblju može upućivati na problem u prikupljanju ili evidentiranju podataka tijekom tog razdoblja, ali na temelju dostupnih podataka ne može se utvrditi njegov točan uzrok.

Ove narudžbe neće se uklanjati iz skladišta podataka jer sadrže relevantne informacije o kupnji i dostavi. Međutim, neće se koristiti u izračunima i analizama za koje je potreban poznat datum i vrijeme odobrenja narudžbe.

### 6.3. Analiza nedostajućih vrijednosti atributa `order_delivered_carrier_date`

Atribut `order_delivered_carrier_date` predstavlja vrijeme kada je narudžba predana dostavnoj službi. U skupu podataka identificirane su 1.783 narudžbe kojima nedostaje vrijednost ovog atributa.

Nedostatak ove vrijednosti može biti očekivan za narudžbe koje su otkazane, nedostupne ili još nisu dosegle fazu slanja. Kako bi se utvrdila povezanost nedostajućih vrijednosti sa stanjem narudžbe, analiziraju se pripadajući statusi narudžbi.

In [512]:
orders[
    orders["order_delivered_carrier_date"].isna()
]["order_status"].value_counts()

order_status
unavailable    609
canceled       550
invoiced       314
processing     301
created          5
approved         2
delivered        2
Name: count, dtype: int64

Analizom 1.783 narudžbe bez evidentirane vrijednosti atributa `order_delivered_carrier_date` utvrđeno je da se velika većina odnosi na narudžbe koje nisu dosegle fazu dostave, odnosno imaju statuse `unavailable`, `canceled`, `invoiced`, `processing`, `created` ili `approved`. U tim slučajevima nedostatak datuma predaje dostavnoj službi može se smatrati očekivanim s obzirom na stanje narudžbe.

Postoje 2 narudžbe sa statusom `delivered` kojima nedostaje datum predaje dostavnoj službi. Budući da su prema statusu narudžbe dostavljene kupcu, ovi zapisi predstavljaju potencijalnu nekonzistentnost ili nedostatak u izvornim podacima. Narudžbe se ne mogu koristiti u analizama koje zahtijevaju poznato vrijeme predaje narudžbe dostavnoj službi.

In [513]:
orders[
    (orders["order_status"] == "delivered") &
    (orders["order_delivered_carrier_date"].isna())
]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
73222,2aa91108853cecb43c84a5dc5b277475,afeb16c7f46396c0ed54acb45ccaaa40,delivered,2017-09-29 08:52:58,2017-09-29 09:07:16,NaT,2017-11-20 19:44:47,2017-11-14
92643,2d858f451373b04fb5c984a1cc2defaf,e08caf668d499a6d643dafd7c5cc498a,delivered,2017-05-25 23:22:43,2017-05-25 23:30:16,NaT,NaT,2017-06-23


Detaljnijim pregledom narudžbi sa statusom `delivered` kojima nedostaje vrijednost atributa `order_delivered_carrier_date` postoje dvije različite situacije.

Kod prve narudžbe evidentirani su vrijeme kupnje, vrijeme odobrenja i stvarni datum dostave kupcu, ali nedostaje vrijeme predaje narudžbe dostavnoj službi. Budući da je stvarna dostava evidentirana, može se zaključiti da nedostaje samo jedan međukorak u evidenciji procesa dostave.

Kod druge narudžbe, osim vremena predaje dostavnoj službi (`order_delivered_carrier_date`), nedostaje i stvarni datum dostave kupcu (`order_delivered_customer_date`), iako narudžba ima status `delivered`. Ovaj zapis stoga sadrži više nedostajućih vremenskih podataka povezanih s procesom dostave.

In [514]:
delivered_s_null_vrijednostima = orders[
    (orders["order_status"] == "delivered") &
    (
        orders["order_approved_at"].isna() |
        orders["order_delivered_carrier_date"].isna() |
        orders["order_delivered_customer_date"].isna()
    )
]

delivered_s_null_vrijednostima

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,ec05a6d8558c6455f0cbbd8a420ad34f,delivered,2017-11-28 17:44:07,2017-11-28 17:56:40,2017-11-30 18:12:23,NaT,2017-12-18
5323,e04abd8149ef81b95221e88f6ed9ab6a,2127dc6603ac33544953ef05ec155771,delivered,2017-02-18 14:40:00,NaT,2017-02-23 12:04:47,2017-03-01 13:25:33,2017-03-17
16567,8a9adc69528e1001fc68dd0aaebbb54a,4c1ccc74e00993733742a3c786dc3c1f,delivered,2017-02-18 12:45:31,NaT,2017-02-23 09:01:52,2017-03-02 10:05:06,2017-03-21
19031,7013bcfc1c97fe719a7b5e05e61c12db,2941af76d38100e0f8740a374f1a5dc3,delivered,2017-02-18 13:29:47,NaT,2017-02-22 16:25:25,2017-03-01 08:07:38,2017-03-17
20618,f5dd62b788049ad9fc0526e3ad11a097,5e89028e024b381dc84a13a3570decb4,delivered,2018-06-20 06:58:43,2018-06-20 07:19:05,2018-06-25 08:05:00,NaT,2018-07-16
22663,5cf925b116421afa85ee25e99b4c34fb,29c35fc91fc13fb5073c8f30505d860d,delivered,2017-02-18 16:48:35,NaT,2017-02-22 11:23:10,2017-03-09 07:28:47,2017-03-31
23156,12a95a3c06dbaec84bcfb0e2da5d228a,1e101e0daffaddce8159d25a8e53f2b2,delivered,2017-02-17 13:05:55,NaT,2017-02-22 11:23:11,2017-03-02 11:09:19,2017-03-20
26800,c1d4211b3dae76144deccd6c74144a88,684cb238dc5b5d6366244e0e0776b450,delivered,2017-01-19 12:48:08,NaT,2017-01-25 14:56:50,2017-01-30 18:16:01,2017-03-01
38290,d69e5d356402adc8cf17e08b5033acfb,68d081753ad4fe22fc4d410a9eb1ca01,delivered,2017-02-19 01:28:47,NaT,2017-02-23 03:11:48,2017-03-02 03:41:58,2017-03-27
39334,d77031d6a3c8a52f019764e68f211c69,0bf35cac6cc7327065da879e2d90fae8,delivered,2017-02-18 11:04:19,NaT,2017-02-23 07:23:36,2017-03-02 16:15:23,2017-03-22


In [515]:
len(delivered_s_null_vrijednostima)

23

### Zajednička analiza nedostajućih vremenskih podataka kod dostavljenih narudžbi

Postoje **23 jedinstvene narudžbe sa statusom `delivered` kojima nedostaje barem jedan od navedenih vremenskih podataka**.

14 dostavljenih narudžbi nema evidentirano vrijeme odobrenja (`order_approved_at`), 2 nemaju evidentirano vrijeme predaje dostavnoj službi (`order_delivered_carrier_date`), a 8 nema evidentiran stvarni datum dostave kupcu (`order_delivered_customer_date`).

Zbroj pojedinačnih slučajeva iznosi 24, dok postoje 23 jedinstvene narudžbe, jer jednoj narudžbi istodobno nedostaju `order_delivered_carrier_date` i `order_delivered_customer_date`.

Ove narudžbe neće se automatski uklanjati iz skladišta podataka jer sadrže druge relevantne informacije o poslovnom procesu. Međutim, prilikom izračuna izvedenih vremenskih mjera i provođenja statističkih analiza koristit će se samo zapisi koji sadrže sve vremenske atribute potrebne za konkretan izračun.

### 6.4. Analiza nedostajućih podataka o proizvodima

U skupu `products` uočeno je 610 proizvoda bez vrijednosti atributa `product_category_name`. Provjerava se nedostaju li tim proizvodima istodobno i ostali opisni atributi.

In [516]:
products[
    products["product_category_name"].isna()
][
    [
        "product_category_name",
        "product_name_lenght",
        "product_description_lenght",
        "product_photos_qty"
    ]
].isna().sum()

product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
dtype: int64

Analiza pokazuje da za svih 610 proizvoda kojima nedostaje kategorija istodobno nedostaju i podaci o duljini naziva, duljini opisa i broju fotografija.

Ovi proizvodi ne bi se trebali automatski ukloniti jer mogu sudjelovati u stvarnim prodajnim transakcijama. Zato će se postaviti vrijednost `Unknown` za nepoznatu kategoriju proizvoda.

In [517]:
products[
    products[
        [
            "product_weight_g",
            "product_length_cm",
            "product_height_cm",
            "product_width_cm"
        ]
    ].isna().any(axis=1)
]

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
8578,09ff539a621711667c43eba6a3bd8466,bebes,60.0,865.0,3.0,NaN,NaN,NaN,NaN
18851,5eb564652db742ff8f28759cd8d2652a,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Dva proizvoda nemaju evidentirane fizičke karakteristike (`product_weight_g`, `product_length_cm`, `product_height_cm` i `product_width_cm`).

Prvi proizvod pripada kategoriji `bebes` i sadrži ostale opisne podatke, ali mu nedostaju sve fizičke karakteristike. Drugi proizvod, osim fizičkih karakteristika, nema evidentiranu ni kategoriju proizvoda niti ostale opisne atribute.

Oba proizvoda zadržat će se u skladištu podataka jer mogu sudjelovati u stvarnim prodajnim transakcijama. 

Proizvodi će biti isključeni samo iz analiza koje zahtijevaju poznatu težinu ili dimenzije proizvoda, primjerice pri analizi povezanosti težine ili volumena proizvoda s troškom dostave.

## 7. Analiza potpunih duplikata

Provjerava se postojanje potpuno identičnih redaka u svakom izvornom skupu podataka. 

In [518]:
for name, df in dataframes.items():
    broj_duplikata = df.duplicated().sum()
    print(f"{name}: {broj_duplikata} potpunih duplikata")

customers: 0 potpunih duplikata
geolocation: 261831 potpunih duplikata
order_items: 0 potpunih duplikata
payments: 0 potpunih duplikata
reviews: 0 potpunih duplikata
orders: 0 potpunih duplikata
products: 0 potpunih duplikata
sellers: 0 potpunih duplikata
category_translation: 0 potpunih duplikata


Potpuni duplikati pronađeni su samo u skupu `geolocation`, gdje je identificiran 261.831 suvišan duplicirani zapis. 

### 7.1. Analiza i uklanjanje potpunih duplikata iz skupa `geolocation`

Izvorni skup `geolocation` sadrži 1.000.163 retka. Analizom je utvrđeno 261.831 potpuno dupliciranih redaka.

Budući da potpuno identični zapisi ne pružaju dodatnu informaciju, uklanjaju se iz skupa podataka. Za svaki skup identičnih zapisa zadržava se jedna pojava.

In [519]:
duplikati = geolocation[geolocation.duplicated(keep=False)]

duplikati = duplikati.sort_values(
    by=[
        "geolocation_zip_code_prefix",
        "geolocation_lat",
        "geolocation_lng",
        "geolocation_city",
        "geolocation_state"
    ]
)

duplikati

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
519,1001,-23.551337,-46.634027,sao paulo,SP
583,1001,-23.551337,-46.634027,sao paulo,SP
818,1001,-23.551337,-46.634027,sao paulo,SP
206,1001,-23.550498,-46.634338,sao paulo,SP
429,1001,-23.550498,-46.634338,sao paulo,SP
...,...,...,...,...,...
999899,99980,-28.387432,-51.847727,david canabarro,RS
999958,99980,-28.387059,-51.848964,david canabarro,RS
1000129,99980,-28.387059,-51.848964,david canabarro,RS
999775,99980,-28.386689,-51.847091,david canabarro,RS


In [520]:
print("Ukupan broj redaka:", len(geolocation))

print(
    "Broj redaka bez potpunih duplikata:",
    len(geolocation.drop_duplicates())
)

print(
    "Broj potpunih duplikata:",
    geolocation.duplicated().sum()
)

Ukupan broj redaka: 1000163
Broj redaka bez potpunih duplikata: 738332
Broj potpunih duplikata: 261831


In [521]:
print(
    "Broj jedinstvenih ZIP prefiksa:",
    geolocation["geolocation_zip_code_prefix"].nunique()
)

Broj jedinstvenih ZIP prefiksa: 19015


In [522]:
geolocation = geolocation.drop_duplicates()

In [523]:
print("Broj redaka nakon uklanjanja duplikata:", len(geolocation))
print("Broj preostalih potpunih duplikata:", geolocation.duplicated().sum())

Broj redaka nakon uklanjanja duplikata: 738332
Broj preostalih potpunih duplikata: 0


In [524]:
geolocation = geolocation.reset_index(drop=True)

Nakon uklanjanja potpunih duplikata skup `geolocation` sadrži **738.332 zapisa**.

Skup sadrži samo **19.015 jedinstvenih ZIP prefiksa**, što znači da se isti ZIP prefix može pojaviti u više različitih geolokacijskih zapisa.

Ti zapisi nisu potpuni duplikati jer se mogu razlikovati prema geografskim koordinatama ili drugim atributima. Zbog toga ih nije opravdano automatski ukloniti. 

## 8. Detaljna analiza pojedinih izvornih skupova podataka

Nakon početne analize kvalitete podataka detaljno se analizira svaki izvorni skup podataka. Cilj je utvrditi značenje i kvalitetu pojedinih atributa, identificirati primarne i strane ključeve, analizirati veze između skupova podataka te odrediti ulogu pojedinih atributa u budućem dimenzijskom modelu skladišta podataka.

### 8.1. Analiza skupa `geolocation`


In [525]:
print("Broj redaka:", len(geolocation))

print(
    "Broj jedinstvenih ZIP prefiksa:",
    geolocation["geolocation_zip_code_prefix"].nunique()
)

print(
    "Broj dupliciranih ZIP prefiksa:",
    geolocation["geolocation_zip_code_prefix"].duplicated().sum()
)

geolocation[
    [
        "geolocation_zip_code_prefix",
        "geolocation_lat",
        "geolocation_lng"
    ]
].describe()

Broj redaka: 738332
Broj jedinstvenih ZIP prefiksa: 19015
Broj dupliciranih ZIP prefiksa: 719317


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng
count,738332.000000,738332.000000,738332.000000
mean,38316.086800,-20.998353,-46.461098
std,30632.496675,5.892315,4.393705
min,1001.000000,-36.605374,-101.466766
25%,12600.000000,-23.603061,-48.867822
50%,29144.000000,-22.873588,-46.647278
75%,65950.000000,-19.923336,-43.836974
max,99990.000000,45.065933,121.105394


In [526]:
broj_lokacija_po_zipu = (
    geolocation
    .groupby("geolocation_zip_code_prefix")
    .size()
)

broj_lokacija_po_zipu.describe()

count    19015.000000
mean        38.828925
std         50.875735
min          1.000000
25%          8.000000
50%         23.000000
75%         49.000000
max        779.000000
dtype: float64

In [527]:
broj_koordinata_po_zipu = (
    geolocation
    .groupby("geolocation_zip_code_prefix")
    .apply(
        lambda x: x[
            ["geolocation_lat", "geolocation_lng"]
        ].drop_duplicates().shape[0]
    )
)

broj_koordinata_po_zipu.describe()

count    19015.000000
mean        37.872942
std         49.998113
min          1.000000
25%          8.000000
50%         22.000000
75%         48.000000
max        746.000000
dtype: float64

In [528]:
print(
    "ZIP prefiksi s jednom jedinstvenom koordinatom:",
    (broj_koordinata_po_zipu == 1).sum()
)

print(
    "ZIP prefiksi s više jedinstvenih koordinata:",
    (broj_koordinata_po_zipu > 1).sum()
)

print(
    "Maksimalan broj različitih koordinata za jedan ZIP prefiks:",
    broj_koordinata_po_zipu.max()
)

ZIP prefiksi s jednom jedinstvenom koordinatom: 1234
ZIP prefiksi s više jedinstvenih koordinata: 17781
Maksimalan broj različitih koordinata za jedan ZIP prefiks: 746


In [529]:
lokacije_po_zipu = (
    geolocation
    .groupby("geolocation_zip_code_prefix")
    .agg(
        broj_gradova=("geolocation_city", "nunique"),
        broj_drzava=("geolocation_state", "nunique")
    )
)

lokacije_po_zipu.describe()

,broj_gradova,broj_drzava
count,19015.000000,19015.000000
mean,1.467631,1.000421
std,0.538505,0.020508
min,1.000000,1.000000
25%,1.000000,1.000000
50%,1.000000,1.000000
75%,2.000000,1.000000
max,5.000000,2.000000


In [530]:
print(
    "ZIP prefiksi povezani s više gradova:",
    (lokacije_po_zipu["broj_gradova"] > 1).sum()
)

print(
    "ZIP prefiksi povezani s više saveznih država:",
    (lokacije_po_zipu["broj_drzava"] > 1).sum()
)

print(
    "Maksimalan broj gradova za isti ZIP prefiks:",
    lokacije_po_zipu["broj_gradova"].max()
)

print(
    "Maksimalan broj saveznih država za isti ZIP prefiks:",
    lokacije_po_zipu["broj_drzava"].max()
)

ZIP prefiksi povezani s više gradova: 8556
ZIP prefiksi povezani s više saveznih država: 8
Maksimalan broj gradova za isti ZIP prefiks: 5
Maksimalan broj saveznih država za isti ZIP prefiks: 2


In [531]:
lokacije_po_zipu[
    (lokacije_po_zipu["broj_gradova"] > 1) |
    (lokacije_po_zipu["broj_drzava"] > 1)
].sort_values(
    ["broj_drzava", "broj_gradova"],
    ascending=False
).head(30)

,broj_gradova,broj_drzava
geolocation_zip_code_prefix,,
72915,3,2
2116,2,2
4011,2,2
78557,2,2
80630,2,2
21550,1,2
23056,1,2
79750,1,2
6900,5,1


In [532]:
geo_lokacije = geolocation[
    [
        "geolocation_zip_code_prefix",
        "geolocation_city",
        "geolocation_state"
    ]
].drop_duplicates()

customer_geo_test = customers.merge(
    geo_lokacije,
    left_on=[
        "customer_zip_code_prefix",
        "customer_city",
        "customer_state"
    ],
    right_on=[
        "geolocation_zip_code_prefix",
        "geolocation_city",
        "geolocation_state"
    ],
    how="left"
)

print("Ukupno customer zapisa:", len(customer_geo_test))

print(
    "Pronađena lokacija:",
    customer_geo_test["geolocation_zip_code_prefix"].notna().sum()
)

print(
    "Nije pronađena lokacija:",
    customer_geo_test["geolocation_zip_code_prefix"].isna().sum()
)

Ukupno customer zapisa: 99441
Pronađena lokacija: 99123
Nije pronađena lokacija: 318


In [533]:
seller_geo_test = sellers.merge(
    geo_lokacije,
    left_on=[
        "seller_zip_code_prefix",
        "seller_city",
        "seller_state"
    ],
    right_on=[
        "geolocation_zip_code_prefix",
        "geolocation_city",
        "geolocation_state"
    ],
    how="left"
)

print("Ukupno seller zapisa:", len(seller_geo_test))

print(
    "Pronađena lokacija:",
    seller_geo_test["geolocation_zip_code_prefix"].notna().sum()
)

print(
    "Nije pronađena lokacija:",
    seller_geo_test["geolocation_zip_code_prefix"].isna().sum()
)

Ukupno seller zapisa: 3095
Pronađena lokacija: 2958
Nije pronađena lokacija: 137


### Zaključak analize geografskih podataka

Analizom skupa `geolocation` utvrđeno je da prefiks poštanskog broja nije jedinstven identifikator geografske lokacije. Pojedini prefiksi povezani su s više zapisa različitih geografskih koordinata, a uočeni su i slučajevi u kojima je isti prefiks povezan s različitim nazivima gradova ili saveznih država.

Takve karakteristike potrebno je uzeti u obzir pri povezivanju geografskih podataka jer izravno spajanje prema prefiksu poštanskog broja može dovesti do umnožavanja zapisa.

Budući da prostorna analiza na temelju geografskih koordinata nije predviđena u okviru ovog rada, odlučeno je da se skup `geolocation` neće koristiti pri izgradnji skladišta podataka. Za potrebe geografskog grupiranja i analize koristit će se prefiksi poštanskih brojeva, gradovi i savezne države iz izvornih skupova `customers` i `sellers`. Njihova standardizacija provest će se u fazi pripreme podataka.

### 8.2. Analiza skupa `customers`

Skup `customers` sadrži podatke o kupcima i njihovoj geografskoj lokaciji. Posebna pažnja posvećuje se razlikovanju atributa `customer_id` i `customer_unique_id`, budući da oni imaju različitu ulogu u identifikaciji kupaca i povezivanju narudžbi.

In [534]:
customers.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   customer_id               99441 non-null  str  
 1   customer_unique_id        99441 non-null  str  
 2   customer_zip_code_prefix  99441 non-null  int64
 3   customer_city             99441 non-null  str  
 4   customer_state            99441 non-null  str  
dtypes: int64(1), str(4)
memory usage: 3.8 MB


In [535]:
customers.describe(include="all")

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
count,99441,99441,99441.000000,99441,99441
unique,99441,96096,NaN,4119,27
top,06b8999e2fba1a1fbc88172c00ba8bc7,8d50f5eadf50201ccdcedfb9e2ac8455,NaN,sao paulo,SP
freq,1,17,NaN,15540,41746
mean,NaN,NaN,35137.474583,NaN,NaN
std,NaN,NaN,29797.938996,NaN,NaN
min,NaN,NaN,1003.000000,NaN,NaN
25%,NaN,NaN,11347.000000,NaN,NaN
50%,NaN,NaN,24416.000000,NaN,NaN
75%,NaN,NaN,58900.000000,NaN,NaN


#### Analiza atributa

Skup `customers` sadrži pet atributa. Atribut `customer_id` koristi se za povezivanje kupca s konkretnom narudžbom u skupu `orders`, dok `customer_unique_id` omogućuje prepoznavanje istog kupca kroz potencijalno više narudžbi.

Preostala tri atributa opisuju geografsku lokaciju kupca: ZIP prefix, grad i saveznu državu. Ovi atributi mogu se koristiti za geografsku analizu prodaje i ponašanja kupaca.

In [536]:
print("Broj redaka:", len(customers))
print("Broj jedinstvenih customer_id:", customers["customer_id"].nunique())
print("Broj dupliciranih customer_id:", customers["customer_id"].duplicated().sum())
print("Broj NULL customer_id:", customers["customer_id"].isna().sum())

Broj redaka: 99441
Broj jedinstvenih customer_id: 99441
Broj dupliciranih customer_id: 0
Broj NULL customer_id: 0


Atribut `customer_id` jedinstven je za svih 99.441 zapisa i ne sadrži nedostajuće vrijednosti. Stoga predstavlja primarni ključ izvornog skupa `customers`.


In [537]:
print("Broj customer_id:", customers["customer_id"].nunique())
print("Broj customer_unique_id:", customers["customer_unique_id"].nunique())

Broj customer_id: 99441
Broj customer_unique_id: 96096


In [538]:
broj_customer_id_po_kupcu = (
    customers
    .groupby("customer_unique_id")["customer_id"]
    .nunique()
)

broj_customer_id_po_kupcu.value_counts().sort_index()

customer_id
1     93099
2      2745
3       203
4        30
5         8
6         6
7         3
9         1
17        1
Name: count, dtype: int64

In [539]:
broj_customer_id_po_kupcu.describe()

count    96096.000000
mean         1.034809
std          0.214384
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max         17.000000
Name: customer_id, dtype: float64

In [540]:
kupci_s_vise_customer_id = broj_customer_id_po_kupcu[
    broj_customer_id_po_kupcu > 1
]

print(
    "Broj kupaca s više od jednog customer_id:",
    len(kupci_s_vise_customer_id)
)

kupci_s_vise_customer_id.head(10)

Broj kupaca s više od jednog customer_id: 2997


customer_unique_id
00172711b30d52eea8b313a7f2cced02    2
004288347e5e88a27ded2bb23747066c    2
004b45ec5c64187465168251cd1c9c2f    2
0058f300f57d7b93c477a131a59b36c3    2
00a39521eb40f7012db50455bf083460    2
00cc12a6d8b578b8ebd21ea4e2ae8b27    2
011575986092c30523ecb71ff10cb473    2
011b4adcd54683b480c4d841250a987f    2
012452d40dafae4df401bced74cdb490    2
012a218df8995d3ec3bb221828360c86    2
Name: customer_id, dtype: int64

In [541]:
customer_ids_u_orders = set(orders["customer_id"])
customer_ids_u_customers = set(customers["customer_id"])

nedostajuci_customer_ids = (
    customer_ids_u_orders - customer_ids_u_customers
)

print(
    "Broj customer_id iz orders koji ne postoje u customers:",
    len(nedostajuci_customer_ids)
)

Broj customer_id iz orders koji ne postoje u customers: 0


Provjerom referencijalnog integriteta između skupova `orders` i `customers` utvrđeno je da svaki `customer_id` prisutan u skupu `orders` postoji i u skupu `customers`. Time je potvrđena potpunost veze između narudžbi i pripadajućih zapisa o kupcima.

In [542]:
orders_po_customer_id = (
    orders
    .groupby("customer_id")["order_id"]
    .nunique()
)

orders_po_customer_id.value_counts().sort_index()

order_id
1    99441
Name: count, dtype: int64

In [543]:
print(
    "Broj saveznih država:",
    customers["customer_state"].nunique()
)

Broj saveznih država: 27


In [544]:
customers["customer_state"].value_counts()

customer_state
SP    41746
RJ    12852
MG    11635
RS     5466
PR     5045
SC     3637
BA     3380
DF     2140
ES     2033
GO     2020
PE     1652
CE     1336
PA      975
MT      907
MA      747
MS      715
PB      536
PI      495
RN      485
AL      413
SE      350
TO      280
RO      253
AM      148
AC       81
AP       68
RR       46
Name: count, dtype: int64

In [545]:
print(
    "Broj različitih gradova:",
    customers["customer_city"].nunique()
)

Broj različitih gradova: 4119


In [546]:
customers["customer_city"].value_counts().head(20)

customer_city
sao paulo                15540
rio de janeiro            6882
belo horizonte            2773
brasilia                  2131
curitiba                  1521
campinas                  1444
porto alegre              1379
salvador                  1245
guarulhos                 1189
sao bernardo do campo      938
niteroi                    849
santo andre                797
osasco                     746
santos                     713
goiania                    692
sao jose dos campos        691
fortaleza                  654
sorocaba                   633
recife                     613
florianopolis              570
Name: count, dtype: int64

In [547]:
print(
    "Broj različitih ZIP prefiksa kupaca:",
    customers["customer_zip_code_prefix"].nunique()
)

Broj različitih ZIP prefiksa kupaca: 14994


In [548]:
zip_state = (
    customers
    .groupby("customer_zip_code_prefix")["customer_state"]
    .nunique()
)

print(
    "Broj ZIP prefiksa povezanih s više država:",
    (zip_state > 1).sum()
)

Broj ZIP prefiksa povezanih s više država: 0


In [549]:
zip_city = (
    customers
    .groupby("customer_zip_code_prefix")["customer_city"]
    .nunique()
)

print(
    "Broj ZIP prefiksa povezanih s više gradova:",
    (zip_city > 1).sum()
)

Broj ZIP prefiksa povezanih s više gradova: 39


In [550]:
customer_zip_codes = set(
    customers["customer_zip_code_prefix"]
)

geolocation_zip_codes = set(
    geolocation["geolocation_zip_code_prefix"]
)

nedostajuci_zip_codes = (
    customer_zip_codes - geolocation_zip_codes
)

print(
    "Broj ZIP prefiksa kupaca kojih nema u geolocation:",
    len(nedostajuci_zip_codes)
)

Broj ZIP prefiksa kupaca kojih nema u geolocation: 157


In [551]:
customers_bez_geolokacije = customers[
    customers["customer_zip_code_prefix"].isin(
        nedostajuci_zip_codes
    )
]

print(
    "Broj zapisa kupaca bez odgovarajućeg ZIP-a u geolocation:",
    len(customers_bez_geolokacije)
)

Broj zapisa kupaca bez odgovarajućeg ZIP-a u geolocation: 278


#### Zaključak analize skupa `customers`

Analizom skupa `customers` utvrđeno je da sadrži 99.441 zapis i 96.096 jedinstvenih kupaca identificiranih atributom `customer_unique_id`. Atribut `customer_id` jedinstven je za svaki zapis u skupu `customers`, ne sadrži nedostajuće vrijednosti te predstavlja primarni ključ izvornog skupa podataka.

Provjerom povezanosti sa skupom `orders` utvrđeno je da svaki `customer_id` odgovara točno jednoj narudžbi te da svi identifikatori kupaca prisutni u skupu `orders` postoje u skupu `customers`. Time je potvrđen referencijalni integritet između ova dva izvorna skupa podataka.

Atribut `customer_id`, međutim, ne predstavlja trajni identifikator stvarnog kupca. Za tu svrhu koristi se `customer_unique_id`. Od ukupno 96.096 jedinstvenih kupaca, 2.997 kupaca povezano je s više od jednog `customer_id`. Većina kupaca ima samo jedan `customer_id`, dok najveći zabilježeni broj iznosi 17. Zbog toga će se za analize ponašanja kupaca kroz više narudžbi, kao što su broj kupnji, ukupna potrošnja, učestalost kupnje i segmentacija kupaca, koristiti `customer_unique_id`.

Geografska analiza pokazala je da podaci obuhvaćaju 27 saveznih država, 4.119 gradova i 14.994 različita ZIP prefiksa. Najveći broj zapisa kupaca dolazi iz savezne države Sao Paulo (SP), koja je znatno zastupljenija od ostalih država.

Provjerom konzistentnosti geografskih atributa utvrđeno je da nijedan ZIP prefix nije povezan s više saveznih država, dok je 39 ZIP prefiksa povezano s više različitih gradova. Ovi slučajevi zahtijevaju dodatnu analizu zajedno sa skupom `geolocation` prije konačne izgradnje geografske dimenzije.

Usporedbom sa skupom `geolocation` utvrđeno je da 157 ZIP prefiksa prisutnih u skupu `customers` nema odgovarajući zapis u skupu `geolocation`. Ti ZIP prefiksi odnose se na ukupno 278 zapisa kupaca. Za navedene zapise neće biti moguće izravno pridružiti geografsku širinu i dužinu na temelju podataka iz skupa `geolocation`, što će se uzeti u obzir prilikom ETL procesa i izgradnje geografske dimenzije.


In [552]:
lokacije_po_kupcu = (
    customers
    .groupby("customer_unique_id")
    .agg(
        broj_customer_id=("customer_id", "nunique"),
        broj_zip_prefiksa=("customer_zip_code_prefix", "nunique"),
        broj_gradova=("customer_city", "nunique"),
        broj_drzava=("customer_state", "nunique")
    )
)

lokacije_po_kupcu.head()

,broj_customer_id,broj_zip_prefiksa,broj_gradova,broj_drzava
customer_unique_id,,,,
0000366f3b9a7992bf8c76cfdf3221e2,1,1,1,1
0000b849f77a49e4a4ce2b2a4ca5be3f,1,1,1,1
0000f46a3911fa3c0805444483337064,1,1,1,1
0000f6ccb0745a6a4b88665a16c9f078,1,1,1,1
0004aac84e0df4da2b147fca70cf8255,1,1,1,1


In [553]:
print(
    "Kupci s više ZIP prefiksa:",
    (lokacije_po_kupcu["broj_zip_prefiksa"] > 1).sum()
)

print(
    "Kupci s više gradova:",
    (lokacije_po_kupcu["broj_gradova"] > 1).sum()
)

print(
    "Kupci s više saveznih država:",
    (lokacije_po_kupcu["broj_drzava"] > 1).sum()
)

Kupci s više ZIP prefiksa: 250
Kupci s više gradova: 122
Kupci s više saveznih država: 39


In [554]:
kupci_s_vise_lokacija = lokacije_po_kupcu[
    lokacije_po_kupcu["broj_zip_prefiksa"] > 1
]

kupci_s_vise_lokacija.sort_values(
    "broj_zip_prefiksa",
    ascending=False
).head(20)

,broj_customer_id,broj_zip_prefiksa,broj_gradova,broj_drzava
customer_unique_id,,,,
3e43e6105506432c953e165fb2acf44c,9,3,1,1
9832ae2f7d3e5fa4c7a1a06e9551bc61,3,3,2,1
b9badb100ff8ecc16a403111209e3a06,3,3,2,2
d44ccec15f5f86d14d6a2cfa67da1975,3,3,3,3
012452d40dafae4df401bced74cdb490,2,2,1,1
031ea691b99fc101dcad357d1a83573f,2,2,1,1
0451ef5a31cea3e14ce02d768d7c9943,2,2,1,1
0178b244a5c281fb2ade54038dd4b161,2,2,2,1
018b5a7502c30eb5f230f1b4eb23a156,2,2,1,1


In [555]:
customers[
    customers["customer_unique_id"].isin(
        kupci_s_vise_lokacija.index
    )
].sort_values(
    ["customer_unique_id", "customer_zip_code_prefix"]
).head(50)

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
87012,d95f60d70d9ea9a7fe37c53c931940bb,004b45ec5c64187465168251cd1c9c2f,57035,maceio,AL
72451,49cf243e0d353cd418ca77868e24a670,004b45ec5c64187465168251cd1c9c2f,57055,maceio,AL
61403,f530197ea86ced9488a03d055e118ebf,0058f300f57d7b93c477a131a59b36c3,40731,salvador,BA
36269,8ac44e9c15d396b8c3c7cbab0fff4536,0058f300f57d7b93c477a131a59b36c3,41370,salvador,BA
45307,f8e05a045de09e7855c9df3bb8815e0e,012452d40dafae4df401bced74cdb490,3220,sao paulo,SP
54038,e024c1515b1062b1e7e613f840c35ba3,012452d40dafae4df401bced74cdb490,3984,sao paulo,SP
91727,483468a56a54dbbbf8f2b2354cc8a729,0178b244a5c281fb2ade54038dd4b161,12518,guaratingueta,SP
57218,ea6ba2b2e27f7efea73bdeab7fd6e4a0,0178b244a5c281fb2ade54038dd4b161,14960,novo horizonte,SP
82312,026ca919d538ee454a6d43fc8bd01de0,018b5a7502c30eb5f230f1b4eb23a156,74936,aparecida de goiania,GO
64800,b98e0d5aca458a26a6ff30fee0416ca2,018b5a7502c30eb5f230f1b4eb23a156,74976,aparecida de goiania,GO


In [556]:
ponovljeni_kupci = lokacije_po_kupcu[
    lokacije_po_kupcu["broj_customer_id"] > 1
]

print("Ukupno ponovljenih kupaca:", len(ponovljeni_kupci))

print(
    "Ponovljeni kupci s više ZIP prefiksa:",
    (ponovljeni_kupci["broj_zip_prefiksa"] > 1).sum()
)

print(
    "Ponovljeni kupci s više gradova:",
    (ponovljeni_kupci["broj_gradova"] > 1).sum()
)

print(
    "Ponovljeni kupci s više saveznih država:",
    (ponovljeni_kupci["broj_drzava"] > 1).sum()
)

Ukupno ponovljenih kupaca: 2997
Ponovljeni kupci s više ZIP prefiksa: 250
Ponovljeni kupci s više gradova: 122
Ponovljeni kupci s više saveznih država: 39


#### Analiza promjene lokacije kupaca

Budući da atribut `customer_unique_id` omogućuje identifikaciju istog kupca kroz više narudžbi, dodatno je analizirano ostaje li geografska lokacija istog kupca nepromijenjena kroz različite zapise atributa `customer_id`.

Od ukupno 96.096 jedinstvenih kupaca, njih 2.997 povezano je s više od jednog atributa `customer_id`. Analizom geografskih atributa tih ponovljenih kupaca utvrđeno je da 250 kupaca ima više različitih ZIP prefiksa, 122 kupca imaju zapise povezane s više različitih gradova, dok 39 kupaca ima zapise povezane s više različitih saveznih država.

Rezultati pokazuju da geografska lokacija nije nužno trajno i nepromjenjivo svojstvo kupca identificiranog atributom `customer_unique_id`. Različiti `customer_id` zapisi istog kupca mogu sadržavati različite lokacije. U pojedinim slučajevima razlikuje se samo ZIP prefix unutar istog grada, dok u drugim slučajevima postoje razlike u gradu ili čak saveznoj državi.

Zbog toga se pri projektiranju dimenzijskog modela lokacija neće automatski vezati isključivo uz jedinstveni identitet kupca (`customer_unique_id`). Potrebno je očuvati informaciju o lokaciji povezanoj s konkretnim `customer_id`, odnosno konkretnom narudžbom, kako bi se omogućila ispravna geografska analiza pojedinih transakcija.


## 8.3. Analiza skupa `sellers`

In [557]:
print("Broj zapisa:", len(sellers))
print("Broj jedinstvenih prodavača:", sellers["seller_id"].nunique())

print("Nedostajući identifikatori:",
      sellers["seller_id"].isna().sum())

print("Duplicirani identifikatori:",
      sellers["seller_id"].duplicated().sum())

print("Potpuni duplikati:",
      sellers.duplicated().sum())

Broj zapisa: 3095
Broj jedinstvenih prodavača: 3095
Nedostajući identifikatori: 0
Duplicirani identifikatori: 0
Potpuni duplikati: 0


In [558]:
print("Broj različitih ZIP prefiksa:",
      sellers["seller_zip_code_prefix"].nunique())

print("Broj različitih gradova:",
      sellers["seller_city"].nunique())

print("Broj različitih saveznih država:",
      sellers["seller_state"].nunique())

print("\nNedostajuće vrijednosti:")
print(sellers[
    ["seller_zip_code_prefix", "seller_city", "seller_state"]
].isna().sum())

Broj različitih ZIP prefiksa: 2246
Broj različitih gradova: 611
Broj različitih saveznih država: 23

Nedostajuće vrijednosti:
seller_zip_code_prefix    0
seller_city               0
seller_state              0
dtype: int64


In [559]:
gradovi_drzave = (
    sellers.groupby("seller_city")["seller_state"]
    .nunique()
    .sort_values(ascending=False)
)

print(gradovi_drzave[gradovi_drzave > 1])

seller_city
marechal candido rondon    3
rio de janeiro             3
andradas                   2
ipira                      2
itajai                     2
blumenau                   2
caxias do sul              2
guaira                     2
jacutinga                  2
chapeco                    2
curitiba                   2
juiz de fora               2
palhoca                    2
belo horizonte             2
londrina                   2
florianopolis              2
pinhais                    2
pinhalzinho                2
porto alegre               2
rio bonito                 2
sao jose dos pinhais       2
vila velha                 2
volta redonda              2
Name: seller_state, dtype: int64


In [560]:
zip_sellers = set(sellers["seller_zip_code_prefix"].dropna())
zip_geo = set(geolocation["geolocation_zip_code_prefix"].dropna())

nedostajuci_zip = zip_sellers - zip_geo

print("Broj različitih nepokrivenih ZIP prefiksa:",
      len(nedostajuci_zip))

print("Broj prodavača s nepokrivenim ZIP prefiksom:",
      sellers["seller_zip_code_prefix"]
      .isin(nedostajuci_zip).sum())

Broj različitih nepokrivenih ZIP prefiksa: 7
Broj prodavača s nepokrivenim ZIP prefiksom: 7


In [561]:
seller_ids = set(sellers["seller_id"].dropna())
item_seller_ids = set(order_items["seller_id"].dropna())

print("Prodavači u skupu sellers:", len(seller_ids))

print("Prodavači prisutni u stavkama:",
      len(item_seller_ids))

print("Prodavači bez stavki narudžbi:",
      len(seller_ids - item_seller_ids))

print("Identifikatori prodavača bez zapisa u sellers:",
      len(item_seller_ids - seller_ids))

Prodavači u skupu sellers: 3095
Prodavači prisutni u stavkama: 3095
Prodavači bez stavki narudžbi: 0
Identifikatori prodavača bez zapisa u sellers: 0


In [562]:
aktivnost_prodavaca = (
    order_items.groupby("seller_id")
    .agg(
        broj_stavki=("order_item_id", "size"),
        broj_narudzbi=("order_id", "nunique")
    )
)

print(aktivnost_prodavaca.describe())

       broj_stavki  broj_narudzbi
count  3095.000000    3095.000000
mean     36.397415      32.313409
std     119.193461     105.139763
min       1.000000       1.000000
25%       2.000000       2.000000
50%       8.000000       6.000000
75%      24.000000      21.500000
max    2033.000000    1854.000000


## 8.4. Analiza skupa `products`

In [563]:
print("Broj zapisa:", len(products))
print("Broj jedinstvenih proizvoda:",
      products["product_id"].nunique())

print("Nedostajući identifikatori:",
      products["product_id"].isna().sum())

print("Duplicirani identifikatori:",
      products["product_id"].duplicated().sum())

print("Potpuni duplikati:",
      products.duplicated().sum())

Broj zapisa: 32951
Broj jedinstvenih proizvoda: 32951
Nedostajući identifikatori: 0
Duplicirani identifikatori: 0
Potpuni duplikati: 0


In [564]:
nedostajuce = (
    products.isna()
    .sum()
    .to_frame("broj_nedostajucih")
)

nedostajuce["udio_%"] = (
    nedostajuce["broj_nedostajucih"] / len(products) * 100
).round(2)

nedostajuce

,broj_nedostajucih,udio_%
product_id,0,0.00
product_category_name,610,1.85
product_name_lenght,610,1.85
product_description_lenght,610,1.85
product_photos_qty,610,1.85
product_weight_g,2,0.01
product_length_cm,2,0.01
product_height_cm,2,0.01
product_width_cm,2,0.01


In [565]:
opisni_atributi = [
    "product_category_name",
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty"
]

print(
    "Proizvodi kojima nedostaju sva četiri opisna atributa:",
    products[opisni_atributi].isna().all(axis=1).sum()
)

fizicki_atributi = [
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

print(
    "Proizvodi kojima nedostaju svi fizički atributi:",
    products[fizicki_atributi].isna().all(axis=1).sum()
)

Proizvodi kojima nedostaju sva četiri opisna atributa: 610
Proizvodi kojima nedostaju svi fizički atributi: 2


In [566]:
fizicki_atributi = [
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

for atribut in fizicki_atributi:
    print(f"\n{atribut}")
    print("Nulte vrijednosti:", (products[atribut] == 0).sum())
    print("Negativne vrijednosti:", (products[atribut] < 0).sum())
    print("Minimum:", products[atribut].min())
    print("Maksimum:", products[atribut].max())


product_weight_g
Nulte vrijednosti: 4
Negativne vrijednosti: 0
Minimum: 0.0
Maksimum: 40425.0

product_length_cm
Nulte vrijednosti: 0
Negativne vrijednosti: 0
Minimum: 7.0
Maksimum: 105.0

product_height_cm
Nulte vrijednosti: 0
Negativne vrijednosti: 0
Minimum: 2.0
Maksimum: 105.0

product_width_cm
Nulte vrijednosti: 0
Negativne vrijednosti: 0
Minimum: 6.0
Maksimum: 118.0


In [567]:
products[fizicki_atributi].describe().round(2)

,product_weight_g,product_length_cm,product_height_cm,product_width_cm
count,32949.00,32949.00,32949.00,32949.00
mean,2276.47,30.82,16.94,23.20
std,4282.04,16.91,13.64,12.08
min,0.00,7.00,2.00,6.00
25%,300.00,18.00,8.00,15.00
50%,700.00,25.00,13.00,20.00
75%,1900.00,38.00,21.00,30.00
max,40425.00,105.00,105.00,118.00


In [568]:
opisni_brojcani = [
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty"
]

products[opisni_brojcani].describe().round(2)

,product_name_lenght,product_description_lenght,product_photos_qty
count,32341.00,32341.00,32341.00
mean,48.48,771.50,2.19
std,10.25,635.12,1.74
min,5.00,4.00,1.00
25%,42.00,339.00,1.00
50%,51.00,595.00,1.00
75%,57.00,972.00,3.00
max,76.00,3992.00,20.00


In [569]:
for atribut in opisni_brojcani:
    print(
        atribut,
        "nultih vrijednosti:",
        (products[atribut] == 0).sum(),
        "negativnih vrijednosti:",
        (products[atribut] < 0).sum()
    )

product_name_lenght nultih vrijednosti: 0 negativnih vrijednosti: 0
product_description_lenght nultih vrijednosti: 0 negativnih vrijednosti: 0
product_photos_qty nultih vrijednosti: 0 negativnih vrijednosti: 0


In [570]:
print(
    "Broj različitih kategorija:",
    products["product_category_name"].nunique()
)

print(
    "Broj proizvoda bez kategorije:",
    products["product_category_name"].isna().sum()
)

print(
    "Broj proizvoda po kategoriji:"
)

products["product_category_name"].value_counts(
    dropna=False
).head(15)

Broj različitih kategorija: 73
Broj proizvoda bez kategorije: 610
Broj proizvoda po kategoriji:


product_category_name
cama_mesa_banho                3029
esporte_lazer                  2867
moveis_decoracao               2657
beleza_saude                   2444
utilidades_domesticas          2335
automotivo                     1900
informatica_acessorios         1639
brinquedos                     1411
relogios_presentes             1329
telefonia                      1134
bebes                           919
perfumaria                      868
papelaria                       849
fashion_bolsas_e_acessorios     849
cool_stuff                      789
Name: count, dtype: int64

In [571]:
kategorije_proizvoda = set(
    products["product_category_name"].dropna()
)

prevedene_kategorije = set(
    category_translation["product_category_name"].dropna()
)

print(
    "Kategorije bez prijevoda:",
    kategorije_proizvoda - prevedene_kategorije
)

Kategorije bez prijevoda: {'portateis_cozinha_e_preparadores_de_alimentos', 'pc_gamer'}


In [572]:
product_ids = set(products["product_id"].dropna())
item_product_ids = set(order_items["product_id"].dropna())

print("Proizvodi u skupu products:", len(product_ids))

print(
    "Različiti proizvodi u stavkama narudžbi:",
    len(item_product_ids)
)

print(
    "Proizvodi bez pripadajućih stavki:",
    len(product_ids - item_product_ids)
)

print(
    "Identifikatori proizvoda bez zapisa u products:",
    len(item_product_ids - product_ids)
)


Proizvodi u skupu products: 32951
Različiti proizvodi u stavkama narudžbi: 32951
Proizvodi bez pripadajućih stavki: 0
Identifikatori proizvoda bez zapisa u products: 0


In [573]:
usporedba_proizvoda = (
    products[["product_id"]]
    .merge(
        order_items[["product_id"]].drop_duplicates(),
        on="product_id",
        how="outer",
        indicator=True,
        validate="one_to_one"
    )
)

print("Rezultati povezivanja:")
print(usporedba_proizvoda["_merge"].value_counts())

Rezultati povezivanja:
_merge
both          32951
left_only         0
right_only        0
Name: count, dtype: int64


In [574]:
provjera_stavki = order_items.merge(
    products[["product_id"]],
    on="product_id",
    how="left",
    indicator=True,
    validate="many_to_one"
)

print(provjera_stavki["_merge"].value_counts())

print(
    "Broj stavki bez odgovarajućeg proizvoda:",
    (provjera_stavki["_merge"] == "left_only").sum()
)

_merge
both          112650
left_only          0
right_only         0
Name: count, dtype: int64
Broj stavki bez odgovarajućeg proizvoda: 0


In [575]:
print(
    "Stavke s nedostajućim product_id:",
    order_items["product_id"].isna().sum()
)

Stavke s nedostajućim product_id: 0


In [576]:
proizvodi_bez_kategorije = products.loc[
    products["product_category_name"].isna(),
    "product_id"
]

print(
    "Broj proizvoda bez kategorije koji se pojavljuju u stavkama:",
    order_items.loc[
        order_items["product_id"].isin(proizvodi_bez_kategorije),
        "product_id"
    ].nunique()
)

print(
    "Broj pripadajućih stavki narudžbi:",
    order_items["product_id"]
    .isin(proizvodi_bez_kategorije)
    .sum()
)

Broj proizvoda bez kategorije koji se pojavljuju u stavkama: 610
Broj pripadajućih stavki narudžbi: 1603


In [577]:
proizvodi_nulta_tezina = products[
    products["product_weight_g"] == 0
]

proizvodi_nulta_tezina[
    [
        "product_id",
        "product_category_name",
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm"
    ]
]

,product_id,product_category_name,product_weight_g,product_length_cm,product_height_cm,product_width_cm
9769,81781c0fed9fe1ad6e8c81fca1e1cb08,cama_mesa_banho,0.0,30.0,25.0,30.0
13683,8038040ee2a71048d4bdbbdc985b69ab,cama_mesa_banho,0.0,30.0,25.0,30.0
14997,36ba42dd187055e1fbe943b2d11430ca,cama_mesa_banho,0.0,30.0,25.0,30.0
32079,e673e90efa65a5409ff4196c038bb5af,cama_mesa_banho,0.0,30.0,25.0,30.0


In [578]:
proizvodi_nulta_tezina_stavke = order_items[
    order_items["product_id"].isin(
        proizvodi_nulta_tezina["product_id"]
    )
]

print("Broj stavki:", len(proizvodi_nulta_tezina_stavke))

print(
    proizvodi_nulta_tezina_stavke[
        ["product_id", "price", "freight_value"]
    ]
)

Broj stavki: 8
                             product_id  price  freight_value
2972   e673e90efa65a5409ff4196c038bb5af  129.9          23.71
2973   e673e90efa65a5409ff4196c038bb5af  129.9          23.71
3052   36ba42dd187055e1fbe943b2d11430ca  100.0          23.85
3053   36ba42dd187055e1fbe943b2d11430ca  100.0          23.85
14080  81781c0fed9fe1ad6e8c81fca1e1cb08  100.0          19.89
31488  e673e90efa65a5409ff4196c038bb5af  129.9          23.71
32984  8038040ee2a71048d4bdbbdc985b69ab  129.9          14.49
79374  e673e90efa65a5409ff4196c038bb5af  129.9          23.71


## 8.5. Analiza skupa `orders`

Skup `orders` predstavlja središnji skup podataka Olist baze te sadrži osnovne informacije o narudžbama i njihovom životnom ciklusu. Svaki zapis predstavlja jednu narudžbu identificiranu atributom `order_id`.

Skup omogućuje povezivanje narudžbe s kupcem putem atributa `customer_id`, dok atribut `order_status` opisuje trenutni status narudžbe. Preostali atributi predstavljaju vremenske oznake pojedinih faza obrade i dostave narudžbe.

Narudžba predstavlja jedan od središnjih poslovnih procesa koji će se analizirati. 

In [579]:
orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26


In [580]:
orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  str           
 1   customer_id                    99441 non-null  str           
 2   order_status                   99441 non-null  str           
 3   order_purchase_timestamp       99441 non-null  datetime64[us]
 4   order_approved_at              99281 non-null  datetime64[us]
 5   order_delivered_carrier_date   97658 non-null  datetime64[us]
 6   order_delivered_customer_date  96476 non-null  datetime64[us]
 7   order_estimated_delivery_date  99441 non-null  datetime64[us]
dtypes: datetime64[us](5), str(3)
memory usage: 6.1 MB


In [581]:
orders.describe(include="all")

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
count,99441,99441,99441,99441,99281,97658,96476,99441
unique,99441,99441,8,NaN,NaN,NaN,NaN,NaN
top,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,NaN,NaN,NaN,NaN,NaN
freq,1,1,96478,NaN,NaN,NaN,NaN,NaN
mean,NaN,NaN,NaN,2017-12-31 08:43:12.776581,2017-12-31 18:35:24.098800,2018-01-04 21:49:48.138278,2018-01-14 12:09:19.035542,2018-01-24 03:08:37.730111
min,NaN,NaN,NaN,2016-09-04 21:15:19,2016-09-15 12:16:38,2016-10-08 10:34:01,2016-10-11 13:46:32,2016-09-30 00:00:00
25%,NaN,NaN,NaN,2017-09-12 14:46:19,2017-09-12 23:24:16,2017-09-15 22:28:50.250000,2017-09-25 22:07:22.250000,2017-10-03 00:00:00
50%,NaN,NaN,NaN,2018-01-18 23:04:36,2018-01-19 11:36:13,2018-01-24 16:10:58,2018-02-02 19:28:10.500000,2018-02-15 00:00:00
75%,NaN,NaN,NaN,2018-05-04 15:42:16,2018-05-04 20:35:10,2018-05-08 13:37:45,2018-05-15 22:48:52.250000,2018-05-25 00:00:00
max,NaN,NaN,NaN,2018-10-17 17:30:18,2018-09-03 17:40:06,2018-09-11 19:48:28,2018-10-17 13:22:46,2018-11-12 00:00:00


### Analiza atributa

Skup `orders` sadrži osam atributa:

- `order_id` – jedinstveni identifikator narudžbe,
- `customer_id` – identifikator kupca povezanog s konkretnom narudžbom,
- `order_status` – status narudžbe,
- `order_purchase_timestamp` – vrijeme kreiranja odnosno kupnje narudžbe,
- `order_approved_at` – vrijeme odobrenja narudžbe,
- `order_delivered_carrier_date` – vrijeme predaje narudžbe dostavnoj službi,
- `order_delivered_customer_date` – vrijeme dostave narudžbe kupcu,
- `order_estimated_delivery_date` – procijenjeni datum dostave.

Vremenski atributi omogućuju praćenje životnog ciklusa narudžbe od trenutka kupnje do konačne dostave kupcu. Na temelju njih kasnije je moguće izvesti dodatne mjere, npr. vrijeme potrebno za odobrenje narudžbe, vrijeme obrade prije predaje dostavnoj službi, trajanje transporta, ukupno vrijeme dostave te odstupanje stvarnog datuma dostave od procijenjenog.

In [582]:
print("Broj redaka:", len(orders))
print("Broj jedinstvenih order_id:", orders["order_id"].nunique())
print("Broj dupliciranih order_id:", orders["order_id"].duplicated().sum())
print("Broj NULL order_id:", orders["order_id"].isna().sum())

Broj redaka: 99441
Broj jedinstvenih order_id: 99441
Broj dupliciranih order_id: 0
Broj NULL order_id: 0


Atribut `order_id` jedinstven je za svih 99.441 zapisa u skupu `orders` te ne sadrži duplicirane ni nedostajuće vrijednosti. Stoga predstavlja primarni ključ izvornog skupa `orders` i omogućuje jednoznačnu identifikaciju svake narudžbe.

Analizom odnosa između atributa `customer_id` i `orde_id` utvrđeno je da je svaki od 99.441 atributa `customer_id` povezan s točno jednom narudžbom. Ovaj rezultat potvrđuje prethodne zaključke dobivene analizom skupa `customers`.



In [583]:
orders["order_status"].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

In [584]:
orders["order_status"].value_counts(normalize=True).mul(100).round(2)

order_status
delivered      97.02
shipped         1.11
canceled        0.63
unavailable     0.61
invoiced        0.32
processing      0.30
created         0.01
approved        0.00
Name: proportion, dtype: float64

### Analiza statusa narudžbi

U skupu podataka pojavljuje se osam različitih statusa narudžbi. Većina narudžbi, njih 96.478 odnosno približno 97,02 %, ima status `delivered`, što označava uspješno dostavljene narudžbe.

Preostali dio skupa čine narudžbe u različitim fazama obrade ili narudžbe koje nisu uspješno završene. Status `shipped` ima 1.107 narudžbi, dok je 625 narudžbi otkazano (`canceled`), a 609 označeno kao nedostupno (`unavailable`). Manji broj narudžbi nalazi se u statusima `invoiced`, `processing`, `created` i `approved`.

Ovakva raspodjela važna je za daljnju analizu i izgradnju skladišta podataka. Za analize koje zahtijevaju informacije o stvarnom vremenu dostave moguće je koristiti samo narudžbe za koje je dostava završena i za koje postoji odgovarajuća vremenska oznaka. S druge strane, pri analizi ukupnog broja kreiranih narudžbi, otkazivanja ili neuspješnih narudžbi potrebno je zadržati i ostale statuse.

Stoga se narudžbe s drugim statusima neće automatski uklanjati iz skupa podataka. Njihova primjenjivost ovisit će o konkretnoj mjeri i vrsti analize koja se provodi u skladištu podataka.

In [585]:
print(
    "Najranija kupnja:",
    orders["order_purchase_timestamp"].min()
)

print(
    "Najkasnija kupnja:",
    orders["order_purchase_timestamp"].max()
)

Najranija kupnja: 2016-09-04 21:15:19
Najkasnija kupnja: 2018-10-17 17:30:18


In [586]:
orders["purchase_year"] = orders["order_purchase_timestamp"].dt.year

orders["purchase_year"].value_counts().sort_index()

purchase_year
2016      329
2017    45101
2018    54011
Name: count, dtype: int64

In [587]:
narudzbe_po_mjesecima = (
    orders
    .set_index("order_purchase_timestamp")
    .resample("ME")["order_id"]
    .count()
)

narudzbe_po_mjesecima

order_purchase_timestamp
2016-09-30       4
2016-10-31     324
2016-11-30       0
2016-12-31       1
2017-01-31     800
2017-02-28    1780
2017-03-31    2682
2017-04-30    2404
2017-05-31    3700
2017-06-30    3245
2017-07-31    4026
2017-08-31    4331
2017-09-30    4285
2017-10-31    4631
2017-11-30    7544
2017-12-31    5673
2018-01-31    7269
2018-02-28    6728
2018-03-31    7211
2018-04-30    6939
2018-05-31    6873
2018-06-30    6167
2018-07-31    6292
2018-08-31    6512
2018-09-30      16
2018-10-31       4
Freq: ME, Name: order_id, dtype: int64

### Vremenski raspon podataka o narudžbama

Najranija zabilježena kupnja izvršena 4. rujna 2016. godine, dok je najkasnija zabilježena kupnja izvršena 17. listopada 2018. godine. Skup podataka stoga formalno obuhvaća razdoblje od rujna 2016. do listopada 2018. godine.

Promatranjem broja narudžbi po godinama utvrđeno je da se 329 narudžbi odnosi na 2016. godinu, 45.101 narudžba na 2017. godinu, a 54.011 narudžbi na 2018. godinu.

Raspodjela narudžbi po mjesecima pokazuje da podaci nisu jednako zastupljeni tijekom cijelog promatranog razdoblja. U rujnu 2016. zabilježene su samo 4 narudžbe, u listopadu 324, u studenom nije zabilježena nijedna narudžba, dok je u prosincu zabilježena samo jedna narudžba. Zbog toga se 2016. godina ne može promatrati kao potpuna kalendarska godina podataka.

Prije provođenja vremenskih analiza i usporedbi prodaje između godina potrebno je detaljnije analizirati mjesečnu raspodjelu narudžbi, posebno početno i završno razdoblje skupa podataka. Nepotpune mjesece i godine potrebno je uzeti u obzir kako se razlike u broju narudžbi ne bi pogrešno protumačile kao stvarne promjene u poslovanju.

In [588]:
narudzbe_po_godinama_i_mjesecima = (
    orders
    .groupby(
        [
            orders["order_purchase_timestamp"].dt.year,
            orders["order_purchase_timestamp"].dt.month
        ]
    )["order_id"]
    .count()
    .unstack(fill_value=0)
)

narudzbe_po_godinama_i_mjesecima

order_purchase_timestamp,1,2,3,4,5,6,7,8,9,10,11,12
order_purchase_timestamp,,,,,,,,,,,,
2016,0,0,0,0,0,0,0,0,4,324,0,1
2017,800,1780,2682,2404,3700,3245,4026,4331,4285,4631,7544,5673
2018,7269,6728,7211,6939,6873,6167,6292,6512,16,4,0,0


In [589]:
orders[
    orders["order_purchase_timestamp"].dt.year == 2018
].groupby(
    orders["order_purchase_timestamp"].dt.month
)["order_id"].count()

order_purchase_timestamp
1     7269
2     6728
3     7211
4     6939
5     6873
6     6167
7     6292
8     6512
9       16
10       4
Name: order_id, dtype: int64

#### Analiza mjesečne pokrivenosti podataka

Detaljnijom analizom broja narudžbi po godinama i mjesecima utvrđeno je da skup podataka, iako formalno obuhvaća razdoblje od rujna 2016. do listopada 2018. godine, ne sadrži jednako potpunu vremensku pokrivenost tijekom cijelog tog razdoblja.

Podaci za 2016. godinu vrlo su ograničeni. U rujnu su zabilježene samo 4 narudžbe, u listopadu 324, u studenom nije zabilježena nijedna narudžba, dok je u prosincu zabilježena samo jedna narudžba.

Tijekom 2017. godine podaci su prisutni kontinuirano u svim mjesecima, pri čemu broj narudžbi raste od 800 narudžbi u siječnju do nekoliko tisuća narudžbi mjesečno tijekom ostatka godine.

U 2018. godini kontinuiran veći broj narudžbi prisutan je od siječnja do kolovoza, s približno 6.000 do 7.000 narudžbi mjesečno. Nakon toga dolazi do naglog pada na samo 16 narudžbi u rujnu i 4 narudžbe u listopadu, dok za studeni i prosinac nema zabilježenih narudžbi.

Na temelju toga može se zaključiti da se razdoblje od siječnja 2017. do kolovoza 2018. može smatrati glavnim kontinuiranim razdobljem podataka. Podaci izvan tog razdoblja neće se nužno uklanjati iz skladišta podataka, ali će se njihova nepotpuna vremenska pokrivenost uzeti u obzir pri provođenju statističkih analiza i usporedbi vremenskih razdoblja. Posebno treba izbjegavati izravnu usporedbu ukupnog broja narudžbi između 2016., 2017. i 2018. godine bez uzimanja u obzir činjenice da 2016. i 2018. ne predstavljaju potpune kalendarske godine.

In [590]:
orders[
    orders["order_purchase_timestamp"] >= "2018-09-01"
][
    [
        "order_id",
        "order_status",
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]
].sort_values("order_purchase_timestamp")

,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
62360,54282e97f61c23b78330c15b154c867d,shipped,2018-09-03 09:06:57,2018-09-03 17:40:06,2018-09-04 15:25:00,NaT,2018-09-06
39919,4637ca194b6387e2d538dc89b124b0ee,canceled,2018-09-03 14:14:25,NaT,NaT,NaT,2018-09-10
24410,0d3adebce4bebc1f80a7f36e9833f497,canceled,2018-09-03 18:40:50,NaT,NaT,NaT,2018-10-02
73341,cfdfd7862e532c4ec1ed1c6a1b56d320,canceled,2018-09-03 18:45:34,NaT,NaT,NaT,2018-09-14
53501,afb61112fb99b07fe17e591c68c0c84c,canceled,2018-09-06 10:48:12,NaT,NaT,NaT,2018-09-19
20778,4d59cc2a2e7bb6c0a851725f5888a9b5,canceled,2018-09-06 14:50:07,NaT,NaT,NaT,2018-09-21
99347,a89abace0dcc01eeb267a9660b5ac126,canceled,2018-09-06 18:45:47,NaT,NaT,NaT,2018-09-27
31170,21a00b08cbeb5716bbb66105e3dbd850,canceled,2018-09-10 17:51:25,NaT,NaT,NaT,2018-09-20
9976,87ae60ef8b08ae0e5f903cacb53a6904,canceled,2018-09-11 16:45:54,NaT,NaT,NaT,2018-09-26
80019,1ad22e16129698e1a3cc11e4350d2cb7,canceled,2018-09-12 15:32:16,NaT,NaT,NaT,2018-09-26


In [591]:
orders[
    orders["order_purchase_timestamp"] >= "2018-09-01"
]["order_status"].value_counts()

order_status
canceled    19
shipped      1
Name: count, dtype: int64

#### Analiza vremenskog obuhvata podataka

Ističe se razdoblje nakon kolovoza 2018. u kojem je zabilježeno samo 20 narudžbi: 16 u rujnu i 4 u listopadu.

Dodatnom analizom statusa tih narudžbi utvrđeno je da je 19 od 20 narudžbi otkazano (canceled), dok jedna narudžba ima status shipped. Nijedna narudžba iz tog razdoblja nema status delivered. Stoga se razdoblje nakon kolovoza 2018. ne može smatrati potpunim i reprezentativnim razdobljem poslovanja.

Izvorni zapisi neće se automatski uklanjati iz skupa podataka jer predstavljaju valjane zapise o narudžbama. Međutim, pri analizama vremenskih trendova potrebno je uzeti u obzir nepotpunost podataka na rubovima vremenskog razdoblja. Za usporedbu mjesečnih trendova kao potpuno razdoblje promatrat će se razdoblje do kolovoza 2018., dok će se podaci iz rujna i listopada 2018. tretirati zasebno.

In [592]:
orders[
    orders["order_purchase_timestamp"].dt.year == 2016
]["order_status"].value_counts()

order_status
delivered      267
canceled        26
invoiced        18
shipped          9
unavailable      7
processing       2
Name: count, dtype: int64

In [593]:
statusi_po_mjesecima_2016 = (
    orders[
        orders["order_purchase_timestamp"].dt.year == 2016
    ]
    .groupby(
        [
            orders["order_purchase_timestamp"].dt.month,
            "order_status"
        ]
    )["order_id"]
    .count()
    .unstack(fill_value=0)
)

statusi_po_mjesecima_2016

order_status,canceled,delivered,invoiced,processing,shipped,unavailable
order_purchase_timestamp,,,,,,
9,2,1,0,0,1,0
10,24,265,18,2,8,7
12,0,1,0,0,0,0


#### Zaključak analize vremenskog obuhvata

Analizom podataka utvrđeno je da skup `orders` vremenski obuhvaća razdoblje od rujna 2016. do listopada 2018. godine. Podaci na početku i kraju promatranog razdoblja nisu jednako zastupljeni kao tijekom središnjeg dijela skupa.

Tijekom 2016. godine zabilježeno je 329 narudžbi, od kojih se čak 324 odnose na listopad. Od 324 listopadske narudžbe njih 265 ima status `delivered`, što upućuje na stvarne zapise o izvršenim narudžbama, zbog čega se ti podaci neće uklanjati iz skupa. Ipak, zbog nepotpune zastupljenosti mjeseci 2016. godina nije prikladna za izravnu usporedbu godišnjih trendova s 2017. godinom.

Nakon kolovoza 2018. zabilježeno je samo 20 narudžbi, od kojih je 19 otkazano, a jedna ima status `shipped`.

Stoga će se svi zapisi zadržati u skupu podataka, dok će se za analize koje zahtijevaju kontinuirano i usporedivo vremensko razdoblje prvenstveno promatrati razdoblje od siječnja 2017. do kolovoza 2018. godine. Rubna razdoblja iz 2016. te rujna i listopada 2018. analizirat će se zasebno ili će se izuzeti iz analiza vremenskih trendova kada bi njihova nepotpunost mogla dovesti do pogrešnih zaključaka.

### Analiza vremenskog tijeka narudžbe

Skup `orders` sadrži nekoliko vremenskih atributa koji opisuju različite faze životnog ciklusa narudžbe:

- `order_purchase_timestamp` – vrijeme kada je kupac izvršio narudžbu,
- `order_approved_at` – vrijeme kada je narudžba zaprimljena i odobrena,
- `order_delivered_carrier_date` – vrijeme kada je narudžba predana dostavnoj službi,
- `order_delivered_customer_date` – vrijeme kada je narudžba dostavljena kupcu,
- `order_estimated_delivery_date` – procijenjeni datum dostave.

Na temelju ovih atributa moguće je analizirati trajanje pojedinih faza procesa narudžbe te usporediti stvarno i procijenjeno vrijeme dostave. Prije izračuna vremenskih intervala potrebno je provjeriti kronološku konzistentnost vremenskih zapisa.

In [594]:
orders[
    orders["order_approved_at"]
    < orders["order_purchase_timestamp"]
][
    [
        "order_id",
        "order_status",
        "order_purchase_timestamp",
        "order_approved_at"
    ]
]

,order_id,order_status,order_purchase_timestamp,order_approved_at


In [595]:
print(
    "Broj narudžbi odobrenih prije vremena kupnje:",
    (
        orders["order_approved_at"]
        < orders["order_purchase_timestamp"]
    ).sum()
)

Broj narudžbi odobrenih prije vremena kupnje: 0


In [596]:
print(
    "Broj narudžbi predanih prijevozniku prije odobrenja:",
    (
        orders["order_delivered_carrier_date"]
        < orders["order_approved_at"]
    ).sum()
)

Broj narudžbi predanih prijevozniku prije odobrenja: 1359


In [597]:
orders[
    orders["order_delivered_carrier_date"]
    < orders["order_approved_at"]
][
    [
        "order_id",
        "order_status",
        "order_approved_at",
        "order_delivered_carrier_date"
    ]
]

,order_id,order_status,order_approved_at,order_delivered_carrier_date
15,dcb36b511fcac050b97cd5c05de84dc3,delivered,2018-06-12 23:31:02,2018-06-11 14:54:00
64,688052146432ef8253587b930b01a06d,delivered,2018-04-24 18:25:22,2018-04-23 19:19:14
199,58d4c4747ee059eeeb865b349b41f53a,delivered,2018-07-26 23:31:53,2018-07-24 12:57:00
210,412fccb2b44a99b36714bca3fef8ad7b,delivered,2018-07-23 12:31:53,2018-07-23 12:24:00
415,56a4ac10a4a8f2ba7693523bb439eede,delivered,2018-07-27 23:31:09,2018-07-24 14:03:00
...,...,...,...,...
99091,240ead1a7284667e0ec71d01f80e4d5e,delivered,2018-07-05 16:17:59,2018-07-05 14:11:00
99230,78008d03bd8ef7fcf1568728b316553c,delivered,2018-07-05 16:32:52,2018-07-03 12:57:00
99266,76a948cd55bf22799753720d4545dd2d,delivered,2018-02-04 23:31:46,2018-01-31 18:11:58
99377,a6bd1f93b7ff72cc348ca07f38ec4bee,delivered,2018-04-24 19:26:10,2018-04-23 17:18:40


In [598]:
print(
    "Broj narudžbi dostavljenih kupcu prije predaje prijevozniku:",
    (
        orders["order_delivered_customer_date"]
        < orders["order_delivered_carrier_date"]
    ).sum()
)

Broj narudžbi dostavljenih kupcu prije predaje prijevozniku: 23


In [599]:
orders[
    orders["order_delivered_customer_date"]
    < orders["order_delivered_carrier_date"]
][
    [
        "order_id",
        "order_status",
        "order_delivered_carrier_date",
        "order_delivered_customer_date"
    ]
]

,order_id,order_status,order_delivered_carrier_date,order_delivered_customer_date
6437,a1abeb653a4d4cd1e142ccb8c82cd069,delivered,2017-07-28 16:57:58,2017-07-25 19:32:56
9553,383aa8b2724fe452d9ccd9934a8c628b,delivered,2017-07-07 17:22:41,2017-07-06 14:27:51
13487,cb1134f9010d242e9515ad1c78ec0c39,delivered,2017-07-20 19:22:02,2017-07-19 14:13:28
14474,dceb62e8fa94b46006c9554fed743df0,delivered,2017-08-01 18:23:30,2017-07-26 18:09:10
19268,5f9d46795c3126674e52becb3a1a517f,delivered,2017-07-20 23:03:42,2017-07-20 18:52:41
21338,8c78d01de3a9009e23d6877a7cc9be20,delivered,2016-10-26 11:41:53,2016-10-25 17:51:46
22520,b27af682321527a6349f1761eb3f360c,delivered,2017-06-27 14:51:54,2017-06-26 15:45:35
25393,1cc3ae63caffff2d6c3ee3e78e074acf,delivered,2017-08-10 18:28:56,2017-08-10 18:05:38
25646,e37f11cae9985ca58f0b56f268720537,delivered,2017-08-01 18:17:47,2017-07-31 17:49:56
27470,fa3e37584f4fdb1ded0e0de700dfcb4e,delivered,2017-08-09 18:18:43,2017-08-01 21:13:01


In [600]:
delivered_orders = orders[
    orders["order_status"] == "delivered"
].copy()

print("Broj dostavljenih narudžbi:", len(delivered_orders))

print(
    "Broj dostavljenih narudžbi s poznatim datumom stvarne dostave:",
    delivered_orders["order_delivered_customer_date"].notna().sum()
)

print(
    "Broj dostavljenih narudžbi s poznatim procijenjenim datumom dostave:",
    delivered_orders["order_estimated_delivery_date"].notna().sum()
)

Broj dostavljenih narudžbi: 96478
Broj dostavljenih narudžbi s poznatim datumom stvarne dostave: 96470
Broj dostavljenih narudžbi s poznatim procijenjenim datumom dostave: 96478


#### Zaključak: 

Analiza vremenske konzistentnosti pokazala je da ne postoje narudžbe koje su odobrene prije evidentiranog vremena kupnje. 

Pronađeno je 1359 narudžbi kod kojih je datum predaje dostavljaču raniji od datuma odobrenja narudžbe te 23 narudžbe kod kojih je evidentirani datum dostave kupcu raniji od datuma predaje dostavljaču. Ovi zapisi zahtijevaju dodatnu analizu prije donošenja odluke o njihovom tretmanu tijekom čišćenja podataka. Od ukupno 96 478 narudžbi sa statusom `delivered`, njih 96 470 ima evidentiran stvarni datum dostave, dok svih 96 478 ima poznat procijenjeni datum dostave.

In [601]:
# Razlika između odobrenja i predaje dostavljaču
problem_carrier = orders[
    orders["order_delivered_carrier_date"] < orders["order_approved_at"]
].copy()

problem_carrier["razlika_sati"] = (
    problem_carrier["order_approved_at"]
    - problem_carrier["order_delivered_carrier_date"]
).dt.total_seconds() / 3600

problem_carrier["razlika_sati"].describe()

count    1359.000000
mean       24.751987
std       115.307793
min         0.005833
25%         1.415417
50%        17.167778
75%        25.956667
max      4109.256111
Name: razlika_sati, dtype: float64

In [602]:
# Razlika između predaje dostavljaču i dostave kupcu
problem_delivery = orders[
    orders["order_delivered_customer_date"] <
    orders["order_delivered_carrier_date"]
].copy()

problem_delivery["razlika_sati"] = (
    problem_delivery["order_delivered_carrier_date"]
    - problem_delivery["order_delivered_customer_date"]
).dt.total_seconds() / 3600

problem_delivery["razlika_sati"].describe()

count     23.000000
mean      78.455133
std       89.308070
min        0.388333
25%       23.338333
50%       39.864444
75%      128.759028
max      386.308056
Name: razlika_sati, dtype: float64

In [603]:
problem_carrier[
    [
        "order_id",
        "order_status",
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "razlika_sati"
    ]
].sort_values("razlika_sati", ascending=False).head(20)

,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,razlika_sati
25883,7c48bb55e8e4f7e56d412e9653db37bc,delivered,2018-07-16 18:40:53,2018-07-16 18:50:22,2018-01-26 13:35:00,4109.256111
14562,1fab4ac9d85079b3da72a11475ae1685,delivered,2017-09-01 19:04:22,2017-09-13 22:06:11,2017-09-04 13:10:23,224.930000
46163,0184d4ddb259e1a4cfc2871888cf97b8,delivered,2017-09-01 20:04:28,2017-09-13 22:17:15,2017-09-04 14:05:50,224.190278
98710,1378f9601350615613cc8832d6789c5d,delivered,2017-09-01 20:28:02,2017-09-13 22:03:51,2017-09-04 18:07:55,219.932222
41592,8554cb37f7158cb0b082a841d24a4589,delivered,2017-09-01 18:40:44,2017-09-13 21:58:04,2017-09-04 19:12:19,218.762500
11738,cf72398d0690f841271b695bbfda82d2,delivered,2017-09-01 18:45:33,2017-09-13 22:04:39,2017-09-04 20:12:41,217.866111
85393,40de47dfa620d667117e4a6067b6e1ec,delivered,2017-09-01 20:05:55,2017-09-13 21:58:38,2017-09-04 20:36:58,217.361111
31211,bc4854efd86d9f42140c951c595d20c1,delivered,2017-09-01 20:05:42,2017-09-13 22:00:51,2017-09-04 20:49:57,217.181667
55302,77ca435b03fbf991e5027e3776e37885,delivered,2017-09-01 18:49:54,2017-09-13 21:56:08,2017-09-04 20:50:00,217.102222
68315,580f3268bc2075c5961b2d2929c7a35b,delivered,2017-09-01 20:17:09,2017-09-13 22:03:42,2017-09-05 11:58:18,202.090000


In [604]:
problem_delivery[
    [
        "order_id",
        "order_status",
        "order_purchase_timestamp",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "razlika_sati"
    ]
].sort_values("razlika_sati", ascending=False).head(20)

,order_id,order_status,order_purchase_timestamp,order_delivered_carrier_date,order_delivered_customer_date,razlika_sati
34939,c1e2bf2b7dd3309f2f5356c6b63968fa,delivered,2017-02-10 10:19:10,2017-03-02 17:34:26,2017-02-14 15:15:57,386.308056
27470,fa3e37584f4fdb1ded0e0de700dfcb4e,delivered,2017-07-30 19:32:23,2017-08-09 18:18:43,2017-08-01 21:13:01,189.095000
45302,29941903985f944b0ffc49c479c1547d,delivered,2017-05-29 16:16:50,2017-06-09 15:07:29,2017-06-02 11:09:16,171.970278
14474,dceb62e8fa94b46006c9554fed743df0,delivered,2017-07-20 20:58:05,2017-08-01 18:23:30,2017-07-26 18:09:10,144.238889
49933,76458889992169d3135b264dc13aec67,delivered,2016-10-07 10:05:16,2016-10-26 11:43:06,2016-10-20 18:03:17,137.663611
71227,19feb5627c41ea1b36a8e50a469b3644,delivered,2016-10-07 17:09:56,2016-10-26 11:42:05,2016-10-20 19:07:54,136.569722
74967,d5558a097766363b8e76b38c43332e8a,delivered,2017-02-04 19:01:33,2017-02-15 08:55:26,2017-02-10 07:58:32,120.948333
41636,b866af202be0692766081310cd4085e1,delivered,2017-01-27 14:59:17,2017-02-20 02:32:08,2017-02-15 03:53:46,118.639444
6437,a1abeb653a4d4cd1e142ccb8c82cd069,delivered,2017-07-20 11:20:52,2017-07-28 16:57:58,2017-07-25 19:32:56,69.417222
78556,ea1dcb4757a844d2642547797bd5feb0,delivered,2017-07-18 13:38:29,2017-07-27 19:21:31,2017-07-25 19:43:10,47.639167


Postoje 1359 narudžbe kod kojih je datum predaje narudžbe prijevozniku (`order_delivered_carrier_date`) raniji od datuma odobrenja narudžbe (`order_approved_at`). Medijan vremenskog odstupanja iznosi približno 17,17 sati, dok prosjek iznosi 24,75 sati. Većina odstupanja relativno je mala, ali postoje i izraziti ekstremni slučajevi. Najveće zabilježeno odstupanje iznosi približno 4109 sati, odnosno 171 dan, pri čemu je datum predaje prijevozniku evidentiran nekoliko mjeseci prije samog nastanka i odobrenja narudžbe. Što napraviti s tim zapisima?


Postoje 23 narudžbe kod kojih je evidentirani datum dostave kupcu (`order_delivered_customer_date`) raniji od datuma predaje narudžbe prijevozniku (`order_delivered_carrier_date`). Medijan odstupanja iznosi približno 39,86 sati, dok najveće odstupanje iznosi približno 386 sati, odnosno više od 16 dana. 

Takav redoslijed događaja nije u skladu s očekivanim tijekom procesa dostave, pa ovi zapisi predstavljaju vremenski nekonzistentne podatke.

In [605]:
delivered_orders = orders[
    (orders["order_status"] == "delivered") &
    (orders["order_delivered_customer_date"].notna()) &
    (orders["order_estimated_delivery_date"].notna())
].copy()

print("Broj narudžbi za analizu:", len(delivered_orders))

Broj narudžbi za analizu: 96470


In [606]:
delivered_orders["odstupanje_dostave_dani"] = (
    delivered_orders["order_delivered_customer_date"]
    - delivered_orders["order_estimated_delivery_date"]
).dt.total_seconds() / (24 * 3600)

delivered_orders["odstupanje_dostave_dani"].describe()

count    96470.000000
mean       -11.178126
std         10.184354
min       -146.016123
25%        -16.244065
50%        -11.948102
75%         -6.389815
max        188.975081
Name: odstupanje_dostave_dani, dtype: float64

In [607]:
stvarni_datum = delivered_orders[
    "order_delivered_customer_date"
].dt.normalize()

procijenjeni_datum = delivered_orders[
    "order_estimated_delivery_date"
].dt.normalize()

broj_prije_roka = (stvarni_datum < procijenjeni_datum).sum()
broj_na_dan_roka = (stvarni_datum == procijenjeni_datum).sum()
broj_nakon_roka = (stvarni_datum > procijenjeni_datum).sum()

print("Dostavljeno prije procijenjenog datuma:", broj_prije_roka)
print("Dostavljeno na procijenjeni datum:", broj_na_dan_roka)
print("Dostavljeno nakon procijenjenog datuma:", broj_nakon_roka)

Dostavljeno prije procijenjenog datuma: 88644
Dostavljeno na procijenjeni datum: 1292
Dostavljeno nakon procijenjenog datuma: 6534


In [608]:
ukupno = len(delivered_orders)

print(
    "Prije roka:",
    round(broj_prije_roka / ukupno * 100, 2),
    "%"
)

print(
    "Na procijenjeni datum:",
    round(broj_na_dan_roka / ukupno * 100, 2),
    "%"
)

print(
    "Nakon roka:",
    round(broj_nakon_roka / ukupno * 100, 2),
    "%"
)

Prije roka: 91.89 %
Na procijenjeni datum: 1.34 %
Nakon roka: 6.77 %


In [609]:
zakasnjele_narudzbe = delivered_orders[
    stvarni_datum > procijenjeni_datum
].copy()

zakasnjele_narudzbe["kasnjenje_dani"] = (
    zakasnjele_narudzbe["order_delivered_customer_date"].dt.normalize()
    - zakasnjele_narudzbe["order_estimated_delivery_date"].dt.normalize()
).dt.days

zakasnjele_narudzbe["kasnjenje_dani"].describe()

count    6534.000000
mean       10.620141
std        14.644955
min         1.000000
25%         3.000000
50%         7.000000
75%        13.000000
max       188.000000
Name: kasnjenje_dani, dtype: float64

In [610]:
zakasnjele_narudzbe[
    [
        "order_id",
        "order_purchase_timestamp",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
        "kasnjenje_dani"
    ]
].sort_values(
    "kasnjenje_dani",
    ascending=False
).head(20)

,order_id,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date,kasnjenje_dani
55619,1b3190b2dfa9d789e1f14c05b647a14a,2018-02-23 14:57:35,2018-09-19 23:24:07,2018-03-15,188
19590,ca07593549f1816d26a572e06dc1eab6,2017-02-21 23:31:27,2017-09-19 14:36:39,2017-03-22,181
11399,47b40429ed8cce3aee9199792275433f,2018-01-03 09:44:01,2018-07-13 20:51:31,2018-01-19,175
81401,2fe324febf907e3ea3f2aa9650869fa5,2017-03-13 20:17:10,2017-09-19 17:00:07,2017-04-05,167
89130,285ab9426d6982034523a855f55a885e,2017-03-08 22:47:40,2017-09-19 14:00:04,2017-04-06,166
61610,440d0d17af552815d15a9e41abe49359,2017-03-07 23:59:51,2017-09-19 15:12:50,2017-04-07,165
68769,c27815f7e3dd0b926b58552628481575,2017-03-15 23:23:17,2017-09-19 17:14:25,2017-04-10,162
40847,d24e8541128cea179a11a65176e0a96f,2017-06-12 13:14:11,2017-12-04 18:36:29,2017-06-26,161
38509,0f4519c5f1c541ddec9f21b3bddd533a,2017-03-09 13:26:57,2017-09-19 14:38:21,2017-04-11,161
54480,2d7561026d542c8dbd8f0daeadf67a43,2017-03-15 11:24:27,2017-09-19 14:38:18,2017-04-13,159


#### Zaključak:

Analiza pravovremenosti dostave provedena je na 96.470 narudžbi sa statusom `delivered` za koje su poznati stvarni i procijenjeni datum dostave. Većina narudžbi (91,89 %) dostavljena je prije procijenjenog datuma, dok je 1,34 % dostavljeno na procijenjeni datum. Ukupno 6,77 % narudžbi dostavljeno je nakon procijenjenog datuma. Kod zakašnjelih narudžbi medijan kašnjenja iznosi 7 dana, a prosječno kašnjenje približno 10,62 dana. Maksimalno zabilježeno kašnjenje iznosi 188 dana, što ukazuje na postojanje ekstremnih vrijednosti koje će biti potrebno dodatno razmotriti tijekom čišćenja i statističke analize podataka. Rezultati pokazuju da se iz vremenskih atributa mogu izvesti nove mjere, poput broja dana kašnjenja i indikatora zakašnjele dostave, koje mogu biti korisne u procesu izgradnje skladišta podataka i kasnijoj analizi čimbenika povezanih s pravovremenošću dostave.

In [611]:
status_datumi = (
    orders
    .groupby("order_status")
    .agg(
        broj_narudzbi=("order_id", "count"),
        ima_approved_at=("order_approved_at", "count"),
        ima_carrier_date=("order_delivered_carrier_date", "count"),
        ima_customer_date=("order_delivered_customer_date", "count")
    )
)

status_datumi

,broj_narudzbi,ima_approved_at,ima_carrier_date,ima_customer_date
order_status,,,,
approved,2,2,0,0
canceled,625,484,75,6
created,5,0,0,0
delivered,96478,96464,96476,96470
invoiced,314,314,0,0
processing,301,301,0,0
shipped,1107,1107,1107,0
unavailable,609,609,0,0


In [612]:
# 1. Status delivered, ali nema datuma dostave kupcu
delivered_bez_customer_date = orders[
    (orders["order_status"] == "delivered") &
    (orders["order_delivered_customer_date"].isna())
]

print(
    "Delivered bez datuma dostave kupcu:",
    len(delivered_bez_customer_date)
)


# 2. Status NIJE delivered, ali postoji datum dostave kupcu
nije_delivered_ima_customer_date = orders[
    (orders["order_status"] != "delivered") &
    (orders["order_delivered_customer_date"].notna())
]

print(
    "Status nije delivered, ali postoji datum dostave kupcu:",
    len(nije_delivered_ima_customer_date)
)


# 3. Status delivered, ali nema datuma predaje dostavljaču
delivered_bez_carrier_date = orders[
    (orders["order_status"] == "delivered") &
    (orders["order_delivered_carrier_date"].isna())
]

print(
    "Delivered bez datuma predaje prijevozniku:",
    len(delivered_bez_carrier_date)
)


# 4. Status shipped, ali postoji datum dostave kupcu
shipped_ima_customer_date = orders[
    (orders["order_status"] == "shipped") &
    (orders["order_delivered_customer_date"].notna())
]

print(
    "Shipped s datumom dostave kupcu:",
    len(shipped_ima_customer_date)
)


# 5. Status canceled, ali postoji datum predaje dostavljaču
canceled_ima_carrier_date = orders[
    (orders["order_status"] == "canceled") &
    (orders["order_delivered_carrier_date"].notna())
]

print(
    "Canceled s datumom predaje prijevozniku:",
    len(canceled_ima_carrier_date)
)


# 6. Status canceled, ali postoji datum dostave kupcu
canceled_ima_customer_date = orders[
    (orders["order_status"] == "canceled") &
    (orders["order_delivered_customer_date"].notna())
]

print(
    "Canceled s datumom dostave kupcu:",
    len(canceled_ima_customer_date)
)

Delivered bez datuma dostave kupcu: 8
Status nije delivered, ali postoji datum dostave kupcu: 6
Delivered bez datuma predaje prijevozniku: 2
Shipped s datumom dostave kupcu: 0
Canceled s datumom predaje prijevozniku: 75
Canceled s datumom dostave kupcu: 6


In [613]:
# Narudžba ima datum dostave kupcu,
# ali nema datum predaje dostavljaču
customer_date_bez_carrier_date = orders[
    (orders["order_delivered_customer_date"].notna()) &
    (orders["order_delivered_carrier_date"].isna())
]

print(
    "Postoji datum dostave kupcu, ali ne postoji datum predaje prijevozniku:",
    len(customer_date_bez_carrier_date)
)

Postoji datum dostave kupcu, ali ne postoji datum predaje prijevozniku: 1


In [614]:
anomalije_status_datumi = orders[
    (
        (orders["order_status"] == "delivered") &
        (
            orders["order_approved_at"].isna() |
            orders["order_delivered_carrier_date"].isna() |
            orders["order_delivered_customer_date"].isna()
        )
    )
    |
    (
        (orders["order_status"] != "delivered") &
        orders["order_delivered_customer_date"].notna()
    )
].copy()

anomalije_status_datumi[
    [
        "order_id",
        "customer_id",
        "order_status",
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]
].sort_values(
    ["order_status", "order_purchase_timestamp"]
)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
92636,65d1e226dfaeb8cdc42f665422522d14,70fc57eeae292675927697fe03ad3ff5,canceled,2016-10-03 21:01:41,2016-10-04 10:18:57,2016-10-25 12:14:28,2016-11-08 10:58:34,2016-11-25
58266,770d331c84e5b214bd9dc70a10b829d0,6c57e6119369185e575b36712766b0ef,canceled,2016-10-07 14:52:30,2016-10-07 15:07:10,2016-10-11 15:07:11,2016-10-14 15:07:11,2016-11-29
59332,8beb59392e21af5eb9547ae1a9938d06,bf609b5741f71697f65ce3852c5d2623,canceled,2016-10-08 20:17:50,2016-10-09 14:34:30,2016-10-14 22:45:26,2016-10-19 18:47:43,2016-11-30
8791,dabf2b0e35b423f94618bf965fcb7514,5cdec0bb8cbdf53ffc8fdc212cd247c6,canceled,2016-10-09 00:56:52,2016-10-09 13:36:58,2016-10-13 13:36:59,2016-10-16 14:36:59,2016-11-30
94399,2c45c33d2f9cb8ff8b1c86cc28c11c30,de4caa97afa80c8eeac2ff4c8da5b72e,canceled,2016-10-09 15:39:56,2016-10-10 10:40:49,2016-10-14 10:40:50,2016-11-09 14:53:50,2016-12-08
2921,1950d777989f6a877539f53795b4c3c3,1bccb206de9f0f25adc6871a1bcf77b2,canceled,2018-02-19 19:48:52,2018-02-19 20:56:05,2018-02-20 19:57:13,2018-03-21 22:03:51,2018-03-09
26800,c1d4211b3dae76144deccd6c74144a88,684cb238dc5b5d6366244e0e0776b450,delivered,2017-01-19 12:48:08,NaT,2017-01-25 14:56:50,2017-01-30 18:16:01,2017-03-01
48401,7002a78c79c519ac54022d4f8a65e6e8,d5de688c321096d15508faae67a27051,delivered,2017-01-19 22:26:59,NaT,2017-01-27 11:08:05,2017-02-06 14:22:19,2017-03-16
23156,12a95a3c06dbaec84bcfb0e2da5d228a,1e101e0daffaddce8159d25a8e53f2b2,delivered,2017-02-17 13:05:55,NaT,2017-02-22 11:23:11,2017-03-02 11:09:19,2017-03-20
72407,3c0b8706b065f9919d0505d3b3343881,d85919cb3c0529589c6fa617f5f43281,delivered,2017-02-17 15:53:27,NaT,2017-02-22 11:31:30,2017-03-03 11:47:47,2017-03-23


In [615]:
canceled_ali_dostavljene = orders[
    (orders["order_status"] == "canceled") &
    (orders["order_delivered_customer_date"].notna())
].copy()

canceled_ali_dostavljene[
    [
        "order_id",
        "customer_id",
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]
]

,order_id,customer_id,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
2921,1950d777989f6a877539f53795b4c3c3,1bccb206de9f0f25adc6871a1bcf77b2,2018-02-19 19:48:52,2018-02-19 20:56:05,2018-02-20 19:57:13,2018-03-21 22:03:51,2018-03-09
8791,dabf2b0e35b423f94618bf965fcb7514,5cdec0bb8cbdf53ffc8fdc212cd247c6,2016-10-09 00:56:52,2016-10-09 13:36:58,2016-10-13 13:36:59,2016-10-16 14:36:59,2016-11-30
58266,770d331c84e5b214bd9dc70a10b829d0,6c57e6119369185e575b36712766b0ef,2016-10-07 14:52:30,2016-10-07 15:07:10,2016-10-11 15:07:11,2016-10-14 15:07:11,2016-11-29
59332,8beb59392e21af5eb9547ae1a9938d06,bf609b5741f71697f65ce3852c5d2623,2016-10-08 20:17:50,2016-10-09 14:34:30,2016-10-14 22:45:26,2016-10-19 18:47:43,2016-11-30
92636,65d1e226dfaeb8cdc42f665422522d14,70fc57eeae292675927697fe03ad3ff5,2016-10-03 21:01:41,2016-10-04 10:18:57,2016-10-25 12:14:28,2016-11-08 10:58:34,2016-11-25
94399,2c45c33d2f9cb8ff8b1c86cc28c11c30,de4caa97afa80c8eeac2ff4c8da5b72e,2016-10-09 15:39:56,2016-10-10 10:40:49,2016-10-14 10:40:50,2016-11-09 14:53:50,2016-12-08


In [616]:
orders[
    orders["order_delivered_customer_date"].notna() &
    orders["order_delivered_carrier_date"].isna()
][
    [
        "order_id",
        "order_status",
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]
]

,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
73222,2aa91108853cecb43c84a5dc5b277475,delivered,2017-09-29 08:52:58,2017-09-29 09:07:16,NaT,2017-11-20 19:44:47,2017-11-14


#### Zaključak analize konzistentnosti statusa i vremenskih atributa:

Analizom odnosa između statusa narudžbi i pripadajućih vremenskih atributa utvrđen je manji broj nekonzistentnih ili nepotpunih zapisa. Među narudžbama sa statusom `delivered`, 14 zapisa nema poznato vrijeme odobrenja, 2 zapisa nemaju poznato vrijeme predaje prijevozniku, a 8 zapisa nema poznato vrijeme dostave kupcu. Jedna dostavljena narudžba ima poznat datum dostave kupcu, ali nema evidentiran datum predaje prijevozniku.

Dodatno je identificirano 6 narudžbi sa statusom `canceled` koje imaju evidentiran datum dostave kupcu. Sve navedene narudžbe imaju i evidentirane datume odobrenja i predaje prijevozniku, zbog čega vremenski atributi upućuju na to da su prošle kroz cijeli proces dostave. 

Identificirane nepravilnosti bit će uzete u obzir tijekom čišćenja podataka i definiranja kriterija za pojedine analize. Zapisi s nedostajućim vremenskim atributima neće se koristiti za izračun mjera koje zahtijevaju upravo te atribute, dok će izvorne vrijednosti statusa i datuma biti sačuvane kako bi se izbjeglo uvođenje neutemeljenih pretpostavki u podatke.

In [617]:
canceled_dostavljene_reviews = (
    canceled_ali_dostavljene[
        [
            "order_id",
            "order_purchase_timestamp",
            "order_delivered_customer_date"
        ]
    ]
    .merge(
        reviews[
            [
                "order_id",
                "review_score",
                "review_comment_title",
                "review_comment_message"
            ]
        ],
        on="order_id",
        how="left"
    )
)

canceled_dostavljene_reviews

,order_id,order_purchase_timestamp,order_delivered_customer_date,review_score,review_comment_title,review_comment_message
0,1950d777989f6a877539f53795b4c3c3,2018-02-19 19:48:52,2018-03-21 22:03:51,3,NaN,"O produto não foi enviado, porque eu errei ao ..."
1,dabf2b0e35b423f94618bf965fcb7514,2016-10-09 00:56:52,2016-10-16 14:36:59,5,NaN,Compraria novamente pq cumprem com o prazo de ...
2,770d331c84e5b214bd9dc70a10b829d0,2016-10-07 14:52:30,2016-10-14 15:07:11,1,NaN,"o produto é falso, ridiculo!"
3,8beb59392e21af5eb9547ae1a9938d06,2016-10-08 20:17:50,2016-10-19 18:47:43,1,NaN,Produto não foi entregue. Não foi enviada nota...
4,65d1e226dfaeb8cdc42f665422522d14,2016-10-03 21:01:41,2016-11-08 10:58:34,1,NaN,"O produto foi comprado em 10/10/2016, não cheg..."
5,2c45c33d2f9cb8ff8b1c86cc28c11c30,2016-10-09 15:39:56,2016-11-09 14:53:50,1,NaN,consta que meu pedido foi entregue mas nao che...


In [618]:
print(
    "Broj canceled i dostavljenih narudžbi:",
    len(canceled_ali_dostavljene)
)

print(
    "Broj narudžbi koje imaju review:",
    canceled_dostavljene_reviews["review_score"].notna().sum()
)

print(
    "Broj narudžbi s tekstualnim komentarom:",
    canceled_dostavljene_reviews["review_comment_message"].notna().sum()
)

print(
    "Prosječni review score:",
    canceled_dostavljene_reviews["review_score"].mean()
)

print(
    "Raspodjela ocjena:"
)

print(
    canceled_dostavljene_reviews["review_score"]
    .value_counts()
    .sort_index()
)

Broj canceled i dostavljenih narudžbi: 6
Broj narudžbi koje imaju review: 6
Broj narudžbi s tekstualnim komentarom: 6
Prosječni review score: 2.0
Raspodjela ocjena:
review_score
1    4
3    1
5    1
Name: count, dtype: int64


In [619]:
canceled_dostavljene_reviews[
    canceled_dostavljene_reviews["review_comment_message"].notna()
][
    [
        "order_id",
        "review_score",
        "review_comment_title",
        "review_comment_message"
    ]
]

,order_id,review_score,review_comment_title,review_comment_message
0,1950d777989f6a877539f53795b4c3c3,3,NaN,"O produto não foi enviado, porque eu errei ao ..."
1,dabf2b0e35b423f94618bf965fcb7514,5,NaN,Compraria novamente pq cumprem com o prazo de ...
2,770d331c84e5b214bd9dc70a10b829d0,1,NaN,"o produto é falso, ridiculo!"
3,8beb59392e21af5eb9547ae1a9938d06,1,NaN,Produto não foi entregue. Não foi enviada nota...
4,65d1e226dfaeb8cdc42f665422522d14,1,NaN,"O produto foi comprado em 10/10/2016, não cheg..."
5,2c45c33d2f9cb8ff8b1c86cc28c11c30,1,NaN,consta que meu pedido foi entregue mas nao che...


#### Analiza recenzija otkazanih narudžbi s evidentiranim datumom dostave

Kako bi se dodatno istražilo šest narudžbi sa statusom `canceled` koje istodobno imaju evidentiran `order_delivered_customer_date`, podaci o tim narudžbama povezani su sa skupom `reviews`. Analizirane su ocjene kupaca i pripadajući tekstualni komentari (koji su prevedeni uz pomoć ChatGPT-ja) kako bi se utvrdilo mogu li pružiti dodatni kontekst za prividnu nekonzistentnost između statusa narudžbe i evidentiranog datuma dostave.

U sljedećoj tablici prikazane su ocjene te prijevod i interpretacija dostupnih komentara kupaca:

| `order_id` | Ocjena (`review_score`) | Prijevod i značenje komentara |
|---|---:|---|
| `1950d777...` | 3 | **„Proizvod nije poslan jer sam pogrešno upisala broj svoje kuće.”** Komentar upućuje na problem s adresom dostave. Kupac navodi da proizvod nije poslan, iako za narudžbu postoji vrijednost atributa `order_delivered_customer_date`. |
| `dabf2b0e...` | 5 | **„Ponovno bih kupila jer poštuju rok isporuke i proizvodi dolaze odlično zapakirani.”** Komentar sugerira da je proizvod uspješno dostavljen. Status `canceled` stoga nije u skladu s iskustvom kupca opisanim u recenziji. |
| `770d331c...` | 1 | **„Proizvod je lažan, smiješno!”** Komentar upućuje na to da je proizvod vjerojatno primljen jer kupac komentira njegovu autentičnost. Moguće je da je nakon primitka došlo do reklamacije, povrata ili otkazivanja, ali to nije moguće potvrditi na temelju dostupnih podataka. |
| `8beb5939...` | 1 | **„Proizvod nije dostavljen. Nije poslan račun. Telefon za reklamacije ne radi.”** Kupac izričito navodi da proizvod nije dostavljen, unatoč tome što za narudžbu postoji evidentiran datum dostave. |
| `65d1e226...` | 1 | **„Proizvod je kupljen 10. 10. 2016., nije stigao, a sada dobivam ovu anketu koja me pita što mislim o njemu. Vrlo nezadovoljna!”** Kupac jasno navodi da proizvod nije primio, što je u suprotnosti s postojanjem evidentiranog datuma dostave. |
| `2c45c33d...` | 1 | **„Piše da je moja narudžba dostavljena, ali nije stigla. Postoji li mogućnost da mi kažete tko je potpisao primitak na toj lokaciji?”** Prema sustavu je narudžba evidentirana kao dostavljena, ali kupac tvrdi da je nije primio. |


Analiza komentara pokazala je da u četiri slučaja kupci navode probleme povezane s nedostavljanjem narudžbe ili adresom dostave. Tri kupca izričito navode da proizvod nisu primili, dok jedan navodi da proizvod nije poslan zbog pogrešno unesene adrese. U jednom slučaju kupac navodi da je proizvod lažan, što upućuje na to da je proizvod vjerojatno primljen, dok jedan kupac daje najvišu ocjenu i navodi da je dostava izvršena u roku.

Rezultati pokazuju da postojanje vrijednosti atributa `order_delivered_customer_date` ne mora nužno predstavljati pouzdanu potvrdu da je krajnji kupac stvarno primio proizvod. U pojedinim slučajevima moguće je da logistički sustav pošiljku evidentira kao dostavljenu iako je kupac nije primio. Također nije moguće isključiti mogućnost naknadnog otkazivanja, reklamacije ili povrata proizvoda, budući da skup podataka ne sadrži dovoljno informacija za potvrdu tih događaja.

Zbog toga se status ovih narudžbi neće mijenjati na temelju samog datuma dostave. Izvorni order_status zadržat će se kao konačni status narudžbe, dok će se vremenski atributi koristiti uz odgovarajuće provjere kvalitete i ovisno o cilju konkretne analize.

In [620]:
delivered_problematicne = orders[
    (orders["order_status"] == "delivered") &
    (
        orders["order_approved_at"].isna() |
        orders["order_delivered_carrier_date"].isna() |
        orders["order_delivered_customer_date"].isna()
    )
].copy()

print(
    "Broj problematičnih delivered narudžbi:",
    len(delivered_problematicne)
)

Broj problematičnih delivered narudžbi: 23


In [621]:
delivered_problematicne_reviews = (
    delivered_problematicne[
        [
            "order_id",
            "order_purchase_timestamp",
            "order_approved_at",
            "order_delivered_carrier_date",
            "order_delivered_customer_date",
            "order_estimated_delivery_date"
        ]
    ]
    .merge(
        reviews[
            [
                "order_id",
                "review_score",
                "review_comment_title",
                "review_comment_message"
            ]
        ],
        on="order_id",
        how="left"
    )
)

delivered_problematicne_reviews

,order_id,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,review_score,review_comment_title,review_comment_message
0,2d1e2d5bf4dc7227b3bfebb81328c15f,2017-11-28 17:44:07,2017-11-28 17:56:40,2017-11-30 18:12:23,NaT,2017-12-18,5,NaN,Chegou rápido tudo ok
1,e04abd8149ef81b95221e88f6ed9ab6a,2017-02-18 14:40:00,NaT,2017-02-23 12:04:47,2017-03-01 13:25:33,2017-03-17,4,NaN,"Muita demora, mas, tudo ok.\r\nTerezinha"
2,8a9adc69528e1001fc68dd0aaebbb54a,2017-02-18 12:45:31,NaT,2017-02-23 09:01:52,2017-03-02 10:05:06,2017-03-21,5,NaN,NaN
3,7013bcfc1c97fe719a7b5e05e61c12db,2017-02-18 13:29:47,NaT,2017-02-22 16:25:25,2017-03-01 08:07:38,2017-03-17,5,NaN,NaN
4,f5dd62b788049ad9fc0526e3ad11a097,2018-06-20 06:58:43,2018-06-20 07:19:05,2018-06-25 08:05:00,NaT,2018-07-16,5,Entrega super rápida.,"Produto novo, muito bom."
5,5cf925b116421afa85ee25e99b4c34fb,2017-02-18 16:48:35,NaT,2017-02-22 11:23:10,2017-03-09 07:28:47,2017-03-31,5,NaN,NaN
6,12a95a3c06dbaec84bcfb0e2da5d228a,2017-02-17 13:05:55,NaT,2017-02-22 11:23:11,2017-03-02 11:09:19,2017-03-20,5,NaN,NaN
7,c1d4211b3dae76144deccd6c74144a88,2017-01-19 12:48:08,NaT,2017-01-25 14:56:50,2017-01-30 18:16:01,2017-03-01,4,NaN,Bom custo x benefício pra proteger o crânio da...
8,d69e5d356402adc8cf17e08b5033acfb,2017-02-19 01:28:47,NaT,2017-02-23 03:11:48,2017-03-02 03:41:58,2017-03-27,5,NaN,Excelente produto
9,d77031d6a3c8a52f019764e68f211c69,2017-02-18 11:04:19,NaT,2017-02-23 07:23:36,2017-03-02 16:15:23,2017-03-22,5,NaN,"o produto chegou antes do prazo, é muito bom"


In [622]:
delivered_bez_approved_reviews = (
    orders[
        (orders["order_status"] == "delivered") &
        (orders["order_approved_at"].isna())
    ]
    .merge(
        reviews[
            [
                "order_id",
                "review_score",
                "review_comment_title",
                "review_comment_message"
            ]
        ],
        on="order_id",
        how="left"
    )
)

delivered_bez_approved_reviews[
    [
        "order_id",
        "order_purchase_timestamp",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "review_score",
        "review_comment_title",
        "review_comment_message"
    ]
]

,order_id,order_purchase_timestamp,order_delivered_carrier_date,order_delivered_customer_date,review_score,review_comment_title,review_comment_message
0,e04abd8149ef81b95221e88f6ed9ab6a,2017-02-18 14:40:00,2017-02-23 12:04:47,2017-03-01 13:25:33,4,NaN,"Muita demora, mas, tudo ok.\r\nTerezinha"
1,8a9adc69528e1001fc68dd0aaebbb54a,2017-02-18 12:45:31,2017-02-23 09:01:52,2017-03-02 10:05:06,5,NaN,NaN
2,7013bcfc1c97fe719a7b5e05e61c12db,2017-02-18 13:29:47,2017-02-22 16:25:25,2017-03-01 08:07:38,5,NaN,NaN
3,5cf925b116421afa85ee25e99b4c34fb,2017-02-18 16:48:35,2017-02-22 11:23:10,2017-03-09 07:28:47,5,NaN,NaN
4,12a95a3c06dbaec84bcfb0e2da5d228a,2017-02-17 13:05:55,2017-02-22 11:23:11,2017-03-02 11:09:19,5,NaN,NaN
5,c1d4211b3dae76144deccd6c74144a88,2017-01-19 12:48:08,2017-01-25 14:56:50,2017-01-30 18:16:01,4,NaN,Bom custo x benefício pra proteger o crânio da...
6,d69e5d356402adc8cf17e08b5033acfb,2017-02-19 01:28:47,2017-02-23 03:11:48,2017-03-02 03:41:58,5,NaN,Excelente produto
7,d77031d6a3c8a52f019764e68f211c69,2017-02-18 11:04:19,2017-02-23 07:23:36,2017-03-02 16:15:23,5,NaN,"o produto chegou antes do prazo, é muito bom"
8,7002a78c79c519ac54022d4f8a65e6e8,2017-01-19 22:26:59,2017-01-27 11:08:05,2017-02-06 14:22:19,2,NaN,Não gostei. O material não é tão bom. A cor do...
9,2eecb0d85f281280f79fa00f9cec1a95,2017-02-17 17:21:55,2017-02-22 11:42:51,2017-03-03 12:16:03,5,NaN,NaN


In [623]:
delivered_bez_customer_reviews = (
    orders[
        (orders["order_status"] == "delivered") &
        (orders["order_delivered_customer_date"].isna())
    ]
    .merge(
        reviews[
            [
                "order_id",
                "review_score",
                "review_comment_title",
                "review_comment_message"
            ]
        ],
        on="order_id",
        how="left"
    )
)

delivered_bez_customer_reviews[
    [
        "order_id",
        "order_purchase_timestamp",
        "order_delivered_carrier_date",
        "order_estimated_delivery_date",
        "review_score",
        "review_comment_title",
        "review_comment_message"
    ]
]

,order_id,order_purchase_timestamp,order_delivered_carrier_date,order_estimated_delivery_date,review_score,review_comment_title,review_comment_message
0,2d1e2d5bf4dc7227b3bfebb81328c15f,2017-11-28 17:44:07,2017-11-30 18:12:23,2017-12-18,5,NaN,Chegou rápido tudo ok
1,f5dd62b788049ad9fc0526e3ad11a097,2018-06-20 06:58:43,2018-06-25 08:05:00,2018-07-16,5,Entrega super rápida.,"Produto novo, muito bom."
2,2ebdfc4f15f23b91474edf87475f108e,2018-07-01 17:05:11,2018-07-03 13:57:00,2018-07-30,5,NaN,NaN
3,e69f75a717d64fc5ecdfae42b2e8e086,2018-07-01 22:05:55,2018-07-03 13:57:00,2018-07-30,5,NaN,NaN
4,0d3268bad9b086af767785e3f0fc0133,2018-07-01 21:14:02,2018-07-03 09:28:00,2018-07-24,5,Excelente!,O produto chegou muito antes do prazo previsto...
5,2d858f451373b04fb5c984a1cc2defaf,2017-05-25 23:22:43,NaT,2017-06-23,5,NaN,NaN
6,ab7c89dc1bf4a1ead9d6ec1ec8968a84,2018-06-08 12:09:39,2018-06-12 14:10:00,2018-06-26,1,Péssimo,Comprei um produto de uma marca e recebi outro...
7,20edc82cf5400ce95e1afacc25798b31,2018-06-27 16:09:12,2018-07-03 19:26:00,2018-07-19,5,Muito bom,Adorei


In [624]:
delivered_bez_carrier_reviews = (
    orders[
        (orders["order_status"] == "delivered") &
        (orders["order_delivered_carrier_date"].isna())
    ]
    .merge(
        reviews[
            [
                "order_id",
                "review_score",
                "review_comment_title",
                "review_comment_message"
            ]
        ],
        on="order_id",
        how="left"
    )
)

delivered_bez_carrier_reviews[
    [
        "order_id",
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
        "review_score",
        "review_comment_title",
        "review_comment_message"
    ]
]

,order_id,order_purchase_timestamp,order_approved_at,order_delivered_customer_date,order_estimated_delivery_date,review_score,review_comment_title,review_comment_message
0,2aa91108853cecb43c84a5dc5b277475,2017-09-29 08:52:58,2017-09-29 09:07:16,2017-11-20 19:44:47,2017-11-14,5,NaN,NaN
1,2d858f451373b04fb5c984a1cc2defaf,2017-05-25 23:22:43,2017-05-25 23:30:16,NaT,2017-06-23,5,NaN,NaN


In [625]:
print("DELIVERED BEZ APPROVED_AT")
print(
    delivered_bez_approved_reviews["review_score"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nDELIVERED BEZ CUSTOMER_DATE")
print(
    delivered_bez_customer_reviews["review_score"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nDELIVERED BEZ CARRIER_DATE")
print(
    delivered_bez_carrier_reviews["review_score"]
    .value_counts(dropna=False)
    .sort_index()
)

DELIVERED BEZ APPROVED_AT
review_score
2    1
3    1
4    4
5    8
Name: count, dtype: int64

DELIVERED BEZ CUSTOMER_DATE
review_score
1    1
5    7
Name: count, dtype: int64

DELIVERED BEZ CARRIER_DATE
review_score
5    2
Name: count, dtype: int64


#### Analiza recenzija problematičnih narudžbi sa statusom `delivered`

Kako bi se dodatno istražile narudžbe sa statusom `delivered` kojima nedostaje jedan ili više vremenskih atributa (`order_approved_at`, `order_delivered_carrier_date` ili `order_delivered_customer_date`), podaci su povezani sa skupom `reviews`.

Cilj analize je provjeriti pružaju li ocjene i tekstualni komentari kupaca dodatne informacije o tome je li narudžba stvarno realizirana i dostavljena te radi li se vjerojatnije o nedostajućem podatku u evidenciji nego o pogrešno postavljenom statusu narudžbe.

| `order_id` | Problem u vremenskim podacima | Ocjena | Prijevod i značenje komentara |
|---|---|---:|---|
| `2d1e2d5b...` | Nedostaje `order_delivered_customer_date` | 5 | **„Stiglo je brzo, sve je u redu.”** Komentar jasno potvrđuje da je narudžba primljena. Nedostatak datuma dostave najvjerojatnije predstavlja nedostajući podatak u evidenciji. |
| `e04abd81...` | Nedostaje `order_approved_at` | 4 | **„Dugo je trajalo, ali sve je u redu.”** Komentar sugerira da je narudžba uspješno završena. Nedostatak vremena odobrenja vjerojatno je samo nedostatak u evidenciji. |
| `8a9adc69...` | Nedostaje `order_approved_at` | 5 | Nema tekstualnog komentara. Visoka ocjena podržava pretpostavku da je kupac bio zadovoljan, ali sama po sebi ne daje izravnu potvrdu procesa dostave. |
| `7013bcfc...` | Nedostaje `order_approved_at` | 5 | Nema tekstualnog komentara. Visoka ocjena ne upućuje na problem s narudžbom. |
| `f5dd62b7...` | Nedostaje `order_delivered_customer_date` | 5 | Naslov: **„Super brza dostava.”** Komentar: **„Nov proizvod, vrlo dobar.”** Recenzija snažno potvrđuje da je proizvod primljen, pa je vjerojatno riječ o nedostajućem datumu dostave. |
| `5cf925b1...` | Nedostaje `order_approved_at` | 5 | Nema tekstualnog komentara. Ocjena ne upućuje na problem s realizacijom narudžbe. |
| `12a95a3c...` | Nedostaje `order_approved_at` | 5 | Nema tekstualnog komentara. Ocjena ne upućuje na problem s realizacijom narudžbe. |
| `c1d4211b...` | Nedostaje `order_approved_at` | 4 | **„Dobar omjer cijene i kvalitete za zaštitu glave djece.”** Kupac komentira konkretan proizvod, što sugerira da ga je primio. |
| `d69e5d35...` | Nedostaje `order_approved_at` | 5 | **„Odličan proizvod.”** Komentar sugerira da je proizvod primljen i ocijenjen nakon korištenja. |
| `d77031d6...` | Nedostaje `order_approved_at` | 5 | **„Proizvod je stigao prije roka, vrlo je dobar.”** Komentar izravno potvrđuje dostavu. |
| `2ebdfc4f...` | Nedostaje `order_delivered_customer_date` | 5 | Nema tekstualnog komentara. Visoka ocjena ne upućuje na problem, ali ne daje izravnu potvrdu dostave. |
| `7002a78c...` | Nedostaje `order_approved_at` | 2 | **„Nije mi se svidjelo. Materijal nije tako dobar. Boja uzoraka je blijeda, nije kao što je prikazano na fotografiji.”** Kupac očito komentira primljeni proizvod. Niska ocjena odnosi se na kvalitetu proizvoda, a ne na problem s dostavom. |
| `2eecb0d8...` | Nedostaje `order_approved_at` | 5 | Nema tekstualnog komentara. Ocjena ne upućuje na problem s narudžbom. |
| `51eb2eeb...` | Nedostaje `order_approved_at` | 5 | Nema tekstualnog komentara. Ocjena ne upućuje na problem s narudžbom. |
| `88083e8f...` | Nedostaje `order_approved_at` | 4 | **„Usluga stranice ovaj put bila je odlična. Što se tiče proizvoda i dostave, omjer cijene i kvalitete nije baš dobar; za tu cijenu proizvod bi mogao biti bolje izrađen, a nosač ima samo dvije razine nagiba.”** Kupac jasno komentira primljeni proizvod. |
| `3c0b8706...` | Nedostaje `order_approved_at` | 3 | Nema tekstualnog komentara. Ocjena ne omogućuje dodatni zaključak. |
| `2aa91108...` | Nedostaje `order_delivered_carrier_date` | 5 | Nema tekstualnog komentara. Budući da postoji datum dostave kupcu i visoka ocjena, nedostatak datuma predaje prijevozniku vjerojatno predstavlja nedostajući podatak u evidenciji. |
| `e69f75a7...` | Nedostaje `order_delivered_customer_date` | 5 | Nema tekstualnog komentara. Visoka ocjena podržava pretpostavku da nije bilo značajnog problema, ali ne potvrđuje izravno datum dostave. |
| `0d3268ba...` | Nedostaje `order_delivered_customer_date` | 5 | Naslov: **„Odlično!”** Komentar: **„Proizvod je stigao puno prije predviđenog roka, došao je u odličnom stanju i vrlo dobro zapakiran. Preporučujem!”** Komentar izravno potvrđuje dostavu, pa je nedostatak datuma dostave očito problem evidencije. |
| `2babbb4b...` | Nedostaje `order_approved_at` | 4 | **„Osim što je proizvod stigao prije očekivanog roka, iznenadila me i njegova kvaliteta! Oduševljena sam!”** Komentar izravno potvrđuje uspješnu dostavu. |
| `2d858f45...` | Nedostaju `order_delivered_carrier_date` i `order_delivered_customer_date` | 5 | Nema tekstualnog komentara. Visoka ocjena sugerira zadovoljstvo, ali bez komentara nije moguće potvrditi je li proizvod stvarno dostavljen. |
| `ab7c89dc...` | Nedostaje `order_delivered_customer_date` | 1 | Naslov: **„Užasno.”** Komentar: **„Kupila sam proizvod jedne marke, a dobila drugi, lošiji od onoga koji sam kupila. Osim toga, naplatili su mi dostavu, a morala sam sama preuzeti proizvod u pošti.”** Kupac jasno potvrđuje da je proizvod primio/preuzeo, ali je nezadovoljan jer je dobio pogrešan proizvod i morao ga preuzeti u pošti. Nedostatak datuma dostave ne znači da proizvod nije stigao. |
| `20edc82c...` | Nedostaje `order_delivered_customer_date` | 5 | Naslov: **„Vrlo dobro.”** Komentar: **„Oduševljena sam.”** Komentar sugerira pozitivno iskustvo s proizvodom, ali ne daje izravnu informaciju o datumu dostave. |


Analiza recenzija pokazuje da većina problematičnih narudžbi sa statusom `delivered` vjerojatno zaista predstavlja realizirane narudžbe, dok nedostajuće vremenske vrijednosti uglavnom predstavljaju nedostatke u evidenciji.

Kod narudžbi kojima nedostaje `order_approved_at`, nekoliko komentara izravno potvrđuje primitak proizvoda i uspješnu dostavu. Zbog toga nema razloga smatrati da narudžbe stvarno nisu bile odobrene; vjerojatnije je da vrijeme odobrenja jednostavno nije zabilježeno.

Kod narudžbi kojima nedostaje `order_delivered_customer_date`, više komentara izravno potvrđuje primitak proizvoda, uključujući komentare o brzoj dostavi, dolasku prije procijenjenog roka i kvaliteti zaprimljenog proizvoda. To pokazuje da nedostatak datuma dostave ne mora značiti da narudžba nije dostavljena.

Posebno je zanimljiv slučaj narudžbe `ab7c89dc...`, kod koje kupac navodi da je proizvod morao sam preuzeti u pošti. To može objasniti zašto standardni datum dostave na adresu kupca nije evidentiran i pokazuje da proces završetka dostave može imati iznimke koje nisu potpuno opisane dostupnim atributima.

Za jednu narudžbu kojoj istodobno nedostaju `order_delivered_carrier_date` i `order_delivered_customer_date` dostupna je samo najviša ocjena bez tekstualnog komentara. U tom slučaju nije moguće pouzdano zaključiti što se dogodilo.

Na temelju ove analize izvorni status `delivered` neće se mijenjati. Nedostajuće vremenske vrijednosti tretirat će se kao nedostajući podaci, a zapisi će se isključivati samo iz onih izračuna koji zahtijevaju upravo nedostajući vremenski atribut.

### Analiza trajanja pojedinih faza procesa narudžbe

Vremenski atributi skupa `orders` omogućuju analizu trajanja pojedinih faza životnog ciklusa narudžbe. Analiziraju se sljedeći vremenski intervali:

1. vrijeme od kreiranja narudžbe do odobrenja,
2. vrijeme od odobrenja do predaje narudžbe dostavljaču,
3. vrijeme od predaje narudžbe dostavljaču do dostave kupcu,
4. ukupno vrijeme od kreiranja narudžbe do dostave kupcu.

Trajanje pojedine faze računa se samo za narudžbe kod kojih postoje oba potrebna vremenska atributa. Za analizu trajanja između pojedinih faza dodatno se uzima u obzir vremenska konzistentnost zapisa, kako prethodno identificirane anomalije ne bi dovele do negativnih i pogrešnih vrijednosti trajanja.

In [626]:
vrijeme_do_odobrenja = (
    orders["order_approved_at"]
    - orders["order_purchase_timestamp"]
).dt.total_seconds() / 3600

print(
    "Broj narudžbi uključenih u analizu:",
    vrijeme_do_odobrenja.notna().sum()
)

vrijeme_do_odobrenja.dropna().describe()

Broj narudžbi uključenih u analizu: 99281


count    99281.000000
mean        10.419094
std         26.038004
min          0.000000
25%          0.215000
50%          0.343333
75%         14.580833
max       4509.180556
dtype: float64

In [627]:
vrijeme_obrade = orders[
    orders["order_approved_at"].notna() &
    orders["order_delivered_carrier_date"].notna() &
    (
        orders["order_delivered_carrier_date"]
        >= orders["order_approved_at"]
    )
].copy()

vrijeme_obrade["vrijeme_obrade_sati"] = (
    vrijeme_obrade["order_delivered_carrier_date"]
    - vrijeme_obrade["order_approved_at"]
).dt.total_seconds() / 3600

print(
    "Broj narudžbi uključenih u analizu:",
    len(vrijeme_obrade)
)

vrijeme_obrade["vrijeme_obrade_sati"].describe()

Broj narudžbi uključenih u analizu: 96285


count    96285.000000
mean        68.620465
std         83.965777
min          0.004167
25%         21.636944
50%         44.440000
75%         86.977222
max       3018.301667
Name: vrijeme_obrade_sati, dtype: float64

In [628]:
vrijeme_transporta = orders[
    orders["order_delivered_carrier_date"].notna() &
    orders["order_delivered_customer_date"].notna() &
    (
        orders["order_delivered_customer_date"]
        >= orders["order_delivered_carrier_date"]
    )
].copy()

vrijeme_transporta["vrijeme_transporta_dani"] = (
    vrijeme_transporta["order_delivered_customer_date"]
    - vrijeme_transporta["order_delivered_carrier_date"]
).dt.total_seconds() / (24 * 3600)

print(
    "Broj narudžbi uključenih u analizu:",
    len(vrijeme_transporta)
)

vrijeme_transporta["vrijeme_transporta_dani"].describe()

Broj narudžbi uključenih u analizu: 96452


count    96452.000000
mean         9.333551
std          8.758825
min          0.000000
25%          4.101525
50%          7.100312
75%         12.030443
max        205.190972
Name: vrijeme_transporta_dani, dtype: float64

In [629]:
ukupno_vrijeme_dostave = orders[
    (orders["order_status"] == "delivered") &
    orders["order_delivered_customer_date"].notna()
].copy()

ukupno_vrijeme_dostave["ukupno_vrijeme_dostave_dani"] = (
    ukupno_vrijeme_dostave["order_delivered_customer_date"]
    - ukupno_vrijeme_dostave["order_purchase_timestamp"]
).dt.total_seconds() / (24 * 3600)

print(
    "Broj narudžbi uključenih u analizu:",
    len(ukupno_vrijeme_dostave)
)

ukupno_vrijeme_dostave[
    "ukupno_vrijeme_dostave_dani"
].describe()

Broj narudžbi uključenih u analizu: 96470


count    96470.000000
mean        12.558217
std          9.546156
min          0.533414
25%          6.766204
50%         10.217477
75%         15.720182
max        209.628611
Name: ukupno_vrijeme_dostave_dani, dtype: float64

### Zaključak analize trajanja procesa narudžbe

Za 99.281 narudžbu za koju je poznato vrijeme kupnje i odobrenja, prosječno vrijeme do odobrenja iznosi približno 10,42 sata, dok medijan iznosi samo 0,34 sata, odnosno približno 21 minutu. Velika razlika između aritmetičke sredine i medijana upućuje na postojanje manjeg broja narudžbi s izrazito dugim vremenom odobrenja. Maksimalno zabilježeno vrijeme iznosi približno 4.509 sati, odnosno 188 dana.

Nakon isključivanja prethodno identificiranih vremenski nekonzistentnih zapisa, za 96.285 narudžbi analizirano je vrijeme između odobrenja i predaje dostavljaču. Prosječno vrijeme iznosi 68,62 sata, dok je medijan 44,44 sata. I u ovoj fazi postoje izrazito visoke vrijednosti, pri čemu maksimum iznosi približno 3.018 sati, odnosno 126 dana.

Vrijeme transporta, definirano kao razdoblje između predaje narudžbe dostavljaču i evidentirane dostave kupcu, analizirano je za 96.452 vremenski konzistentne narudžbe. Prosječno vrijeme transporta iznosi 9,33 dana, a medijan 7,10 dana. 

Za 96.470 narudžbi sa statusom `delivered` i poznatim datumom dostave kupcu, ukupno vrijeme od kupnje do dostave prosječno iznosi 12,56 dana, dok je medijan 10,22 dana. 

U svim analiziranim vremenskim intervalima aritmetička sredina veća je od medijana, što upućuje na desnu asimetriju distribucija i prisutnost ekstremno visokih vrijednosti. Zbog toga će u daljnjoj statističkoj analizi, uz aritmetičku sredinu, biti važno promatrati medijan, kvartile i mjere raspršenosti. Ekstremne vrijednosti potrebno je dodatno analizirati prije donošenja odluke o njihovom tretmanu tijekom čišćenja i pripreme podataka.

In [630]:
# Spajanje narudžbi sa stavkama narudžbe
orders_geo = (
    orders[
        [
            "order_id",
            "customer_id",
            "order_status",
            "order_purchase_timestamp",
            "order_delivered_carrier_date",
            "order_delivered_customer_date",
            "order_estimated_delivery_date"
        ]
    ]
    .merge(
        order_items[
            [
                "order_id",
                "product_id",
                "seller_id"
            ]
        ],
        on="order_id",
        how="left"
    )
)

# Dodavanje lokacije kupca
orders_geo = orders_geo.merge(
    customers[
        [
            "customer_id",
            "customer_zip_code_prefix",
            "customer_city",
            "customer_state"
        ]
    ],
    on="customer_id",
    how="left"
)

# Dodavanje lokacije prodavača
orders_geo = orders_geo.merge(
    sellers[
        [
            "seller_id",
            "seller_zip_code_prefix",
            "seller_city",
            "seller_state"
        ]
    ],
    on="seller_id",
    how="left"
)

orders_geo.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,product_id,seller_id,customer_zip_code_prefix,customer_city,customer_state,seller_zip_code_prefix,seller_city,seller_state
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,3149,sao paulo,SP,9350.0,maua,SP
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,595fac2a385ac33a80bd5114aec74eb8,289cdb325fb7e7f891c38608bf9e0962,47813,barreiras,BA,31570.0,belo horizonte,SP
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,aa4383b373c6aca5d8797843e5594415,4869f7a5dfa277a7dca6462dcf3b52b2,75265,vianopolis,GO,14840.0,guariba,SP
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,d0b61bfb1de832b15ba9d266ca96e5b0,66922902710d126a0e7d26b0e3805106,59296,sao goncalo do amarante,RN,31842.0,belo horizonte,MG
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,65266b2da20d04dbe00c5c2d3bb7859e,2c9e548be18521d1c43cde1c582c6de8,9195,santo andre,SP,8752.0,mogi das cruzes,SP


In [631]:
print("Broj redaka:", len(orders_geo))
print("Broj jedinstvenih narudžbi:", orders_geo["order_id"].nunique())
print("Broj jedinstvenih prodavača:", orders_geo["seller_id"].nunique())

broj_prodavaca_po_narudzbi = (
    orders_geo
    .groupby("order_id")["seller_id"]
    .nunique()
)

print("\nBroj narudžbi s jednim prodavačem:")
print((broj_prodavaca_po_narudzbi == 1).sum())

print("\nBroj narudžbi s više prodavača:")
print((broj_prodavaca_po_narudzbi > 1).sum())

broj_prodavaca_po_narudzbi.describe()

Broj redaka: 113425
Broj jedinstvenih narudžbi: 99441
Broj jedinstvenih prodavača: 3095

Broj narudžbi s jednim prodavačem:
97388

Broj narudžbi s više prodavača:
1278


count    99441.000000
mean         1.005722
std          0.150947
min          0.000000
25%          1.000000
50%          1.000000
75%          1.000000
max          5.000000
Name: seller_id, dtype: float64

In [632]:
narudzbe_bez_prodavaca = orders_geo[
    orders_geo["seller_id"].isna()
]

print(
    "Broj jedinstvenih narudžbi bez prodavača:",
    narudzbe_bez_prodavaca["order_id"].nunique()
)

narudzbe_bez_prodavaca[
    [
        "order_id",
        "customer_id",
        "product_id",
        "order_status",
        "order_purchase_timestamp"
    ]
].drop_duplicates()

Broj jedinstvenih narudžbi bez prodavača: 775


,order_id,customer_id,product_id,order_status,order_purchase_timestamp
306,8e24261a7e58791d10cb1bf9da94df5c,64a254d30eed42cd0e6c36dddb88adf0,NaN,unavailable,2017-11-16 15:09:28
671,c272bcd21c287498b4883c7512019702,9582c5bbecc65eb568e2c1d839b5cba1,NaN,unavailable,2018-01-31 11:31:37
791,37553832a3a89c9b2db59701c357ca67,7607cd563696c27ede287e515812d528,NaN,unavailable,2017-08-14 17:38:02
850,d57e15fb07fd180f06ab3926b39edcd2,470b93b3f1cde85550fc74cd3a476c78,NaN,unavailable,2018-01-08 19:39:03
1294,00b1cb0320190ca0daa2c88b35206009,3532ba38a3fd242259a514ac2b6ae6b6,NaN,canceled,2018-08-28 15:26:39
...,...,...,...,...,...
113204,aaab15da689073f8f9aa978a390a69d1,df20748206e4b865b2f14a5eabbfcf34,NaN,unavailable,2018-01-16 14:27:59
113236,3a3cddda5a7c27851bd96c3313412840,0b0d6095c5555fe083844281f6b093bb,NaN,canceled,2018-08-31 16:13:44
113324,a89abace0dcc01eeb267a9660b5ac126,2f0524a7b1b3845a1a57fcf3910c4333,NaN,canceled,2018-09-06 18:45:47
113325,a69ba794cc7deb415c3e15a0a3877e69,726f0894b5becdf952ea537d5266e543,NaN,unavailable,2017-08-23 16:28:04


In [633]:
narudzbe_bez_prodavaca["order_status"].value_counts()

order_status
unavailable    603
canceled       164
created          5
invoiced         2
shipped          1
Name: count, dtype: int64

### Narudžbe bez povezanog prodavača

Nakon povezivanja podataka o narudžbama sa stavkama narudžbi identificirano je 775 narudžbi za koje nije pronađen povezani prodavač (`seller_id`). Analiza statusa pokazala je da se najveći dio tih narudžbi odnosi na narudžbe sa statusom `unavailable` (603 narudžbe) i `canceled` (164 narudžbe). Zajedno ove dvije skupine čine gotovo 99 % svih narudžbi bez povezanog prodavača.

Preostalih osam narudžbi ima statuse `created` (5), `invoiced` (2) i `shipped` (1). Nijedna narudžba bez povezanog prodavača nema status `delivered`.

Rezultati upućuju na to da nedostatak povezanog prodavača uglavnom nije slučajan problem kvalitete podataka, već je povezan s narudžbama koje nisu uspješno prošle cjelokupan proces realizacije. 

## 8.6. Analiza skupa `order_items`

Skup `order_items` sadrži podatke o pojedinačnim stavkama narudžbi. Svaki zapis povezuje narudžbu s kupljenim proizvodom i prodavačem te sadrži podatke o cijeni proizvoda, trošku dostave i krajnjem roku za predaju pošiljke prijevozniku.


In [634]:
print("Broj zapisa:", len(order_items))

print(
    "Broj jedinstvenih narudžbi:",
    order_items["order_id"].nunique()
)

print(
    "Broj jedinstvenih proizvoda:",
    order_items["product_id"].nunique()
)

print(
    "Broj jedinstvenih prodavača:",
    order_items["seller_id"].nunique()
)

print(
    "Raspon order_item_id:",
    order_items["order_item_id"].min(),
    "-",
    order_items["order_item_id"].max()
)

Broj zapisa: 112650
Broj jedinstvenih narudžbi: 98666
Broj jedinstvenih proizvoda: 32951
Broj jedinstvenih prodavača: 3095
Raspon order_item_id: 1 - 21


In [635]:
broj_stavki_po_narudzbi = (
    order_items
    .groupby("order_id")
    ["order_item_id"]
    .count()
)

broj_stavki_po_narudzbi.describe()

count    98666.000000
mean         1.141731
std          0.538452
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max         21.000000
Name: order_item_id, dtype: float64

In [636]:
broj_stavki_po_narudzbi.value_counts().sort_index()

order_item_id
1     88863
2      7516
3      1322
4       505
5       204
6       198
7        22
8         8
9         3
10        8
11        4
12        5
13        1
14        2
15        2
20        2
21        1
Name: count, dtype: int64

### Broj stavki po narudžbi

Skup `order_items` sadrži ukupno **112.650 stavki**, koje pripadaju **98.666 jedinstvenih narudžbi**. U skupu se pojavljuje **32.951 jedinstveni proizvod** i **3.095 jedinstvenih prodavača**.

Analiza broja stavki pokazuje da narudžba u prosjeku sadrži približno **1,14 stavki**, dok je medijan jednak 1. Najmanji broj stavki po narudžbi iznosi 1, a najveći 21. Čak **88.863 narudžbe** sadrže samo jednu stavku, dok se broj narudžbi s većim brojem stavki postupno smanjuje. U skupu podataka prevladavaju narudžbe s jednom stavkom.

Stupac `order_item_id` predstavlja redni broj stavke unutar pojedine narudžbe, a ne globalno jedinstveni identifikator stavke. Zbog toga se pojedina stavka može jednoznačno identificirati kombinacijom atributa `order_id` i `order_item_id`. 

In [637]:
broj_stavki_i_proizvoda = (
    order_items
    .groupby("order_id")
    .agg(
        broj_stavki=("order_item_id", "count"),
        broj_jedinstvenih_proizvoda=("product_id", "nunique"),
        broj_prodavaca=("seller_id", "nunique")
    )
)

broj_stavki_i_proizvoda["ima_ponovljeni_proizvod"] = (
    broj_stavki_i_proizvoda["broj_stavki"]
    >
    broj_stavki_i_proizvoda["broj_jedinstvenih_proizvoda"]
)

print(
    "Narudžbe s ponovljenim istim proizvodom:",
    broj_stavki_i_proizvoda["ima_ponovljeni_proizvod"].sum()
)

broj_stavki_i_proizvoda.describe()

Narudžbe s ponovljenim istim proizvodom: 6968


,broj_stavki,broj_jedinstvenih_proizvoda,broj_prodavaca
count,98666.000000,98666.000000,98666.000000
mean,1.141731,1.038098,1.013622
std,0.538452,0.226456,0.122297
min,1.000000,1.000000,1.000000
25%,1.000000,1.000000,1.000000
50%,1.000000,1.000000,1.000000
75%,1.000000,1.000000,1.000000
max,21.000000,8.000000,5.000000


In [638]:
broj_stavki_i_proizvoda[
    broj_stavki_i_proizvoda["ima_ponovljeni_proizvod"]
].sort_values(
    "broj_stavki",
    ascending=False
)

,broj_stavki,broj_jedinstvenih_proizvoda,broj_prodavaca,ima_ponovljeni_proizvod
order_id,,,,
8272b63d03f5f79c56e9e4120aec44ef,21,3,1,True
1b15974a0141d54e36626dca3fdc731a,20,1,1,True
ab14fdcfbe524636d65ee38360e22ce8,20,1,1,True
428a2f660dc84138d969ccd69a0ab6d5,15,1,1,True
9ef13efd6949e4573a18964dd1bbe7f5,15,1,1,True
...,...,...,...,...
fea59446572d8db3b8e9709ab5c096d6,2,1,1,True
fe8c0bf415cf9f05b4f32539a80f83fa,2,1,1,True
fe802824215762b3ea5f49760eac0bd4,2,1,1,True


### Odnos stavki, proizvoda i prodavača unutar narudžbe

Dodatno je analiziran odnos između ukupnog broja stavki i broja jedinstvenih proizvoda unutar pojedine narudžbe. Utvrđeno je da **6.968 narudžbi** sadrži ponavljanje istog proizvoda, odnosno da je broj stavki u tim narudžbama veći od broja jedinstvenih proizvoda.

Narudžba u prosjeku sadrži približno **1,14 stavki**, **1,04 jedinstvena proizvoda** i **1,01 jedinstvenog prodavača**. Najveći zabilježeni broj stavki u jednoj narudžbi iznosi **21**, dok jedna narudžba može sadržavati najviše **8 različitih proizvoda** i proizvode od najviše **5 različitih prodavača**.

Rezultati pokazuju da se isti `product_id` može pojaviti više puta unutar iste narudžbe. Stoga `order_item_id` predstavlja pojedinačnu stavku, odnosno redni broj stavke unutar narudžbe, a ne nužno jedinstveni proizvod.


In [639]:
order_items[
    [
        "price",
        "freight_value"
    ]
].describe()

,price,freight_value
count,112650.000000,112650.000000
mean,120.653739,19.990320
std,183.633928,15.806405
min,0.850000,0.000000
25%,39.900000,13.080000
50%,74.990000,16.260000
75%,134.900000,21.150000
max,6735.000000,409.680000


In [640]:
order_items[
    [
        "price",
        "freight_value"
    ]
].isna().sum()

price            0
freight_value    0
dtype: int64

In [641]:
print(
    "Broj stavki s cijenom <= 0:",
    (order_items["price"] <= 0).sum()
)

print(
    "Broj stavki s troškom dostave < 0:",
    (order_items["freight_value"] < 0).sum()
)

print(
    "Broj stavki s troškom dostave = 0:",
    (order_items["freight_value"] == 0).sum()
)

Broj stavki s cijenom <= 0: 0
Broj stavki s troškom dostave < 0: 0
Broj stavki s troškom dostave = 0: 383


In [642]:
order_items[
    [
        "order_id",
        "order_item_id",
        "product_id",
        "seller_id",
        "price",
        "freight_value"
    ]
].sort_values(
    "price",
    ascending=False
).head(20)

,order_id,order_item_id,product_id,seller_id,price,freight_value
3556,0812eb902a67711a1cb742b3cdaa65ae,1,489ae2aa008f021502940f251d4cce7f,e3b4998c7a498169dc7bce44e6bb6277,6735.00,194.31
112233,fefacc66af859508bf1a7934eab1e97f,1,69c590f7ffc7bf8db97190b6cb6ed62e,80ceebb4ee9b31afb6c6a916a574a1e2,6729.00,193.21
107841,f5136e38d1a14a4dbd87dff67da82701,1,1bdf5e6731585cf01aa8169c7028d6ad,ee27a8f15b1dded4d213a468ba4eb391,6499.00,227.66
74336,a96610ab360d42a2e5335a3998b4718a,1,a6492cc69376c469ab6f61d8f44de961,59417c56835dd8e2e72f91f809cd4092,4799.00,151.34
11249,199af31afc78c699f0dbf71fb178d4d4,1,c3ed642d592594bb648ff4a04cee2747,59417c56835dd8e2e72f91f809cd4092,4690.00,74.34
62086,8dbc85d1447242f3b127dda390d56e19,1,259037a6a41845e455183f89c5035f18,c72de06d72748d1a0dfb2125be43ba63,4590.00,91.78
29193,426a9742b533fc6fed17d1fd6d143d7e,1,a1beef8f3992dbd4cd8726796aa69c53,512d298ac2a96d1931b6bd30aa21f61d,4399.87,113.45
45843,68101694e5c5dc7330c91e1bbc36214f,1,6cdf8fc1d741c76586d8b6b15e9eef30,ed4acab38528488b65a9a9c603ff024a,4099.99,75.27
78310,b239ca7cd485940b31882363b52e6674,1,dd113cb02b2af9c8e5787e8f1f0722f6,821fb029fc6e495ca4f08a35d51e53a5,4059.00,104.51
59137,86c4eab1571921a6a6e248ed312f5a5a,1,6902c1962dd19d540807d0ab8fade5c6,fa1c13f2614d7b5c4749cbc52fecda94,3999.90,17.01


In [643]:
order_items[
    [
        "order_id",
        "order_item_id",
        "product_id",
        "seller_id",
        "price",
        "freight_value"
    ]
].sort_values(
    "freight_value",
    ascending=False
).head(20)

,order_id,order_item_id,product_id,seller_id,price,freight_value
73486,a77e1550db865202c56b19ddc6dc4d53,1,ec31d2a17b299511e7c8627be9337b9b,257e61d3251fb5efb9daadddbc2cf7ca,979.00,409.68
28044,3fde74c28a3d5d618c00f26d51baafa0,1,a3cd9517ebf5a50dca25acce54f3b171,6fa9202c10491e472dffd59a3e82b2a3,2338.08,375.28
3303,076d1555fb53a89b0ef4d529e527a0f6,1,a3cd9517ebf5a50dca25acce54f3b171,6fa9202c10491e472dffd59a3e82b2a3,2338.08,375.28
69797,9f49bd16053df810384e793386312674,1,256a9c364b75753b97bee410c9491ad8,5c030029b5916fed0986310385ec9009,1149.00,339.59
16731,264a7e199467906c0727394df82d1a6a,1,97c948ebc8c04b26b7bbb095d4228f2a,17f51e7198701186712e53a39c564617,1050.00,338.30
87936,c7a07ddd52bbe18b61da49a8d89853d3,1,97c948ebc8c04b26b7bbb095d4228f2a,17f51e7198701186712e53a39c564617,1050.00,322.10
5037,0b6230647ed16f4b3e70282dc4b5b87f,1,46e24ce614899e36617e37ea1e4aa6ff,17f51e7198701186712e53a39c564617,1050.00,321.88
3584,0822bcde10bb5d023755a71bc8f7797f,1,363a9f5b97bf194da23858be722a7aa5,9596c870880d900012f2e8e6e30d06d7,990.00,321.46
29787,43bdbd9dc0931d72befdf4765af6c442,1,7e53e051875b2a0c9f22acd8a9a29a20,eeb6de78f79159600292e314a77cbd18,3089.00,317.47
48320,6ddfbf514959b49b6410c01ad93054bb,1,363a9f5b97bf194da23858be722a7aa5,9596c870880d900012f2e8e6e30d06d7,1045.00,314.40


In [644]:
order_items["udio_dostave_u_cijeni"] = (
    order_items["freight_value"] /
    order_items["price"]
)

order_items["udio_dostave_u_cijeni"]

0         0.225637
1         0.083076
2         0.089799
3         0.984604
4         0.090745
            ...   
112645    0.144705
112646    0.104371
112647    0.169670
112648    0.155742
112649    0.297442
Name: udio_dostave_u_cijeni, Length: 112650, dtype: float64

In [645]:
print(
    "Broj stavki kod kojih je dostava skuplja od proizvoda:",
    (
        order_items["freight_value"] >
        order_items["price"]
    ).sum()
)

Broj stavki kod kojih je dostava skuplja od proizvoda: 4124


In [646]:
order_items[
    order_items["freight_value"] > order_items["price"]
][
    [
        "order_id",
        "order_item_id",
        "product_id",
        "seller_id",
        "price",
        "freight_value",
        "udio_dostave_u_cijeni"
    ]
].sort_values(
    "udio_dostave_u_cijeni",
    ascending=False
).head(20)

,order_id,order_item_id,product_id,seller_id,price,freight_value,udio_dostave_u_cijeni
87081,c5bdd8ef3c0ec420232e668302179113,2,8a3254bee785a526d548a81a9bc3c9be,96804ea39d96eb908e7c3afdb671bb9e,0.85,22.30,26.235294
27652,3ee6513ae7ea23bdfab5b9ab60bffcb5,1,8a3254bee785a526d548a81a9bc3c9be,96804ea39d96eb908e7c3afdb671bb9e,0.85,18.23,21.447059
48625,6e864b3f0ec71031117ad4cf46b7f2a1,1,8a3254bee785a526d548a81a9bc3c9be,96804ea39d96eb908e7c3afdb671bb9e,0.85,18.23,21.447059
110535,fb265b2dc558a56445dfc48f8224e201,1,baf25ed4f8f70238cc87230379471454,128f9bfbe4c7d5185033914b1de3d39a,9.90,121.22,12.244444
94495,d642656598ae928a250620315d19e87e,1,b07fffe072c9adc235a35d8da7c0584d,dd533b429f380718b70ad9922c294bae,4.99,37.04,7.422846
57315,8272b63d03f5f79c56e9e4120aec44ef,19,270516a3f41dc035aa87d220228f844c,2709af9587499e95e803a6498a5a56e9,1.20,7.89,6.575000
57310,8272b63d03f5f79c56e9e4120aec44ef,14,270516a3f41dc035aa87d220228f844c,2709af9587499e95e803a6498a5a56e9,1.20,7.89,6.575000
57309,8272b63d03f5f79c56e9e4120aec44ef,13,270516a3f41dc035aa87d220228f844c,2709af9587499e95e803a6498a5a56e9,1.20,7.89,6.575000
57301,8272b63d03f5f79c56e9e4120aec44ef,5,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,1.20,7.89,6.575000
57302,8272b63d03f5f79c56e9e4120aec44ef,6,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,1.20,7.89,6.575000


In [647]:
items_products = order_items.merge(
    products[
        [
            "product_id",
            "product_category_name",
            "product_weight_g",
            "product_length_cm",
            "product_height_cm",
            "product_width_cm"
        ]
    ],
    on="product_id",
    how="left"
)

print("Broj stavki:", len(items_products))
print(
    "Broj stavki bez pronađenog proizvoda:",
    items_products["product_weight_g"].isna().sum()
)

Broj stavki: 112650
Broj stavki bez pronađenog proizvoda: 18


In [648]:
items_products["product_volume_cm3"] = (
    items_products["product_length_cm"]
    * items_products["product_height_cm"]
    * items_products["product_width_cm"]
)

items_products[
    [
        "product_weight_g",
        "product_volume_cm3",
        "price",
        "freight_value"
    ]
].describe()

,product_weight_g,product_volume_cm3,price,freight_value
count,112632.000000,112632.000000,112650.000000,112650.000000
mean,2093.672047,15243.709567,120.653739,19.990320
std,3751.596884,23418.524686,183.633928,15.806405
min,0.000000,168.000000,0.850000,0.000000
25%,300.000000,2851.500000,39.900000,13.080000
50%,700.000000,6480.000000,74.990000,16.260000
75%,1800.000000,18375.000000,134.900000,21.150000
max,40425.000000,296208.000000,6735.000000,409.680000


In [649]:
skupa_dostava = items_products[
    items_products["freight_value"] > items_products["price"]
]

skupa_dostava[
    [
        "price",
        "freight_value",
        "product_weight_g",
        "product_volume_cm3"
    ]
].describe()

,price,freight_value,product_weight_g,product_volume_cm3
count,4124.000000,4124.000000,4124.000000,4124.000000
mean,17.623378,24.332825,809.316440,8241.984724
std,15.994419,22.651010,2677.872122,21752.322030
min,0.850000,7.390000,50.000000,352.000000
25%,10.500000,15.100000,150.000000,1407.000000
50%,13.900000,18.230000,200.000000,3051.000000
75%,18.900000,25.630000,400.000000,5549.500000
max,189.900000,299.160000,30000.000000,274625.000000


In [650]:
print(
    "Broj jedinstvenih product_id iz order_items koji ne postoje u products:",
    (~order_items["product_id"].isin(products["product_id"])).sum()
)

items_products[
    items_products["product_weight_g"].isna()
][
    [
        "order_id",
        "product_id",
        "product_category_name",
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm"
    ]
].head(20)

Broj jedinstvenih product_id iz order_items koji ne postoje u products: 0


,order_id,product_id,product_category_name,product_weight_g,product_length_cm,product_height_cm,product_width_cm
7098,101157d4fae1c9fb74a00a5dee265c25,5eb564652db742ff8f28759cd8d2652a,NaN,NaN,NaN,NaN,NaN
9233,1521c6bb7b1028154c8c67cf80fa809f,5eb564652db742ff8f28759cd8d2652a,NaN,NaN,NaN,NaN,NaN
28715,415cfaaaa8cea49f934470548797fed1,5eb564652db742ff8f28759cd8d2652a,NaN,NaN,NaN,NaN,NaN
28716,415cfaaaa8cea49f934470548797fed1,5eb564652db742ff8f28759cd8d2652a,NaN,NaN,NaN,NaN,NaN
39299,595316a07cd3dea9db7adfcc7e247ae7,5eb564652db742ff8f28759cd8d2652a,NaN,NaN,NaN,NaN,NaN
48424,6e150190fbe04c642a9cf0b80d83ee16,5eb564652db742ff8f28759cd8d2652a,NaN,NaN,NaN,NaN,NaN
48980,6f497c40431d5fb0cfbd6c943dd29215,5eb564652db742ff8f28759cd8d2652a,NaN,NaN,NaN,NaN,NaN
58833,85f8ad45e067abd694b627859fa57453,09ff539a621711667c43eba6a3bd8466,bebes,NaN,NaN,NaN,NaN
71134,a2456e7f02197951664897a94c87242d,5eb564652db742ff8f28759cd8d2652a,NaN,NaN,NaN,NaN,NaN
73556,a7a43f469c0d7bdb0a23a82db125aefa,5eb564652db742ff8f28759cd8d2652a,NaN,NaN,NaN,NaN,NaN


### Analiza cijene proizvoda i troška dostave

U tablici stavki narudžbi nalazi se ukupno **112.650 zapisa**. Vrijednosti atributa `price` i `freight_value` potpune su, odnosno ne sadrže nedostajuće vrijednosti. Nisu pronađene negativne niti nulte cijene proizvoda, dok **383 stavke imaju trošak dostave jednak nuli**, što može predstavljati slučajeve besplatne dostave.

Medijan cijene proizvoda iznosi **74,99 BRL**, dok prosječna cijena iznosi približno **120,65 BRL**. Maksimalna zabilježena cijena iznosi **6.735 BRL**. Razlika između prosjeka i medijana upućuje na desno asimetričnu distribuciju cijena i prisutnost relativno malog broja vrlo skupih proizvoda.

Medijan troška dostave iznosi **16,26 BRL**, prosječni trošak približno **19,99 BRL**, dok je maksimalni zabilježeni trošak dostave **409,68 BRL**.

Analiziran je i odnos troška dostave i cijene proizvoda. Za **4.124 stavke** trošak dostave veći je od cijene samog proizvoda. Međutim, detaljnija analiza pokazuje da se uglavnom radi o proizvodima relativno niske cijene. Medijan cijene tih proizvoda iznosi samo **13,90 BRL**, dok je medijan troška dostave **18,23 BRL**. Istodobno su njihova medijalna težina (**200 g**) i volumen (**3.051 cm³**) manji od medijalne težine (**700 g**) i volumena (**6.480 cm³**) svih analiziranih stavki.

Prema tome, činjenica da je trošak dostave veći od cijene proizvoda sama po sebi ne predstavlja anomaliju. Takav odnos često je posljedica vrlo niske cijene proizvoda.

In [651]:
print(
    "Broj stavki s nedostajućim podatkom o težini proizvoda:",
    items_products["product_weight_g"].isna().sum()
)

print(
    "Broj stavki s nedostajućim podatkom o volumenu proizvoda:",
    items_products["product_volume_cm3"].isna().sum()
)

Broj stavki s nedostajućim podatkom o težini proizvoda: 18
Broj stavki s nedostajućim podatkom o volumenu proizvoda: 18


Sve stavke iz tablice `order_items` uspješno su povezane s odgovarajućim proizvodom u tablici `products`, odnosno nisu pronađeni `product_id` identifikatori koji ne postoje u tablici proizvoda. Za 18 stavki nedostaju podaci o pojedinim fizičkim karakteristikama proizvoda, zbog čega za njih nije moguće u potpunosti analizirati odnos težine i volumena proizvoda s troškom dostave. Ovi će se nedostajući podaci detaljnije obraditi u fazi čišćenja podataka.

## 8.7. Analiza plaćanja (`payments`)

Tablica `payments` sadrži podatke o načinima i iznosima plaćanja pojedinih narudžbi.

U ovom dijelu analize ispitat će se:
- broj zapisa i broj jedinstvenih narudžbi,
- broj plaćanja po narudžbi,
- korišteni načini plaćanja,
- broj rata,
- distribucija iznosa plaćanja,
- nedostajuće i potencijalno nelogične vrijednosti,
- narudžbe plaćene kroz više transakcija ili više načina plaćanja.



In [652]:
print("Broj zapisa:", len(payments))
print("Broj jedinstvenih narudžbi:", payments["order_id"].nunique())

print("\nStupci:")
print(payments.columns.tolist())

print("\nTipovi podataka:")
print(payments.dtypes)

print("\nNedostajuće vrijednosti:")
print(payments.isna().sum())

payments.head()

Broj zapisa: 103886
Broj jedinstvenih narudžbi: 99440

Stupci:
['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']

Tipovi podataka:
order_id                    str
payment_sequential        int64
payment_type                str
payment_installments      int64
payment_value           float64
dtype: object

Nedostajuće vrijednosti:
order_id                0
payment_sequential      0
payment_type            0
payment_installments    0
payment_value           0
dtype: int64


,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


In [653]:
broj_placanja_po_narudzbi = (
    payments
    .groupby("order_id")
    .size()
)

broj_placanja_po_narudzbi.describe()

count    99440.000000
mean         1.044710
std          0.381166
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max         29.000000
dtype: float64

In [654]:
broj_placanja_po_narudzbi.value_counts().sort_index()

1     96479
2      2382
3       301
4       108
5        52
6        36
7        28
8        11
9         9
10        5
11        8
12        8
13        3
14        2
15        2
19        2
21        1
22        1
26        1
29        1
Name: count, dtype: int64

In [655]:
print(
    "Narudžbe s jednim zapisom plaćanja:",
    (broj_placanja_po_narudzbi == 1).sum()
)

print(
    "Narudžbe s više zapisa plaćanja:",
    (broj_placanja_po_narudzbi > 1).sum()
)

print(
    "Maksimalan broj zapisa plaćanja za jednu narudžbu:",
    broj_placanja_po_narudzbi.max()
)

Narudžbe s jednim zapisom plaćanja: 96479
Narudžbe s više zapisa plaćanja: 2961
Maksimalan broj zapisa plaćanja za jednu narudžbu: 29


### Struktura zapisa o plaćanjima

Tablica `payments` sadrži 103.886 zapisa koji se odnose na 99.440 jedinstvenih narudžbi. Nedostajuće vrijednosti nisu pronađene ni u jednom stupcu.

Većina narudžbi (96.479) ima samo jedan zapis plaćanja, dok 2.961 narudžba ima više zapisa plaćanja. Maksimalan broj zapisa plaćanja povezanih s jednom narudžbom iznosi 29.

Više zapisa u tablici `payments` ne treba izravno poistovjetiti s plaćanjem na rate. Pojedini zapis plaćanja identificiran je atributom `payment_sequential`, dok je broj rata zasebno pohranjen u atributu `payment_installments`. Stoga je potrebno dodatno analizirati predstavljaju li višestruki zapisi kombiniranje različitih načina plaćanja ili više transakcija istog načina plaćanja.

Ovakva struktura podataka važna je za kasniju izgradnju skladišta podataka jer izravno povezivanje tablica koje sadrže više zapisa za istu narudžbu može dovesti do multipliciranja redaka i pogrešnog izračuna financijskih mjera.

In [656]:
placanja_po_narudzbi = (
    payments
    .groupby("order_id")
    .agg(
        broj_zapisa_placanja=("payment_sequential", "count"),
        broj_nacina_placanja=("payment_type", "nunique")
    )
)

placanja_po_narudzbi["vise_zapisa"] = (
    placanja_po_narudzbi["broj_zapisa_placanja"] > 1
)

placanja_po_narudzbi["vise_nacina_placanja"] = (
    placanja_po_narudzbi["broj_nacina_placanja"] > 1
)

placanja_po_narudzbi[
    placanja_po_narudzbi["vise_zapisa"]
].describe()

,broj_zapisa_placanja,broj_nacina_placanja
count,2961.00000,2961.000000
mean,2.50152,1.758528
std,1.64093,0.428048
min,2.00000,1.000000
25%,2.00000,2.000000
50%,2.00000,2.000000
75%,2.00000,2.000000
max,29.00000,2.000000


In [657]:
vise_placanja = placanja_po_narudzbi[
    placanja_po_narudzbi["broj_zapisa_placanja"] > 1
]

print(
    "Narudžbe s više zapisa plaćanja:",
    len(vise_placanja)
)

print(
    "Od toga s više različitih načina plaćanja:",
    (vise_placanja["broj_nacina_placanja"] > 1).sum()
)

print(
    "Od toga samo s jednim načinom plaćanja:",
    (vise_placanja["broj_nacina_placanja"] == 1).sum()
)

Narudžbe s više zapisa plaćanja: 2961
Od toga s više različitih načina plaćanja: 2246
Od toga samo s jednim načinom plaćanja: 715


In [658]:
kombinacije_placanja = (
    payments
    .groupby("order_id")["payment_type"]
    .agg(lambda x: " + ".join(sorted(set(x))))
)

kombinacije_placanja.value_counts()

payment_type
credit_card                 74259
boleto                      19784
credit_card + voucher        2245
voucher                      1621
debit_card                   1527
not_defined                     3
credit_card + debit_card        1
Name: count, dtype: int64

In [659]:
narudzbe_najvise_placanja = (
    broj_placanja_po_narudzbi
    .sort_values(ascending=False)
    .head(20)
    .index
)

payments[
    payments["order_id"].isin(narudzbe_najvise_placanja)
].sort_values(
    ["order_id", "payment_sequential"]
)

,order_id,payment_sequential,payment_type,payment_installments,payment_value
42442,1a611328643ae11146ba09a4425d2e12,1,credit_card,1,0.26
71197,1a611328643ae11146ba09a4425d2e12,2,voucher,1,6.34
55316,1a611328643ae11146ba09a4425d2e12,3,voucher,1,6.34
29808,1a611328643ae11146ba09a4425d2e12,4,voucher,1,6.34
64228,1a611328643ae11146ba09a4425d2e12,5,voucher,1,6.34
...,...,...,...,...,...
7548,fedcd9f7ccdc8cba3a18defedd1a5547,15,voucher,1,10.66
64520,fedcd9f7ccdc8cba3a18defedd1a5547,16,voucher,1,9.54
48288,fedcd9f7ccdc8cba3a18defedd1a5547,17,voucher,1,7.51
87292,fedcd9f7ccdc8cba3a18defedd1a5547,18,voucher,1,11.12


### Višestruki zapisi plaćanja po narudžbi

Od ukupno 99.440 narudžbi evidentiranih u tablici `payments`, 2.961 narudžba ima više od jednog zapisa plaćanja. Analizom načina plaćanja utvrđeno je da 2.246 takvih narudžbi koristi više različitih načina plaćanja, dok 715 narudžbi ima više zapisa koji pripadaju istom načinu plaćanja.

Najčešća kombinacija različitih načina plaćanja je `credit_card + voucher`, koja se pojavljuje kod 2.245 narudžbi. Kombinacija `credit_card + debit_card` pojavljuje se samo kod jedne narudžbe.

Pregledom narudžbi s najvećim brojem zapisa plaćanja utvrđeno je da veliki broj zapisa uglavnom proizlazi iz korištenja više vouchera za plaćanje iste narudžbe. Primjerice, narudžba s maksimalnih 29 zapisa plaćanja sastoji se isključivo od plaćanja voucherima. Kod drugih narudžbi s velikim brojem zapisa pojavljuju se plaćanja samo voucherima ili kombinacija kartičnog plaćanja i više vouchera.

Prema tome, atribut `payment_sequential` označava redoslijed pojedinih zapisa plaćanja unutar iste narudžbe i ne predstavlja broj rata. Broj rata zasebno je zabilježen atributom `payment_installments`.


In [660]:
payments["payment_type"].value_counts()

payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

In [661]:
payments["payment_type"].value_counts(normalize=True).mul(100).round(3)

payment_type
credit_card    73.922
boleto         19.044
voucher         5.559
debit_card      1.472
not_defined     0.003
Name: proportion, dtype: float64

In [662]:
nacini_po_narudzbi = (
    payments
    .groupby("order_id")["payment_type"]
    .agg(lambda x: sorted(set(x)))
)

print("Broj narudžbi koje koriste pojedini način plaćanja:")

for tip in payments["payment_type"].unique():
    broj = nacini_po_narudzbi.apply(
        lambda x: tip in x
    ).sum()

    print(f"{tip}: {broj}")

Broj narudžbi koje koriste pojedini način plaćanja:
credit_card: 76505
boleto: 19784
voucher: 3866
debit_card: 1528
not_defined: 3


In [663]:
payments[
    payments["payment_type"] == "not_defined"
]

,order_id,payment_sequential,payment_type,payment_installments,payment_value
51280,4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.0
57411,00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.0
94427,c8c528189310eaa44a745b8d9d26908b,1,not_defined,1,0.0


In [664]:
not_defined_detalji = (
    payments[
        payments["payment_type"] == "not_defined"
    ]
    .merge(
        orders[
            [
                "order_id",
                "customer_id",
                "order_status",
                "order_purchase_timestamp",
                "order_approved_at",
                "order_delivered_carrier_date",
                "order_delivered_customer_date",
                "order_estimated_delivery_date"
            ]
        ],
        on="order_id",
        how="left"
    )
    .merge(
        order_items[
            [
                "order_id",
                "order_item_id",
                "product_id",
                "seller_id",
                "price",
                "freight_value"
            ]
        ],
        on="order_id",
        how="left"
    )
)

not_defined_detalji[
    [
        "order_id",
        "order_status",
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "payment_sequential",
        "payment_installments",
        "payment_value",
        "order_item_id",
        "product_id",
        "seller_id",
        "price",
        "freight_value"
    ]
]

,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,payment_sequential,payment_installments,payment_value,order_item_id,product_id,seller_id,price,freight_value
0,4637ca194b6387e2d538dc89b124b0ee,canceled,2018-09-03 14:14:25,NaT,NaT,NaT,1,1,0.0,NaN,NaN,NaN,NaN,NaN
1,00b1cb0320190ca0daa2c88b35206009,canceled,2018-08-28 15:26:39,NaT,NaT,NaT,1,1,0.0,NaN,NaN,NaN,NaN,NaN
2,c8c528189310eaa44a745b8d9d26908b,canceled,2018-08-28 20:05:14,NaT,NaT,NaT,1,1,0.0,NaN,NaN,NaN,NaN,NaN


In [665]:
not_defined_order_ids = payments.loc[
    payments["payment_type"] == "not_defined",
    "order_id"
].unique()

print(
    "Broj not_defined narudžbi:",
    len(not_defined_order_ids)
)

print(
    "Broj koji postoji u orders:",
    orders["order_id"].isin(not_defined_order_ids).sum()
)

print(
    "Broj koji postoji u order_items:",
    order_items["order_id"].isin(not_defined_order_ids).sum()
)

Broj not_defined narudžbi: 3
Broj koji postoji u orders: 3
Broj koji postoji u order_items: 0


### Analiza zapisa s načinom plaćanja `not_defined`

U tablici `payments` pronađena su samo **3 zapisa** kod kojih je način plaćanja označen kao `not_defined`. Detaljnijom analizom utvrđeno je da sve tri narudžbe:

- imaju status `canceled`,
- imaju vrijednost plaćanja (`payment_value`) jednaku 0,
- nemaju zabilježen datum odobrenja narudžbe (`order_approved_at`),
- nemaju datum predaje narudžbe prijevozniku (`order_delivered_carrier_date`),
- nemaju datum dostave kupcu (`order_delivered_customer_date`),
- nemaju pripadajuće stavke u tablici `order_items`.

Na temelju navedenog može se zaključiti da se ne radi o realiziranim kupnjama s nepoznatim načinom plaćanja, već o otkazanim narudžbama koje nisu prošle kroz proces odobrenja i naplate. Vrijednost `not_defined` stoga je u ovom slučaju logična posljedica činjenice da plaćanje nije realizirano.

**Konačna odluka:** vrijednosti `not_defined` neće se imputirati niti zamjenjivati nekim od postojećih načina plaćanja jer za takvu zamjenu ne postoji opravdana osnova. Zapisi se mogu zadržati u izvornim podacima radi očuvanja potpunosti podataka, pri čemu se `not_defined` tretira kao posebna kategorija koja označava da način plaćanja nije definiran. Međutim, pri analizama realizirane prodaje i načina plaćanja uspješno realiziranih narudžbi ove tri otkazane narudžbe neće biti uključene.

In [666]:
# Osnovna statistika vrijednosti plaćanja i broja rata
payments[
    [
        "payment_value",
        "payment_installments"
    ]
].describe()

,payment_value,payment_installments
count,103886.000000,103886.000000
mean,154.100380,2.853349
std,217.494064,2.687051
min,0.000000,0.000000
25%,56.790000,1.000000
50%,100.000000,1.000000
75%,171.837500,4.000000
max,13664.080000,24.000000


In [667]:
print(
    "Broj zapisa s payment_value < 0:",
    (payments["payment_value"] < 0).sum()
)

print(
    "Broj zapisa s payment_value = 0:",
    (payments["payment_value"] == 0).sum()
)

print(
    "Broj zapisa s payment_installments < 0:",
    (payments["payment_installments"] < 0).sum()
)

print(
    "Broj zapisa s payment_installments = 0:",
    (payments["payment_installments"] == 0).sum()
)

print(
    "Maksimalan broj rata:",
    payments["payment_installments"].max()
)

Broj zapisa s payment_value < 0: 0
Broj zapisa s payment_value = 0: 9
Broj zapisa s payment_installments < 0: 0
Broj zapisa s payment_installments = 0: 2
Maksimalan broj rata: 24


In [668]:
payments[
    payments["payment_installments"] == 0
][
    [
        "order_id",
        "payment_sequential",
        "payment_type",
        "payment_installments",
        "payment_value"
    ]
]

,order_id,payment_sequential,payment_type,payment_installments,payment_value
46982,744bade1fcf9ff3f31d860ace076d422,2,credit_card,0,58.69
79014,1a57108394169c0b47d8f876acc9ba2d,2,credit_card,0,129.94


In [669]:
orders_zero_installments = payments.loc[
    payments["payment_installments"] == 0,
    "order_id"
].unique()

payments[
    payments["order_id"].isin(orders_zero_installments)
].sort_values(
    ["order_id", "payment_sequential"]
)

,order_id,payment_sequential,payment_type,payment_installments,payment_value
79014,1a57108394169c0b47d8f876acc9ba2d,2,credit_card,0,129.94
46982,744bade1fcf9ff3f31d860ace076d422,2,credit_card,0,58.69


In [670]:
zero_payment_orders = payments[
    payments["payment_value"] == 0
]

zero_payment_orders[
    [
        "order_id",
        "payment_sequential",
        "payment_type",
        "payment_installments",
        "payment_value"
    ]
]

,order_id,payment_sequential,payment_type,payment_installments,payment_value
19922,8bcbe01d44d147f901cd3192671144db,4,voucher,1,0.0
36822,fa65dad1b0e818e3ccc5cb0e39231352,14,voucher,1,0.0
43744,6ccb433e00daae1283ccc956189c82ae,4,voucher,1,0.0
51280,4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.0
57411,00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.0
62674,45ed6e85398a87c253db47c2d9f48216,3,voucher,1,0.0
77885,fa65dad1b0e818e3ccc5cb0e39231352,13,voucher,1,0.0
94427,c8c528189310eaa44a745b8d9d26908b,1,not_defined,1,0.0
100766,b23878b3e8eb4d25a158f57d96331b18,4,voucher,1,0.0


In [671]:
zero_payment_detalji = (
    zero_payment_orders
    .merge(
        orders[
            [
                "order_id",
                "order_status",
                "order_purchase_timestamp",
                "order_approved_at",
                "order_delivered_carrier_date",
                "order_delivered_customer_date"
            ]
        ],
        on="order_id",
        how="left"
    )
)

zero_payment_detalji

,order_id,payment_sequential,payment_type,payment_installments,payment_value,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date
0,8bcbe01d44d147f901cd3192671144db,4,voucher,1,0.0,delivered,2018-01-24 23:24:14,2018-01-24 23:37:20,2018-01-30 22:43:30,2018-02-02 21:26:42
1,fa65dad1b0e818e3ccc5cb0e39231352,14,voucher,1,0.0,shipped,2017-04-20 12:45:34,2017-04-22 09:10:13,2017-04-24 11:31:17,NaT
2,6ccb433e00daae1283ccc956189c82ae,4,voucher,1,0.0,delivered,2017-10-26 23:51:35,2017-10-27 00:46:58,2017-10-27 19:53:03,2017-11-08 18:47:00
3,4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.0,canceled,2018-09-03 14:14:25,NaT,NaT,NaT
4,00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.0,canceled,2018-08-28 15:26:39,NaT,NaT,NaT
5,45ed6e85398a87c253db47c2d9f48216,3,voucher,1,0.0,delivered,2017-06-08 21:18:45,2017-06-08 21:30:18,2017-06-09 08:35:18,2017-06-22 17:44:53
6,fa65dad1b0e818e3ccc5cb0e39231352,13,voucher,1,0.0,shipped,2017-04-20 12:45:34,2017-04-22 09:10:13,2017-04-24 11:31:17,NaT
7,c8c528189310eaa44a745b8d9d26908b,1,not_defined,1,0.0,canceled,2018-08-28 20:05:14,NaT,NaT,NaT
8,b23878b3e8eb4d25a158f57d96331b18,4,voucher,1,0.0,delivered,2017-05-27 19:20:24,2017-05-27 19:35:17,2017-05-29 04:49:37,2017-06-07 08:53:10


In [672]:
zero_payment_order_ids = zero_payment_orders["order_id"].unique()

payments[
    payments["order_id"].isin(zero_payment_order_ids)
].sort_values(
    ["order_id", "payment_sequential"]
)

,order_id,payment_sequential,payment_type,payment_installments,payment_value
57411,00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.00
33781,45ed6e85398a87c253db47c2d9f48216,1,voucher,1,21.13
11755,45ed6e85398a87c253db47c2d9f48216,2,voucher,1,50.01
62674,45ed6e85398a87c253db47c2d9f48216,3,voucher,1,0.00
51280,4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.00
40546,6ccb433e00daae1283ccc956189c82ae,1,credit_card,5,84.67
93478,6ccb433e00daae1283ccc956189c82ae,2,voucher,1,14.65
92318,6ccb433e00daae1283ccc956189c82ae,3,voucher,1,22.72
43744,6ccb433e00daae1283ccc956189c82ae,4,voucher,1,0.00
20963,8bcbe01d44d147f901cd3192671144db,1,credit_card,1,36.21


### Zapisi s `payment_installments = 0`

Pronađena su samo 2 zapisa kod kojih je broj rata jednak 0. Oba zapisa odnose se na plaćanje kreditnom karticom i imaju pozitivan iznos plaćanja. Budući da vrijednost 0 nije očekivana za broj rata kod realiziranog kartičnog plaćanja, ovi se zapisi smatraju potencijalno nekonzistentnima.

### Zapisi s `payment_value = 0`

Pronađeno je ukupno 9 zapisa plaćanja s vrijednošću `payment_value = 0`.

Tri zapisa odnose se na otkazane narudžbe s načinom plaćanja `not_defined`. Te narudžbe nisu odobrene ni realizirane te nemaju pripadajuće stavke, pa je vrijednost plaćanja 0 očekivana.

Preostalih šest zapisa odnosi se na plaćanja voucherima. Detaljnijom analizom utvrđeno je da se radi o narudžbama koje imaju više zapisa plaćanja, pri čemu su stvarni pozitivni iznosi evidentirani u drugim zapisima iste narudžbe. Stoga pojedinačni voucher zapis s vrijednošću 0 ne znači da cijela narudžba nije plaćena.


In [673]:
# Ukupno plaćeno po narudžbi
placanje_po_narudzbi = (
    payments
    .groupby("order_id")
    .agg(
        ukupno_placeno=("payment_value", "sum")
    )
    .reset_index()
)

# Ukupna vrijednost proizvoda i dostave po narudžbi
vrijednost_po_narudzbi = (
    order_items
    .groupby("order_id")
    .agg(
        ukupna_cijena_proizvoda=("price", "sum"),
        ukupna_dostava=("freight_value", "sum")
    )
    .reset_index()
)

vrijednost_po_narudzbi["ocekivani_iznos"] = (
    vrijednost_po_narudzbi["ukupna_cijena_proizvoda"]
    + vrijednost_po_narudzbi["ukupna_dostava"]
)

In [674]:
usporedba_placanja = placanje_po_narudzbi.merge(
    vrijednost_po_narudzbi,
    on="order_id",
    how="outer",
    indicator=True
)

usporedba_placanja["_merge"].value_counts()

_merge
both          98665
left_only       775
right_only        1
Name: count, dtype: int64

In [675]:
left_only_ids = usporedba_placanja.loc[
    usporedba_placanja["_merge"] == "left_only",
    "order_id"
]

left_only_detalji = (
    payments[
        payments["order_id"].isin(left_only_ids)
    ]
    .groupby("order_id", as_index=False)
    .agg(
        ukupno_placeno=("payment_value", "sum"),
        broj_zapisa_placanja=("payment_sequential", "count"),
        nacini_placanja=(
            "payment_type",
            lambda x: " + ".join(sorted(set(x)))
        )
    )
    .merge(
        orders[
            [
                "order_id",
                "customer_id",
                "order_status",
                "order_purchase_timestamp",
                "order_approved_at",
                "order_delivered_carrier_date",
                "order_delivered_customer_date"
            ]
        ],
        on="order_id",
        how="left"
    )
)

print(
    "Broj left_only narudžbi:",
    len(left_only_detalji)
)

left_only_detalji[
    "order_status"
].value_counts(dropna=False)

Broj left_only narudžbi: 775


order_status
unavailable    603
canceled       164
created          5
invoiced         2
shipped          1
Name: count, dtype: int64

In [676]:
left_only_detalji.groupby(
    "order_status"
).agg(
    broj_narudzbi=("order_id", "nunique"),
    prosjecno_placeno=("ukupno_placeno", "mean"),
    medijan_placeno=("ukupno_placeno", "median"),
    minimalno_placeno=("ukupno_placeno", "min"),
    maksimalno_placeno=("ukupno_placeno", "max")
).sort_values(
    "broj_narudzbi",
    ascending=False
)

,broj_narudzbi,prosjecno_placeno,medijan_placeno,minimalno_placeno,maksimalno_placeno
order_status,,,,,
unavailable,603,206.200697,104.350,16.95,3782.19
canceled,164,227.669939,110.610,0.00,3184.34
created,5,137.620000,137.600,37.84,245.85
invoiced,2,74.615000,74.615,73.04,76.19
shipped,1,77.730000,77.730,77.73,77.73


In [677]:
right_only_ids = usporedba_placanja.loc[
    usporedba_placanja["_merge"] == "right_only",
    "order_id"
]

right_only_detalji = (
    order_items[
        order_items["order_id"].isin(right_only_ids)
    ]
    .merge(
        orders[
            [
                "order_id",
                "customer_id",
                "order_status",
                "order_purchase_timestamp",
                "order_approved_at",
                "order_delivered_carrier_date",
                "order_delivered_customer_date",
                "order_estimated_delivery_date"
            ]
        ],
        on="order_id",
        how="left"
    )
)

right_only_detalji[
    [
        "order_id",
        "order_status",
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_item_id",
        "product_id",
        "seller_id",
        "price",
        "freight_value"
    ]
]

,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_item_id,product_id,seller_id,price,freight_value
0,bfbd0f9bdef84302105ad712db648a6c,delivered,2016-09-15 12:16:38,2016-09-15 12:16:38,2016-11-07 17:11:53,2016-11-09 07:47:38,1,5a6b04657a4c5ee34285d1e4619a96b4,ecccfa2bb93b34a3bf033cc5d1dcdc69,44.99,2.83
1,bfbd0f9bdef84302105ad712db648a6c,delivered,2016-09-15 12:16:38,2016-09-15 12:16:38,2016-11-07 17:11:53,2016-11-09 07:47:38,2,5a6b04657a4c5ee34285d1e4619a96b4,ecccfa2bb93b34a3bf033cc5d1dcdc69,44.99,2.83
2,bfbd0f9bdef84302105ad712db648a6c,delivered,2016-09-15 12:16:38,2016-09-15 12:16:38,2016-11-07 17:11:53,2016-11-09 07:47:38,3,5a6b04657a4c5ee34285d1e4619a96b4,ecccfa2bb93b34a3bf033cc5d1dcdc69,44.99,2.83


### Narudžbe prisutne samo u jednoj od uspoređivanih tablica

Pronađeno je 775 narudžbi koje imaju zapise u tablici `payments`, ali nemaju pripadajuće stavke u tablici `order_items`. Većinu čine narudžbe sa statusima `unavailable` i `canceled`.

Budući da za te narudžbe nisu dostupni podaci o konkretnim proizvodima, prodavačima, cijenama stavki i troškovima dostave, one se neće koristiti u analizama prodaje na razini proizvoda. Mogu se, međutim, zadržati za analizu statusa i životnog ciklusa narudžbi.

Pronađena je i jedna narudžba koja postoji u tablici `order_items`, ali nema odgovarajući zapis u tablici `payments`. Riječ je o dostavljenoj narudžbi za koju postoje podaci o stavkama, ali nedostaju podaci o načinu i vrijednosti plaćanja. Ti se podaci neće imputirati jer ih nije moguće pouzdano rekonstruirati.

In [678]:
import numpy as np

# Razlika između ukupno plaćenog i očekivanog iznosa
usporedba_placanja["razlika"] = (
    usporedba_placanja["ukupno_placeno"]
    - usporedba_placanja["ocekivani_iznos"]
)

# Promatramo samo narudžbe koje postoje u obje tablice
mask_both = (
    usporedba_placanja["_merge"] == "both"
)

# Provjera podudaranja uz toleranciju od jednog centa
usporedba_placanja["iznosi_se_podudaraju"] = np.isclose(
    usporedba_placanja["ukupno_placeno"],
    usporedba_placanja["ocekivani_iznos"],
    atol=0.01,
    rtol=0
)

print(
    "Broj narudžbi s podudarnim iznosom:",
    (
        mask_both
        & usporedba_placanja["iznosi_se_podudaraju"]
    ).sum()
)

print(
    "Broj narudžbi s nepodudarnim iznosom:",
    (
        mask_both
        & ~usporedba_placanja["iznosi_se_podudaraju"]
    ).sum()
)

len(mask_both)

Broj narudžbi s podudarnim iznosom: 98284
Broj narudžbi s nepodudarnim iznosom: 381


99441

In [679]:
nepodudarna_placanja = usporedba_placanja[
    (usporedba_placanja["_merge"] == "both") &
    (~usporedba_placanja["iznosi_se_podudaraju"])
].copy()

print(
    "Broj narudžbi s nepodudarnim iznosom:",
    len(nepodudarna_placanja)
)

Broj narudžbi s nepodudarnim iznosom: 381


In [680]:
nepodudarna_placanja_detalji = (
    nepodudarna_placanja[
        [
            "order_id",
            "ukupno_placeno",
            "ukupna_cijena_proizvoda",
            "ukupna_dostava",
            "ocekivani_iznos",
            "razlika"
        ]
    ]
    .merge(
        payments[
            [
                "order_id",
                "payment_sequential",
                "payment_type",
                "payment_installments",
                "payment_value"
            ]
        ],
        on="order_id",
        how="left"
    )
)

nepodudarna_placanja_detalji

,order_id,ukupno_placeno,ukupna_cijena_proizvoda,ukupna_dostava,ocekivani_iznos,razlika,payment_sequential,payment_type,payment_installments,payment_value
0,00789ce015e7e5791c7914f32bb4fad4,190.81,154.00,14.83,168.83,21.98,1,credit_card,10,190.81
1,016726239765c18f66826453f39c64e3,265.77,219.99,15.14,235.13,30.64,1,credit_card,10,265.77
2,01e51b7c3025655646143d09b911e1d7,35.02,19.00,14.10,33.10,1.92,1,credit_card,3,35.02
3,028aa7c930356788f861ed1b7f984819,62.94,39.90,17.63,57.53,5.41,1,credit_card,6,62.94
4,02f4dd90ba0feb8ec394cac05862d2b5,141.65,79.90,51.06,130.96,10.69,1,credit_card,6,141.65
...,...,...,...,...,...,...,...,...,...,...
399,fc0f2134e9a7cffb77d02c4754c264e8,193.67,157.80,35.88,193.68,-0.01,1,credit_card,3,193.67
400,fd33085945f15975375cd8ec85440511,212.82,217.99,16.63,234.62,-21.80,1,credit_card,6,212.82
401,fdb034f444a32838124d8907e443d249,33.56,18.90,12.48,31.38,2.18,1,credit_card,3,33.56
402,fe35e99a6a5d4d036bf8167d364e0792,262.59,239.60,23.00,262.60,-0.01,1,credit_card,1,262.59


In [681]:
print(
    "Broj redaka:",
    len(nepodudarna_placanja_detalji)
)

print(
    "Broj jedinstvenih narudžbi:",
    nepodudarna_placanja_detalji["order_id"].nunique()
)

Broj redaka: 404
Broj jedinstvenih narudžbi: 381


In [682]:
kombinacije_nepodudarnih = (
    nepodudarna_placanja_detalji
    .groupby("order_id")["payment_type"]
    .agg(
        lambda x: " + ".join(
            sorted(set(x))
        )
    )
)

kombinacije_nepodudarnih.value_counts()

payment_type
credit_card              335
boleto                    26
credit_card + voucher     10
debit_card                 9
voucher                    1
Name: count, dtype: int64

In [683]:
koristi_voucher = (
    nepodudarna_placanja_detalji
    .groupby("order_id")["payment_type"]
    .apply(
        lambda x: "voucher" in set(x)
    )
)

print(
    "Nepodudarne narudžbe koje koriste voucher:",
    koristi_voucher.sum()
)

print(
    "Nepodudarne narudžbe bez vouchera:",
    (~koristi_voucher).sum()
)

Nepodudarne narudžbe koje koriste voucher: 11
Nepodudarne narudžbe bez vouchera: 370


In [684]:
nepodudarna_placanja[
    [
        "order_id",
        "ukupno_placeno",
        "ukupna_cijena_proizvoda",
        "ukupna_dostava",
        "ocekivani_iznos",
        "razlika"
    ]
].sort_values(
    "razlika",
    key=lambda x: x.abs(),
    ascending=False
)

nepodudarna_placanja["order_id"].nunique()

381

In [685]:
usporedive = usporedba_placanja[
    usporedba_placanja["_merge"] == "both"
]

broj_podudarnih = usporedive["iznosi_se_podudaraju"].sum()
broj_nepodudarnih = (~usporedive["iznosi_se_podudaraju"]).sum()

print("Ukupno uspoređenih:", len(usporedive))
print("Podudarni iznosi:", broj_podudarnih)
print("Nepodudarni iznosi:", broj_nepodudarnih)

assert broj_podudarnih + broj_nepodudarnih == len(usporedive)
assert usporedive["order_id"].is_unique

Ukupno uspoređenih: 98665
Podudarni iznosi: 98284
Nepodudarni iznosi: 381


### Zaključak analize financijske konzistentnosti

Usporedbom tablica `payments` i `order_items` identificirano je 98.665 narudžbi koje imaju podatke u obje tablice.

Za usporedbu ukupno plaćenog i očekivanog iznosa korištena je funkcija `np.isclose` s apsolutnom tolerancijom od 0,01. Na taj se način smanjuje utjecaj numeričke reprezentacije decimalnih brojeva bez izmjene izvornih novčanih vrijednosti.

Utvrđeno je da se kod **98.284 narudžbi** ukupno plaćeni iznos podudara sa zbrojem cijena proizvoda i troškova dostave, dok je kod **381 narudžbi** pronađeno odstupanje veće od definirane tolerancije.

Rezultati pokazuju vrlo visoku razinu financijske konzistentnosti između tablica `payments` i `order_items`. Narudžbe s nepodudarnim iznosima u ovoj se fazi ne uklanjaju niti se njihove vrijednosti automatski mijenjaju, već se zadržavaju radi daljnje analize mogućih uzroka odstupanja.

## 8.8. Analiza recenzija (`reviews`)

Tablica `reviews` sadrži podatke o ocjenama i komentarima koje su kupci ostavili nakon kupnje. Cilj analize je ispitati strukturu i kvalitetu podataka o recenzijama, provjeriti njihovu povezanost s narudžbama te istražiti potencijalne anomalije u ocjenama i vremenskim podacima.


In [686]:
print("Broj zapisa:", len(reviews))
print("Broj jedinstvenih review_id:", reviews["review_id"].nunique())
print("Broj jedinstvenih order_id:", reviews["order_id"].nunique())

print("\nStupci:")
print(reviews.columns.tolist())

print("\nTipovi podataka:")
print(reviews.dtypes)

print("\nNedostajuće vrijednosti:")
print(reviews.isna().sum())

reviews.head()

Broj zapisa: 99224
Broj jedinstvenih review_id: 98410
Broj jedinstvenih order_id: 98673

Stupci:
['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp']

Tipovi podataka:
review_id                             str
order_id                              str
review_score                        int64
review_comment_title                  str
review_comment_message                str
review_creation_date       datetime64[us]
review_answer_timestamp    datetime64[us]
dtype: object

Nedostajuće vrijednosti:
review_id                      0
order_id                       0
review_score                   0
review_comment_title       87656
review_comment_message     58247
review_creation_date           0
review_answer_timestamp        0
dtype: int64


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01,2018-03-02 10:26:53


In [687]:
broj_recenzija_po_narudzbi = (
    reviews
    .groupby("order_id")
    .size()
)

print(
    "Narudžbe s jednom recenzijom:",
    (broj_recenzija_po_narudzbi == 1).sum()
)

print(
    "Narudžbe s više recenzija:",
    (broj_recenzija_po_narudzbi > 1).sum()
)

print(
    "Maksimalan broj recenzija za jednu narudžbu:",
    broj_recenzija_po_narudzbi.max()
)

broj_recenzija_po_narudzbi.value_counts().sort_index()

Narudžbe s jednom recenzijom: 98126
Narudžbe s više recenzija: 547
Maksimalan broj recenzija za jednu narudžbu: 3


1    98126
2      543
3        4
Name: count, dtype: int64

In [688]:
# Analiza dupliciranih review_id vrijednosti

duplicirani_review_id = reviews[
    reviews["review_id"].duplicated(keep=False)
].sort_values(
    ["review_id", "order_id", "review_creation_date"]
)

print(
    "Broj review_id vrijednosti koje se pojavljuju više puta:",
    duplicirani_review_id["review_id"].nunique()
)

print(
    "Ukupan broj zapisa s dupliciranim review_id:",
    len(duplicirani_review_id)
)

duplicirani_review_id

Broj review_id vrijednosti koje se pojavljuju više puta: 789
Ukupan broj zapisa s dupliciranim review_id: 1603


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
29841,00130cbe1f9d422698c812ed8ded1919,04a28263e085d399c97ae49e0b477efa,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07,2018-03-20 18:08:23
46678,00130cbe1f9d422698c812ed8ded1919,dfcdfc43867d1c1381bfaf62d6b9c195,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07,2018-03-20 18:08:23
63193,0115633a9c298b6a98bcbe4eee75345f,0c9850b2c179c1ef60d2855e2751d1fa,5,NaN,NaN,2017-09-21,2017-09-26 03:27:47
90677,0115633a9c298b6a98bcbe4eee75345f,78a4201f58af3463bdab842eea4bc801,5,NaN,NaN,2017-09-21,2017-09-26 03:27:47
57280,0174caf0ee5964646040cd94e15ac95e,74db91e33b4e1fd865356c89a61abf1f,1,NaN,Produto entregue dentro de embalagem do fornec...,2018-03-07,2018-03-08 03:00:53
...,...,...,...,...,...,...,...
40378,fe5c833752953fed3209646f1f63b53c,d3775e436e60258e62e678a0f68a0f8d,1,NaN,"Comprei dois produtos e ambos, mesmo enviados ...",2018-02-28,2018-02-28 13:57:52
82521,ff2fc9e68f8aabfbe18d710b83aabd30,1078d496cc6ab9a8e6f2be77abf5091b,2,NaN,NaN,2018-03-17,2018-03-19 11:44:15
7870,ff2fc9e68f8aabfbe18d710b83aabd30,2da58e0a7dcfa4ce1e00fad9d03ca3b5,2,NaN,NaN,2018-03-17,2018-03-19 11:44:15
73951,ffb8cff872a625632ac983eb1f88843c,c44883fc2529b4aa03ca90e7e09d95b6,3,NaN,NaN,2017-07-22,2017-07-26 13:41:07


In [689]:
# Analiza narudžbi koje imaju više zapisa recenzija

order_ids_vise_recenzija = broj_recenzija_po_narudzbi[
    broj_recenzija_po_narudzbi > 1
].index

vise_recenzija = (
    reviews[
        reviews["order_id"].isin(order_ids_vise_recenzija)
    ]
    .sort_values(
        ["order_id", "review_creation_date", "review_answer_timestamp"]
    )
)

print(
    "Broj narudžbi s više zapisa recenzija:",
    vise_recenzija["order_id"].nunique()
)

print(
    "Ukupan broj zapisa za te narudžbe:",
    len(vise_recenzija)
)

vise_recenzija

Broj narudžbi s više zapisa recenzija: 547
Ukupan broj zapisa za te narudžbe: 1098


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
22423,2a74b0559eb58fc1ff842ecc999594cb,0035246a40f520710769010f752e7507,5,NaN,Estou acostumada a comprar produtos pelo barat...,2017-08-25,2017-08-29 21:45:57
25612,89a02c45c340aeeb1354a24e7d4b2c1e,0035246a40f520710769010f752e7507,5,NaN,NaN,2017-08-29,2017-08-30 01:59:12
22779,ab30810c29da5da8045216f0f62652a2,013056cfe49763c6f66bda03396c5ee3,5,NaN,NaN,2018-02-22,2018-02-23 12:12:30
68633,73413b847f63e02bc752b364f6d05ee9,013056cfe49763c6f66bda03396c5ee3,4,NaN,NaN,2018-03-04,2018-03-05 17:02:00
854,830636803620cdf8b6ffaf1b2f6e92b2,0176a6846bcb3b0d3aa3116a9a768597,5,NaN,NaN,2017-12-30,2018-01-02 10:54:06
...,...,...,...,...,...,...,...
27465,5e78482ee783451be6026e5cf0c72de1,ff763b73e473d03c321bcd5a053316e8,3,NaN,Não sei que haverá acontecido os demais chegaram,2017-11-18,2017-11-18 09:02:48
41355,39de8ad3a1a494fc68cc2d5382f052f4,ff850ba359507b996e8b2fbb26df8d03,5,NaN,Envio rapido... Produto 100%,2017-08-16,2017-08-17 11:56:55
18783,80f25f32c00540d49d57796fb6658535,ff850ba359507b996e8b2fbb26df8d03,5,NaN,"Envio rapido, produto conforme descrito no anu...",2017-08-22,2017-08-25 11:40:22
92230,870d856a4873d3a67252b0c51d79b950,ffaabba06c9d293a3c614e0515ddbabc,3,NaN,NaN,2017-12-20,2017-12-20 18:50:16


In [690]:
print(
    "Broj potpuno dupliciranih redaka:",
    reviews.duplicated().sum()
)

Broj potpuno dupliciranih redaka: 0


In [691]:
# Provjera koliko je različitih narudžbi povezano s istim review_id

review_id_analiza = (
    reviews
    .groupby("review_id")
    .agg(
        broj_zapisa=("order_id", "size"),
        broj_narudzbi=("order_id", "nunique"),
        broj_ocjena=("review_score", "nunique")
    )
)

print(
    "Review ID povezan samo s jednom narudžbom:",
    (review_id_analiza["broj_narudzbi"] == 1).sum()
)

print(
    "Review ID povezan s više različitih narudžbi:",
    (review_id_analiza["broj_narudzbi"] > 1).sum()
)

print(
    "Maksimalan broj narudžbi povezanih s jednim review_id:",
    review_id_analiza["broj_narudzbi"].max()
)

review_id_analiza[
    review_id_analiza["broj_narudzbi"] > 1
].sort_values(
    "broj_narudzbi",
    ascending=False
)

Review ID povezan samo s jednom narudžbom: 97621
Review ID povezan s više različitih narudžbi: 789
Maksimalan broj narudžbi povezanih s jednim review_id: 3


,broj_zapisa,broj_narudzbi,broj_ocjena
review_id,,,
4219a80ab469e3fc9901437b73da3f75,3,3,1
7b606b0d57b078384f0b58eac1d41d78,3,3,1
0c76e7a547a531e7bf9f0b99cba071c1,3,3,1
69a1068c3128a14994e3e422e4539e04,3,3,1
308316408775d1600dad81bd3184556d,3,3,1
...,...,...,...
fde2e6abaf5bb64f7407a44741c24dec,2,2,1
fde5986d35c89aa1b6ce4149de82a0d3,2,2,1
fe5c833752953fed3209646f1f63b53c,2,2,1


In [692]:
# Provjera razlikuju li se višestruke recenzije iste narudžbe

analiza_vise_recenzija = (
    vise_recenzija
    .groupby("order_id")
    .agg(
        broj_zapisa=("review_id", "size"),
        broj_review_id=("review_id", "nunique"),
        broj_razlicitih_ocjena=("review_score", "nunique"),
        minimalna_ocjena=("review_score", "min"),
        maksimalna_ocjena=("review_score", "max")
    )
)

print(
    "Narudžbe s više zapisa, ali istom ocjenom:",
    (analiza_vise_recenzija["broj_razlicitih_ocjena"] == 1).sum()
)

print(
    "Narudžbe s različitim ocjenama:",
    (analiza_vise_recenzija["broj_razlicitih_ocjena"] > 1).sum()
)

analiza_vise_recenzija[
    analiza_vise_recenzija["broj_razlicitih_ocjena"] > 1
].sort_values(
    "broj_zapisa",
    ascending=False
)

Narudžbe s više zapisa, ali istom ocjenom: 345
Narudžbe s različitim ocjenama: 202


,broj_zapisa,broj_review_id,broj_razlicitih_ocjena,minimalna_ocjena,maksimalna_ocjena
order_id,,,,,
03c939fd7fd3b38f8485a0f95798f1f6,3,3,2,3,4
c88b1d1b157a9999ce368f218a407141,3,3,2,3,5
02355020fd0a40a0d56df9f6ff060413,2,2,2,1,3
029863af4b968de1e5d6a82782e662f5,2,2,2,4,5
03eba6d9fef8f5b3e811d4b5a7cca9cd,2,2,2,4,5
...,...,...,...,...,...
fa350da9efbba83c2957b629bf6c2c7c,2,2,2,1,5
fc3d87913825143fa41c48e40267c47a,2,2,2,3,4
fcde2f7449493b4f207a23fae117b0fb,2,2,2,4,5


In [693]:
# Detalji narudžbi koje imaju više različitih ocjena

order_ids_razlicite_ocjene = analiza_vise_recenzija.loc[
    analiza_vise_recenzija["broj_razlicitih_ocjena"] > 1
].index

razlicite_ocjene_detalji = (
    reviews[
        reviews["order_id"].isin(order_ids_razlicite_ocjene)
    ]
    .sort_values(
        [
            "order_id",
            "review_creation_date",
            "review_answer_timestamp"
        ]
    )
)

razlicite_ocjene_detalji[
    [
        "order_id",
        "review_id",
        "review_score",
        "review_comment_title",
        "review_comment_message",
        "review_creation_date",
        "review_answer_timestamp"
    ]
]

,order_id,review_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
22779,013056cfe49763c6f66bda03396c5ee3,ab30810c29da5da8045216f0f62652a2,5,NaN,NaN,2018-02-22,2018-02-23 12:12:30
68633,013056cfe49763c6f66bda03396c5ee3,73413b847f63e02bc752b364f6d05ee9,4,NaN,NaN,2018-03-04,2018-03-05 17:02:00
89888,02355020fd0a40a0d56df9f6ff060413,0c8e7347f1cdd2aede37371543e3d163,3,NaN,UM DOS PRODUTOS (ENTREGA02) COMPRADOS NESTE PE...,2018-03-21,2018-03-22 01:32:08
17582,02355020fd0a40a0d56df9f6ff060413,017f0e1ea6386de662cbeba299c59ad1,1,NaN,ja reclamei varias vezes e ate hoje não sei on...,2018-03-29,2018-03-30 03:16:19
37911,029863af4b968de1e5d6a82782e662f5,04d945e95c788a3aa1ffbee42105637b,5,NaN,NaN,2017-07-14,2017-07-17 13:58:06
...,...,...,...,...,...,...,...
70962,fcde2f7449493b4f207a23fae117b0fb,ca2fb34d8c9fc880f0f94dde4b17e8e3,4,NaN,Faltou os acessórios.,2017-06-14,2017-06-17 15:18:36
1157,fd61441ba2a7b57e6342862e779b10b0,3c625d52413314975e47e144fbc3cb8e,5,NaN,NaN,2017-10-04,2017-10-05 18:02:50
86641,fd61441ba2a7b57e6342862e779b10b0,24ad2fc85ec000ba4fbfd9841d9a1972,4,NaN,"Tem que retirar nos correios, muito incômodo",2017-10-14,2017-10-16 21:23:28
94504,ff763b73e473d03c321bcd5a053316e8,c56a88a404315a0d9e412c1472dda2c4,5,NaN,NaN,2017-11-01,2017-11-01 19:55:52


### Višestruke recenzije i ponavljanje identifikatora recenzija

Analizom tablice `reviews` utvrđeno je da odnos između narudžbi i zapisa recenzija nije u potpunosti jedan-prema-jedan. Od ukupno 98.673 narudžbe zastupljene u tablici recenzija, 547 narudžbi ima više od jednog zapisa recenzije. Među njima 345 narudžbi ima istu ocjenu u svim zapisima, dok 202 narudžbe imaju različite ocjene.

Detaljnim pregledom narudžbi s različitim ocjenama utvrđeno je da višestruki zapisi nisu potpuno duplicirani redci. Recenzije povezane s istom narudžbom mogu imati različite identifikatore recenzije, ocjene, komentare te datume nastanka i odgovora. U pojedinim slučajevima sadržaj komentara upućuje na promjenu korisničkog iskustva tijekom vremena, primjerice zbog problema ili kašnjenja u isporuci.

Također je utvrđeno da 789 vrijednosti `review_id` nije jedinstveno na razini narudžbe, odnosno isti `review_id` može biti povezan s više različitih `order_id`. Zbog toga se `review_id` u daljnjoj obradi neće automatski smatrati pouzdanim jedinstvenim ključem zapisa.

Budući da nisu pronađeni potpuno identični retci, zapisi se u ovoj fazi neće uklanjati kao duplikati. Izvorna granularnost tablice `reviews` bit će sačuvana. Ako za kasnije statističke analize bude potrebna jedna ocjena po narudžbi, izradit će se zaseban agregirani skup na razini `order_id` uz jasno definirano pravilo odabira ili agregacije recenzija.

In [694]:
# Osnovna statistika ocjena

print("Osnovna statistika ocjena:")
print(reviews["review_score"].describe())

print("\nBroj recenzija prema ocjeni:")
print(
    reviews["review_score"]
    .value_counts()
    .sort_index()
)

print("\nUdio pojedinih ocjena (%):")
print(
    reviews["review_score"]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
)

Osnovna statistika ocjena:
count    99224.000000
mean         4.086421
std          1.347579
min          1.000000
25%          4.000000
50%          5.000000
75%          5.000000
max          5.000000
Name: review_score, dtype: float64

Broj recenzija prema ocjeni:
review_score
1    11424
2     3151
3     8179
4    19142
5    57328
Name: count, dtype: int64

Udio pojedinih ocjena (%):
review_score
1    11.51
2     3.18
3     8.24
4    19.29
5    57.78
Name: proportion, dtype: float64


### Distribucija ocjena

Analizom 99.224 zapisa recenzija utvrđeno je da se ocjene kreću u očekivanom rasponu od 1 do 5, bez vrijednosti izvan dopuštenog raspona. Prosječna ocjena iznosi približno 4,09, dok je medijan jednak 5.

Distribucija ocjena izrazito je usmjerena prema višim ocjenama. Najzastupljenija je ocjena 5, koju ima 57,78 % svih zapisa, dok ocjena 4 čini dodatnih 19,29 %. Prema tome, ukupno 77,07 % zapisa ima ocjenu 4 ili 5.

Niže ocjene zastupljene su u manjoj mjeri: ocjena 1 čini 11,51 %, ocjena 2 čini 3,18 %, a ocjena 3 čini 8,24 % svih zapisa.

Rezultati pokazuju općenito visoku razinu zadovoljstva kupaca, ali postoji i skup izrazito nezadovoljnih kupaca s ocjenom 1. 

In [695]:
# Analiza vremena između kreiranja recenzije i odgovora na recenziju

reviews["vrijeme_do_odgovora_sati"] = (
    reviews["review_answer_timestamp"]
    - reviews["review_creation_date"]
).dt.total_seconds() / 3600

print("Osnovna statistika vremena do odgovora (u satima):")
print(
    reviews["vrijeme_do_odgovora_sati"].describe()
)

print(
    "\nBroj zapisa s negativnim vremenom do odgovora:",
    (reviews["vrijeme_do_odgovora_sati"] < 0).sum()
)

print(
    "Broj zapisa s vremenom do odgovora = 0:",
    (reviews["vrijeme_do_odgovora_sati"] == 0).sum()
)

print(
    "Broj zapisa s vremenom do odgovora > 30 dana:",
    (reviews["vrijeme_do_odgovora_sati"] > 30 * 24).sum()
)

Osnovna statistika vremena do odgovora (u satima):
count    99224.000000
mean        75.575842
std        237.361183
min          2.141389
25%         24.116875
50%         40.198750
75%         74.485556
max      12448.781111
Name: vrijeme_do_odgovora_sati, dtype: float64

Broj zapisa s negativnim vremenom do odgovora: 0
Broj zapisa s vremenom do odgovora = 0: 0
Broj zapisa s vremenom do odgovora > 30 dana: 682


### Analiza vremenskog razmaka između kreiranja recenzije i vremena odgovora

Izračunat je vremenski razmak između atributa `review_creation_date` i `review_answer_timestamp`. Prosječan vremenski razmak iznosi približno 75,58 sati, odnosno 3,15 dana, dok medijan iznosi približno 40,20 sati, odnosno 1,67 dana.

 Nisu pronađene negativne vrijednosti ni zapisi kod kojih je vremenski razmak jednak nuli, što upućuje na vremensku konzistentnost promatranih atributa.

U skupu se ipak pojavljuju izrazito velike vrijednosti. Maksimalni zabilježeni vremenski razmak iznosi približno 12.448,78 sati, odnosno oko 520 dana, dok 682 zapisa imaju vremenski razmak dulji od 30 dana. Takve vrijednosti utječu na prosječnu vrijednost, zbog čega je prosjek veći od medijana te distribucija vremenskog razmaka pokazuje desnu asimetriju.

Ekstremne vrijednosti se u ovoj fazi ne uklanjaju jer same po sebi ne predstavljaju nužno pogreške u podacima. 

In [696]:
# Provjera povezivosti recenzija i narudžbi

reviews_orders_provjera = (
    reviews[["order_id"]]
    .drop_duplicates()
    .merge(
        orders[["order_id"]],
        on="order_id",
        how="outer",
        indicator=True
    )
)

print("Povezivost reviews i orders:")
print(reviews_orders_provjera["_merge"].value_counts())

print(
    "\nBroj jedinstvenih narudžbi u reviews:",
    reviews["order_id"].nunique()
)

print(
    "Broj jedinstvenih narudžbi u orders:",
    orders["order_id"].nunique()
)

Povezivost reviews i orders:
_merge
both          98673
right_only      768
left_only         0
Name: count, dtype: int64

Broj jedinstvenih narudžbi u reviews: 98673
Broj jedinstvenih narudžbi u orders: 99441


In [697]:
zadnja_recenzija_po_narudzbi = (
    reviews
    .sort_values(
        ["order_id", "review_creation_date", "review_answer_timestamp"]
    )
    .drop_duplicates(
        subset="order_id",
        keep="last"
    )
    .copy()
)

print(
    "Broj zapisa nakon odabira posljednje recenzije:",
    len(zadnja_recenzija_po_narudzbi)
)

print(
    "Broj jedinstvenih narudžbi:",
    zadnja_recenzija_po_narudzbi["order_id"].nunique()
)

Broj zapisa nakon odabira posljednje recenzije: 98673
Broj jedinstvenih narudžbi: 98673


In [698]:
reviews_dostava = (
    orders[
        (orders["order_status"] == "delivered") &
        (orders["order_delivered_customer_date"].notna())
    ][
        [
            "order_id",
            "order_purchase_timestamp",
            "order_delivered_customer_date",
            "order_estimated_delivery_date"
        ]
    ]
    .merge(
        zadnja_recenzija_po_narudzbi[
            [
                "order_id",
                "review_score"
            ]
        ],
        on="order_id",
        how="inner"
    )
)

print("Broj narudžbi u analizi:", len(reviews_dostava))
print(
    "Broj jedinstvenih narudžbi:",
    reviews_dostava["order_id"].nunique()
)

reviews_dostava.head()

Broj narudžbi u analizi: 95824
Broj jedinstvenih narudžbi: 95824


,order_id,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date,review_score
0,e481f51cbdc54678b7cc49136f2d6af7,2017-10-02 10:56:33,2017-10-10 21:25:13,2017-10-18,4
1,53cdb2fc8bc7dce0b6741e2150273451,2018-07-24 20:41:37,2018-08-07 15:27:45,2018-08-13,4
2,47770eb9100c2d0c44946d9cf07ec65d,2018-08-08 08:38:49,2018-08-17 18:06:29,2018-09-04,5
3,949d5b44dbf5de918fe9c16f97b45f8a,2017-11-18 19:28:06,2017-12-02 00:28:42,2017-12-15,5
4,ad21c59c0840e6cb83a9ceb5573f8159,2018-02-13 21:18:39,2018-02-16 18:17:02,2018-02-26,5


## 8.9. Analiza prijevoda kategorija proizvoda

Tablica `category_translation` sadrži mapiranje izvornih naziva kategorija proizvoda na engleske nazive kategorija.

Cilj ove analize je:
- provjeriti strukturu i kvalitetu podataka,
- provjeriti postoje li nedostajuće vrijednosti,
- provjeriti postoje li duplicirani nazivi kategorija,
- utvrditi je li svaki izvorni naziv kategorije povezan s jednim engleskim prijevodom,
- provjeriti imaju li sve kategorije iz tablice `products` odgovarajući prijevod.



In [699]:
print("Broj zapisa:", len(category_translation))

print(
    "Broj jedinstvenih izvornih kategorija:",
    category_translation["product_category_name"].nunique()
)

print(
    "Broj jedinstvenih engleskih kategorija:",
    category_translation["product_category_name_english"].nunique()
)

print("\nStupci:")
print(category_translation.columns.tolist())

print("\nTipovi podataka:")
print(category_translation.dtypes)

print("\nNedostajuće vrijednosti:")
print(category_translation.isna().sum())

print(
    "\nBroj potpuno dupliciranih redaka:",
    category_translation.duplicated().sum()
)

category_translation.head(10)

Broj zapisa: 71
Broj jedinstvenih izvornih kategorija: 71
Broj jedinstvenih engleskih kategorija: 71

Stupci:
['product_category_name', 'product_category_name_english']

Tipovi podataka:
product_category_name            str
product_category_name_english    str
dtype: object

Nedostajuće vrijednosti:
product_category_name            0
product_category_name_english    0
dtype: int64

Broj potpuno dupliciranih redaka: 0


,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor
5,esporte_lazer,sports_leisure
6,perfumaria,perfumery
7,utilidades_domesticas,housewares
8,telefonia,telephony
9,relogios_presentes,watches_gifts


In [700]:
# Provjera pokrivenosti kategorija iz products tablice prijevodima

kategorije_products = set(
    products["product_category_name"].dropna().unique()
)

kategorije_translation = set(
    category_translation["product_category_name"].dropna().unique()
)

kategorije_bez_prijevoda = (
    kategorije_products - kategorije_translation
)

prijevodi_bez_proizvoda = (
    kategorije_translation - kategorije_products
)

print(
    "Broj jedinstvenih kategorija u products:",
    len(kategorije_products)
)

print(
    "Broj kategorija u category_translation:",
    len(kategorije_translation)
)

print(
    "\nBroj kategorija iz products bez prijevoda:",
    len(kategorije_bez_prijevoda)
)

print(
    "Kategorije iz products bez prijevoda:",
    sorted(kategorije_bez_prijevoda)
)

print(
    "\nBroj prijevoda za kategorije koje se ne pojavljuju u products:",
    len(prijevodi_bez_proizvoda)
)

print(
    "Kategorije s prijevodom koje se ne pojavljuju u products:",
    sorted(prijevodi_bez_proizvoda)
)

Broj jedinstvenih kategorija u products: 73
Broj kategorija u category_translation: 71

Broj kategorija iz products bez prijevoda: 2
Kategorije iz products bez prijevoda: ['pc_gamer', 'portateis_cozinha_e_preparadores_de_alimentos']

Broj prijevoda za kategorije koje se ne pojavljuju u products: 0
Kategorije s prijevodom koje se ne pojavljuju u products: []


# Završni zaključci eksplorativne analize podataka

Provedena eksplorativna analiza omogućila je detaljan uvid u strukturu, kvalitetu i međusobnu povezanost podataka iz Olist skupa podataka. Analizirane su tablice kupaca, prodavača, geografskih podataka, narudžbi, stavki narudžbi, proizvoda, plaćanja i recenzija.

Analizom su identificirani problemi kvalitete podataka koji će se uzeti u obzir tijekom sljedeće faze rada – čišćenja, transformacije i pripreme podataka za izgradnju skladišta podataka.

## Glavne odluke za daljnju obradu

### Narudžbe
- Status narudžbe mora se uzeti u obzir pri interpretaciji nedostajućih vremenskih podataka.
- Nedostajući datumi dostave ne predstavljaju nužno pogrešku jer otkazane ili nedostupne narudžbe nisu morale biti isporučene.
- Analize stvarnog vremena dostave i kašnjenja provode se samo za narudžbe za koje postoji datum dostave kupcu.
- Za analizu kašnjenja uspoređuje se stvarni datum dostave s procijenjenim datumom dostave.

### Geografski podaci
- Tablica geolocation sadrži velik broj potpunih duplikata koje je moguće ukloniti tijekom čišćenja podataka.
- Jedan ZIP prefiks može imati više geografskih koordinata, zbog čega je prije povezivanja s kupcima i prodavačima potrebno definirati reprezentativnu geografsku lokaciju za svaki ZIP prefiks.
- Geografski podaci mogu se koristiti za obogaćivanje dimenzija kupaca i prodavača.

### Proizvodi i stavke narudžbi
- Cijene proizvoda i troškovi dostave nemaju negativnih vrijednosti.
- Trošak dostave jednak nuli ne smatra se automatski pogreškom.
- Postoje stavke kod kojih je trošak dostave veći od cijene proizvoda; takvi zapisi predstavljaju moguće ekstremne vrijednosti, ali valjane poslovne slučajeve i neće se automatski uklanjati.
- Fizičke karakteristike proizvoda mogu se koristiti za dodatne statističke analize povezanosti težine i volumena proizvoda s troškom dostave.
- Nedostajuće karakteristike proizvoda potrebno je posebno obraditi tijekom pripreme podataka.

### Plaćanja
- Jedna narudžba može imati više zapisa plaćanja i koristiti više različitih načina plaćanja.
- Zbog toga se vrijednost plaćanja na razini narudžbe dobiva agregiranjem svih pripadajućih zapisa plaćanja.
- Vrijednosti `payment_type = not_defined` pojavljuju se kod malog broja specifičnih narudžbi i neće se automatski interpretirati kao standardni način plaćanja.
- Zapisi s vrijednošću plaćanja jednakom nuli analizirani su zasebno i neće se automatski uklanjati bez uzimanja u obzir konteksta narudžbe.
- Uočena su određena odstupanja između ukupnog iznosa plaćanja i zbroja cijena proizvoda i dostave. Takvi zapisi neće se automatski uklanjati jer razlike mogu biti povezane s načinom evidentiranja plaćanja, voucherima ili drugim poslovnim pravilima.

### Recenzije
- Jedna narudžba može imati više zapisa recenzija, a jedan `review_id` može biti povezan s više zapisa.
- Potpuno duplicirani retci nisu pronađeni, stoga se višestruke recenzije ne tretiraju kao obični duplikati.
- Za analize na razini narudžbe koristi se posljednja dostupna recenzija prema vremenskom slijedu, čime svaka narudžba sudjeluje jednom, a zadržava se posljednja poznata ocjena korisnika.
- Tekstualni komentari nisu obavezni te njihov izostanak ne predstavlja problem kvalitete podataka.
- Analiza je pokazala jasnu povezanost između kvalitete dostave i ocjene korisnika: narudžbe koje kasne imaju znatno niže ocjene, a prosječna ocjena dodatno pada s povećanjem kašnjenja.

### Prijevodi kategorija proizvoda

- Tablica `category_translation` sadrži ukupno 71 zapis, odnosno 71 jedinstvenu izvornu kategoriju i 71 jedinstvenu englesku kategoriju.
- U tablici nema nedostajućih vrijednosti ni potpuno dupliciranih redaka.
- Sve izvorne i engleske kategorije su jedinstvene, stoga su postojeća mapiranja između izvornih i engleskih naziva kategorija jednoznačna.
- U tablici `products` pojavljuju se ukupno 73 različite kategorije proizvoda, dok `category_translation` sadrži prijevode za 71 kategoriju.
- Utvrđeno je da dvije kategorije koje se pojavljuju u tablici `products`, `pc_gamer` i `portateis_cozinha_e_preparadores_de_alimentos`, nemaju odgovarajući engleski prijevod u tablici `category_translation`.
- Kategorijama bez prijevoda pripada ukupno 13 proizvoda: 3 proizvoda iz kategorije `pc_gamer` i 10 proizvoda iz kategorije `portateis_cozinha_e_preparadores_de_alimentos`.
- Proizvodi iz kategorija bez prijevoda pojavljuju se u ukupno 24 stavke narudžbi, odnosno u 22 različite narudžbe, zbog čega se te kategorije ne bi trebale zanemariti ili ukloniti iz daljnje obrade.
- U tablici `category_translation` nisu pronađeni prijevodi za kategorije koje se ne pojavljuju u tablici `products`.
- U ETL procesu za dvije kategorije bez prijevoda ručno će se definirati odgovarajući engleski nazivi kako bi se osigurala potpuna pokrivenost poznatih kategorija proizvoda.
- Izvorne CSV datoteke pritom će ostati nepromijenjene, a nadopuna nedostajućih prijevoda provest će se tijekom ETL procesa.